# Cellular Senescence Scoring Pipeline

**Module 01: SenePy-Based Senescence Identification**

**Method:** Cell-type and sex-specific senescence scoring using hippocampus-derived gene modules.

**Workflow:**
1. Load preprocessed data
2. Apply SenePy scoring
3. Define senescence threshold (mean + 2SD from youngest donors)
4. Binary classification: SnC vs Non-SnC
5. Validate senescence labels

**Author:** Gerald Gaitos  
**Date:** January 2026

---

## Setup

Import libraries for senescence scoring and analysis.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# SenePy for senescence scoring
try:
    import senepy
    print("✓ senepy installed")
except ImportError:
    print("⚠ senepy not installed. Run: pip install senepy")
    raise

print(f"scanpy: {sc.__version__}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")

In [ ]:
# Figure settings
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'legend.fontsize': 9,
    'font.family': 'sans-serif',
    'axes.linewidth': 1.0,
    'axes.grid': False,
    'pdf.fonttype': 42,
})

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=150, dpi_save=300, facecolor='white', frameon=False)

print("✓ Figure settings configured")

---

## Configuration

Set dataset and paths. Must match Module 00 preprocessing.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# DATASET SELECTION
# ════════════════════════════════════════════════════════════════════════════════

DATASET = 'psychad_aging'  # Options: 'psychad_aging', 'psychad_ad', 'psychencode', 'mathys'

# ════════════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════════════

BASE_DIR = Path('/fs/scratch/PAS2598/senescence_analysis')
INPUT_FILE = BASE_DIR / 'data' / 'processed' / f'{DATASET}_pearson.h5ad'
OUTPUT_FILE = BASE_DIR / 'data' / 'processed' / f'{DATASET}_pearson_senescence_scored.h5ad'
FIGURES_DIR = BASE_DIR / 'figures' / '01_senescence_scoring' / DATASET
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════════════
# SENESCENCE SCORING PARAMETERS
# ════════════════════════════════════════════════════════════════════════════════

SENEPY_TISSUE = 'hippocampus'  # Tissue-specific gene modules
SD_THRESHOLD = 2.0              # Standard deviations above mean for threshold

# Reference group for threshold (dataset-specific):
# - Aging cohorts: Use youngest age group (e.g., 'Age_20_29')
# - Disease cohorts: Use healthy controls (e.g., 'Control', 'Young_Control', 'NCI')
REFERENCE_GROUP = 'Age_20_29'  # Will be set automatically below

# ════════════════════════════════════════════════════════════════════════════════
# COLUMN NAMES (adjust based on your data)
# ════════════════════════════════════════════════════════════════════════════════

CELL_TYPE_COLUMN = 'subclass'   # Harmonized cell types from Module 00
AGE_COLUMN = 'Age'              # Numeric age variable
SEX_COLUMN = 'Sex'     # Sex for sex-specific scoring
DONOR_COLUMN = 'Sample'         # Donor ID

# Study_Group will be created to harmonize across cohorts:
# - Aging cohorts: 10-year age bins (Age_20_29, Age_30_39, etc.)
# - Disease cohorts: Diagnosis groups (Control, AD, MCI, etc.)
STUDY_GROUP_COLUMN = 'Study_Group'

# ════════════════════════════════════════════════════════════════════════════════
# DATASET-SPECIFIC CONFIGURATIONS
# ════════════════════════════════════════════════════════════════════════════════

DATASET_CONFIG = {
    'psychad_aging': {
        'type': 'aging',
        'age_bins': [(20, 29), (30, 39), (40, 49), (50, 59), (60, 69), (70, 79), (80, 100)],
        'reference_group': 'Age_20_29',
        'diagnosis_column': None,  # No diagnosis in aging cohort
    },
    'psychad_aging_norm': {
        'type': 'aging',
        'age_bins': [(20, 29), (30, 39), (40, 49), (50, 59), (60, 69), (70, 79), (80, 100)],
        'reference_group': 'Age_20_29',
        'diagnosis_column': None,  # No diagnosis in aging cohort
    },
    'psychad_ad': {
        'type': 'disease',
        'age_bins': None,  # Use diagnosis instead
        'reference_group': 'Old_Healthy_Control',  # Or 'Young_Control' if you have age groups
        'diagnosis_column': 'Disease_Group',  # Column with Control/AD labels
    },
    'psychencode': {
        'type': 'aging',
        'age_bins': [(30, 39), (40, 49), (50, 59), (60, 69), (70, 79), (80, 100)],
        'reference_group': 'Age_30_39',
        'diagnosis_column': None,
    },
    'psychencode_sub': {
        'type': 'aging',
        'age_bins': [(30, 39), (40, 49), (50, 59), (60, 69), (70, 79), (80, 100)],
        'reference_group': 'Age_30_39',
        'diagnosis_column': None,
    },
    'mathys': {
        'type': 'disease',
        'age_bins': None,
        'reference_group': 'NCI',  # No Cognitive Impairment
        'diagnosis_column': 'Disease_Group',  # Column with NCI/MCI/AD labels
    },
}
config = DATASET_CONFIG[DATASET]
REFERENCE_GROUP = config['reference_group']

print("="*80)
print(f"Dataset: {DATASET}")
print(f"Type: {config['type']}")
print(f"Reference group for threshold: {REFERENCE_GROUP}")
print(f"Input: {INPUT_FILE.name}")
print(f"Output: {OUTPUT_FILE.name}")
print(f"Figures: {FIGURES_DIR}")
print("="*80)

---

## Load Data and Create Study Groups

Load preprocessed data and create harmonized Study_Group column for cross-cohort consistency.

In [ ]:
print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

adata = sc.read_h5ad(INPUT_FILE)

print(f"\nLoaded: {INPUT_FILE.name}")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")

# Verify required columns exist
required_cols = [CELL_TYPE_COLUMN, AGE_COLUMN, SEX_COLUMN, DONOR_COLUMN]
missing_cols = [col for col in required_cols if col not in adata.obs.columns]
if missing_cols:
    print(f"\n⚠ Warning: Missing columns: {missing_cols}")
    print("Available columns:", list(adata.obs.columns))

print("\n" + "="*80)
print("CREATING STUDY_GROUP COLUMN")
print("="*80)

if config['type'] == 'aging':
    # Create age bins
    print(f"\nCreating age bins...")
    age_bins = config['age_bins']
    
    def assign_age_group(age):
        for bin_min, bin_max in age_bins:
            if bin_min <= age <= bin_max:
                return f"Age_{bin_min}_{bin_max}"
        return "Age_Unknown"
    
    adata.obs[STUDY_GROUP_COLUMN] = adata.obs[AGE_COLUMN].apply(assign_age_group)
    
    # Show distribution
    group_counts = adata.obs[STUDY_GROUP_COLUMN].value_counts().sort_index()
    print(f"\nAge group distribution:")
    for group, count in group_counts.items():
        print(f"  {group}: {count:,} cells")
    
elif config['type'] == 'disease':
    # Use diagnosis column
    diag_col = config['diagnosis_column']
    if diag_col and diag_col in adata.obs.columns:
        print(f"\nUsing diagnosis groups from '{diag_col}'...")
        adata.obs[STUDY_GROUP_COLUMN] = adata.obs[diag_col].astype(str)
        
        # Show distribution
        group_counts = adata.obs[STUDY_GROUP_COLUMN].value_counts()
        print(f"\nDiagnosis group distribution:")
        for group, count in group_counts.items():
            print(f"  {group}: {count:,} cells")
    else:
        print(f"\n⚠ Diagnosis column '{diag_col}' not found!")
        print(f"Available columns: {list(adata.obs.columns)}")

# Verify reference group exists
if REFERENCE_GROUP not in adata.obs[STUDY_GROUP_COLUMN].unique():
    print(f"\n⚠ WARNING: Reference group '{REFERENCE_GROUP}' not found in Study_Group!")
    print(f"Available groups: {sorted(adata.obs[STUDY_GROUP_COLUMN].unique())}")
else:
    n_ref = (adata.obs[STUDY_GROUP_COLUMN] == REFERENCE_GROUP).sum()
    print(f"\n✓ Reference group '{REFERENCE_GROUP}': {n_ref:,} cells")

print("\n✓ Data loaded and Study_Group created")

---

## SenePy Senescence Scoring

Apply cell-type and sex-specific senescence scoring using hippocampus-derived gene modules.

**Method:** SenePy uses tissue-specific gene modules and accounts for biological variability.

In [ ]:
print("\n" + "="*80)
print("SENESCENCE SCORING")
print("="*80)

print(f"\nScoring parameters:")
print(f"  Species: Human")
print(f"  Tissue: {SENEPY_TISSUE}")
print(f"  Cell type column: {CELL_TYPE_COLUMN}")
print(f"  Sex column: {SEX_COLUMN}")

# Step 1: Load SenePy hubs
print("\n1. Loading SenePy hubs...")
import senepy as sp
hubs = sp.load_hubs(species='Human')
print(f"   ✓ Loaded {len(hubs.metadata)} hub entries")

# Step 2: Filter for brain tissue
print(f"\n2. Filtering for {SENEPY_TISSUE} tissue...")
brain_hubs = hubs.metadata[hubs.metadata.tissue == SENEPY_TISSUE]
print(f"   ✓ Found {len(brain_hubs)} {SENEPY_TISSUE}-specific hubs")

# Step 3: Merge hubs
print(f"\n3. Merging hubs...")
hubs.merge_hubs(brain_hubs, new_name='brain')
print(f"   ✓ Hubs merged")

# Step 4: Create translator
print(f"\n4. Creating gene translator...")
translator = sp.translator(hub=hubs.hubs, data=adata)
print(f"   ✓ Translator created")

# Step 5: Score all cells
print(f"\n5. Scoring cells (cell-type and sex-specific)...")
print(f"   This may take 10-20 minutes for large datasets...")

adata.obs['senescence_score'] = sp.score_all_cells(
    adata, 
    hubs.hubs['brain'],
    identifiers=[CELL_TYPE_COLUMN, SEX_COLUMN],
    translator=translator
)

print(f"\n✓ Senescence scoring complete")

# Show score statistics
scores = adata.obs['senescence_score']
print(f"\nScore statistics:")
print(f"  Mean: {scores.mean():.3f}")
print(f"  Median: {scores.median():.3f}")
print(f"  Std: {scores.std():.3f}")
print(f"  Min: {scores.min():.3f}")
print(f"  Max: {scores.max():.3f}")

---

## Define Senescence Threshold

Calculate cell-type-specific thresholds using reference group (youngest age group).

**Method:** Mean + 2SD from reference group per cell type.

In [ ]:
print("\n" + "="*80)
print("SENESCENCE THRESHOLD DEFINITION")
print("="*80)

print(f"\nReference group: {REFERENCE_GROUP}")
print(f"Threshold: Mean + {SD_THRESHOLD} SD per cell type")

# Get reference group cells
ref_mask = adata.obs[STUDY_GROUP_COLUMN] == REFERENCE_GROUP
n_ref = ref_mask.sum()

if n_ref == 0:
    raise ValueError(f"Reference group '{REFERENCE_GROUP}' not found!")

print(f"\nReference cells: {n_ref:,}")

# Calculate threshold per cell type
thresholds = {}
print(f"\nCalculating thresholds per cell type...")

for cell_type in adata.obs[CELL_TYPE_COLUMN].unique():
    # Get reference cells of this type
    ref_ct_mask = ref_mask & (adata.obs[CELL_TYPE_COLUMN] == cell_type)
    ref_scores = adata.obs.loc[ref_ct_mask, 'senescence_score']
    
    if len(ref_scores) > 0:
        mean = ref_scores.mean()
        std = ref_scores.std()
        threshold = mean + (SD_THRESHOLD * std)
        thresholds[cell_type] = threshold
        
        print(f"  {cell_type}:")
        print(f"    n={len(ref_scores):,}, mean={mean:.3f}, std={std:.3f}, threshold={threshold:.3f}")
    else:
        print(f"  {cell_type}: No reference cells - using global threshold")
        thresholds[cell_type] = adata.obs['senescence_score'].mean() + (SD_THRESHOLD * adata.obs['senescence_score'].std())

# Apply thresholds to classify cells
print(f"\nClassifying cells as SnC (senescent) or Non-SnC...")
adata.obs['is_senescent'] = False

for cell_type, threshold in thresholds.items():
    mask = (adata.obs[CELL_TYPE_COLUMN] == cell_type) & (adata.obs['senescence_score'] >= threshold)
    adata.obs.loc[mask, 'is_senescent'] = True

# Add binary label
adata.obs['senescence_label'] = adata.obs['is_senescent'].map({True: 'SnC', False: 'Non-SnC'})

# Show results
n_snc = adata.obs['is_senescent'].sum()
pct_snc = n_snc / len(adata.obs) * 100

print(f"\n✓ Classification complete")
print(f"\nOverall results:")
print(f"  SnC (senescent): {n_snc:,} cells ({pct_snc:.1f}%)")
print(f"  Non-SnC: {len(adata.obs) - n_snc:,} cells ({100-pct_snc:.1f}%)")

print(f"\nSnC proportion by cell type:")
for cell_type in sorted(adata.obs[CELL_TYPE_COLUMN].unique()):
    ct_mask = adata.obs[CELL_TYPE_COLUMN] == cell_type
    ct_snc = (ct_mask & adata.obs['is_senescent']).sum()
    ct_total = ct_mask.sum()
    ct_pct = ct_snc / ct_total * 100 if ct_total > 0 else 0
    print(f"  {cell_type}: {ct_snc:,}/{ct_total:,} ({ct_pct:.1f}%)")

---

## UMI Confounding Diagnostic

Assess correlation between senescence scores and library size to quantify technical confounding.

**Interpretation:**
- Spearman |ρ| > 0.3 or UMI fold difference > 1.5× → include `log10(total_counts)` as covariate in Module 02

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# UMI DISTRIBUTION: SnC vs Non-SnC
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("UMI DISTRIBUTION: SnC vs Non-SnC")
print("="*80)

from scipy.stats import mannwhitneyu

# Colors
COLOR_SNC = '#E15759'
COLOR_NONSNC = '#4E79A7'

CELL_TYPE_COLORS = {
    'Excitatory': '#4E79A7', 'Inhibitory': '#F28E2B', 'Astrocyte': '#E15759',
    'Oligodendrocyte': '#76B7B2', 'OPC': '#59A14F', 'Microglia': '#EDC948',
    'Endothelial': '#B07AA1', 'Pericyte': '#FF9DA7', 'VSMC': '#9C755F',
    'VLMC': '#BAB0AC', 'PVM': '#D37295', 'Adaptive': '#FABFD2',
}

snc_mask = adata.obs['is_senescent'] == True
nonsnc_mask = adata.obs['is_senescent'] == False

snc_umi = adata.obs.loc[snc_mask, 'total_counts']
nonsnc_umi = adata.obs.loc[nonsnc_mask, 'total_counts']

# ─────────────────────────────────────────────────────────────────────────────
# A. Overall Statistics
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'─'*60}")
print("A. OVERALL STATISTICS")
print(f"{'─'*60}")

fold_median = snc_umi.median() / nonsnc_umi.median()
fold_mean = snc_umi.mean() / nonsnc_umi.mean()

stat, p_val = mannwhitneyu(snc_umi, nonsnc_umi, alternative='two-sided')

print(f"\n  {'Metric':<15} {'Non-SnC':>12} {'SnC':>12} {'Fold':>10}")
print(f"  {'-'*52}")
print(f"  {'N Cells':<15} {len(nonsnc_umi):>12,} {len(snc_umi):>12,}")
print(f"  {'Min UMI':<15} {nonsnc_umi.min():>12,.0f} {snc_umi.min():>12,.0f}")
print(f"  {'Q25 UMI':<15} {nonsnc_umi.quantile(0.25):>12,.0f} {snc_umi.quantile(0.25):>12,.0f}")
print(f"  {'Median UMI':<15} {nonsnc_umi.median():>12,.0f} {snc_umi.median():>12,.0f} {fold_median:>9.2f}x")
print(f"  {'Q75 UMI':<15} {nonsnc_umi.quantile(0.75):>12,.0f} {snc_umi.quantile(0.75):>12,.0f}")
print(f"  {'Max UMI':<15} {nonsnc_umi.max():>12,.0f} {snc_umi.max():>12,.0f}")
print(f"  {'Mean UMI':<15} {nonsnc_umi.mean():>12,.0f} {snc_umi.mean():>12,.0f} {fold_mean:>9.2f}x")

print(f"\n  Mann-Whitney U test: p = {p_val:.2e}")

if fold_median > 1.5:
    print(f"\n  ⚠ SnC cells have {fold_median:.1f}x higher median UMI than Non-SnC")
    print(f"    → Strong technical confounding detected")
else:
    print(f"\n  ✓ UMI fold difference within acceptable range (≤1.5x)")

# ─────────────────────────────────────────────────────────────────────────────
# B. Per Cell Type Statistics
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'─'*60}")
print("B. PER CELL TYPE STATISTICS")
print(f"{'─'*60}")

ct_order = adata.obs[CELL_TYPE_COLUMN].value_counts().index.tolist()

print(f"\n  {'Cell Type':<16} {'Non-SnC Med':>12} {'SnC Med':>12} {'Fold':>8} {'Flag':>6}")
print(f"  {'-'*58}")

ct_stats = []
for ct in ct_order:
    ct_snc = adata.obs.loc[(adata.obs[CELL_TYPE_COLUMN] == ct) & snc_mask, 'total_counts']
    ct_nonsnc = adata.obs.loc[(adata.obs[CELL_TYPE_COLUMN] == ct) & nonsnc_mask, 'total_counts']
    
    if len(ct_snc) > 0 and len(ct_nonsnc) > 0:
        ct_fold = ct_snc.median() / ct_nonsnc.median()
    else:
        ct_fold = np.nan
    
    flag = '⚠' if ct_fold > 1.5 else ''
    
    ct_stats.append({
        'Cell_Type': ct,
        'N_NonSnC': len(ct_nonsnc),
        'N_SnC': len(ct_snc),
        'Median_NonSnC': ct_nonsnc.median() if len(ct_nonsnc) > 0 else np.nan,
        'Median_SnC': ct_snc.median() if len(ct_snc) > 0 else np.nan,
        'Fold': ct_fold
    })
    
    med_nonsnc = ct_nonsnc.median() if len(ct_nonsnc) > 0 else np.nan
    med_snc = ct_snc.median() if len(ct_snc) > 0 else np.nan
    
    print(f"  {ct:<16} {med_nonsnc:>12,.0f} {med_snc:>12,.0f} {ct_fold:>7.2f}x {flag:>6}")

ct_stats_df = pd.DataFrame(ct_stats)
n_flagged = (ct_stats_df['Fold'] > 1.5).sum()

if n_flagged > 0:
    print(f"\n  ⚠ {n_flagged} cell types with fold > 1.5x")
else:
    print(f"\n  ✓ No cell types with extreme UMI differences")

# ─────────────────────────────────────────────────────────────────────────────
# C. Overall Plots
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'─'*60}")
print("C. OVERALL PLOTS")
print(f"{'─'*60}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Panel 1: Histogram
bins = np.linspace(np.log10(adata.obs['total_counts'].min() + 1), 
                   np.log10(adata.obs['total_counts'].max() + 1), 50)

axes[0].hist(np.log10(nonsnc_umi + 1), bins=bins, color=COLOR_NONSNC, 
             alpha=0.6, density=True, label=f'Non-SnC (n={len(nonsnc_umi):,})')
axes[0].hist(np.log10(snc_umi + 1), bins=bins, color=COLOR_SNC, 
             alpha=0.6, density=True, label=f'SnC (n={len(snc_umi):,})')
axes[0].axvline(np.log10(nonsnc_umi.median() + 1), color=COLOR_NONSNC, 
                linestyle='--', linewidth=2, label=f'Non-SnC med: {nonsnc_umi.median():,.0f}')
axes[0].axvline(np.log10(snc_umi.median() + 1), color=COLOR_SNC, 
                linestyle='--', linewidth=2, label=f'SnC med: {snc_umi.median():,.0f}')
axes[0].set_xlabel('log₁₀(UMI + 1)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'UMI Distribution (fold = {fold_median:.2f}x)')
axes[0].legend(fontsize=7, frameon=False, loc='upper right')

# Panel 2: Box plot
box_data = [nonsnc_umi.values, snc_umi.values]
bp = axes[1].boxplot(box_data, positions=[0, 1], widths=0.6, patch_artist=True, showfliers=False)
bp['boxes'][0].set_facecolor(COLOR_NONSNC)
bp['boxes'][1].set_facecolor(COLOR_SNC)
for box in bp['boxes']:
    box.set_alpha(0.7)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Non-SnC', 'SnC'])
axes[1].set_ylabel('UMI Counts')
axes[1].set_title('UMI by SnC Status')

# Add median annotations
for i, (data, color) in enumerate(zip(box_data, [COLOR_NONSNC, COLOR_SNC])):
    med = np.median(data)
    axes[1].text(i, med, f'{med:,.0f}', ha='center', va='bottom', fontsize=8, color='black')

# Panel 3: Score vs UMI colored by SnC status
axes[2].scatter(nonsnc_umi, adata.obs.loc[nonsnc_mask, 'senescence_score'], 
                s=1, alpha=0.05, c=COLOR_NONSNC, label='Non-SnC', rasterized=True)
axes[2].scatter(snc_umi, adata.obs.loc[snc_mask, 'senescence_score'], 
                s=1, alpha=0.1, c=COLOR_SNC, label='SnC', rasterized=True)
axes[2].set_xlabel('Total UMI')
axes[2].set_ylabel('Senescence Score')
axes[2].set_title('Score vs UMI by SnC Status')
axes[2].legend(fontsize=8, frameon=False, markerscale=5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_umi_snc_comparison_overall.svg', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_umi_snc_comparison_overall.svg', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved: {DATASET}_umi_snc_comparison_overall.svg")

# ─────────────────────────────────────────────────────────────────────────────
# D. Per Cell Type Violin Plots
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'─'*60}")
print("D. PER CELL TYPE VIOLIN PLOTS")
print(f"{'─'*60}")

n_ct = len(ct_order)
fig, ax = plt.subplots(figsize=(max(10, n_ct * 0.8), 4))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

positions_nonsnc = np.arange(n_ct) * 2
positions_snc = np.arange(n_ct) * 2 + 0.7

# Non-SnC violins
nonsnc_data = [np.log10(adata.obs[(adata.obs[CELL_TYPE_COLUMN] == ct) & nonsnc_mask]['total_counts'].values + 1) 
               for ct in ct_order]
vp1 = ax.violinplot(nonsnc_data, positions=positions_nonsnc, showmedians=True, showextrema=False, widths=0.6)
for body in vp1['bodies']:
    body.set_facecolor(COLOR_NONSNC)
    body.set_alpha(0.7)
vp1['cmedians'].set_color('black')

# SnC violins
snc_data = [np.log10(adata.obs[(adata.obs[CELL_TYPE_COLUMN] == ct) & snc_mask]['total_counts'].values + 1) 
            for ct in ct_order]
vp2 = ax.violinplot(snc_data, positions=positions_snc, showmedians=True, showextrema=False, widths=0.6)
for body in vp2['bodies']:
    body.set_facecolor(COLOR_SNC)
    body.set_alpha(0.7)
vp2['cmedians'].set_color('black')

# X-axis
ax.set_xticks(np.arange(n_ct) * 2 + 0.35)
ax.set_xticklabels(ct_order, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('log₁₀(UMI + 1)')
ax.set_title('UMI Distribution by Cell Type and SnC Status')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=COLOR_NONSNC, alpha=0.7, label='Non-SnC'),
                   Patch(facecolor=COLOR_SNC, alpha=0.7, label='SnC')]
ax.legend(handles=legend_elements, loc='upper right', frameon=False)

# Add fold annotations
for i, ct in enumerate(ct_order):
    fold = ct_stats_df[ct_stats_df['Cell_Type'] == ct]['Fold'].values[0]
    if not np.isnan(fold):
        y_pos = max(np.max(nonsnc_data[i]), np.max(snc_data[i])) + 0.1
        color = '#E15759' if fold > 1.5 else '#333333'
        ax.text(positions_nonsnc[i] + 0.35, y_pos, f'{fold:.1f}x', 
                ha='center', va='bottom', fontsize=7, color=color)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_umi_snc_comparison_by_celltype.svg', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_umi_snc_comparison_by_celltype.svg', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {DATASET}_umi_snc_comparison_by_celltype.svg")

# ─────────────────────────────────────────────────────────────────────────────
# E. Summary
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*80)
print("UMI CONFOUNDING SUMMARY (SnC vs Non-SnC)")
print("="*80)

print(f"\n  Overall UMI fold (SnC/Non-SnC): {fold_median:.2f}x")
print(f"  Cell types with fold > 1.5x: {n_flagged}/{len(ct_order)}")

if fold_median > 1.5 or n_flagged > 0:
    print(f"\n  ⚠ CONFIRMATION: UMI strongly confounds SnC classification")
    print(f"    → Module 02 models MUST include log10(total_counts) as covariate")
else:
    print(f"\n  ✓ UMI confounding within acceptable limits")

print("="*80)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# UMI REGRESSION FROM SENESCENCE SCORES
# ════════════════════════════════════════════════════════════════════════════════

from sklearn.linear_model import LinearRegression
from scipy import stats  # <-- This was missing

print("\n" + "="*80)
print("UMI REGRESSION FROM SENESCENCE SCORES")
print("="*80)

# ────────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ────────────────────────────────────────────────────────────────────────────────

SCORE_COL = 'senescence_score'
ADJUSTED_SCORE_COL = 'senescence_score_adjusted'
UMI_COL = 'total_counts'
MIN_CELLS_REGRESSION = 50

print(f"\nConfiguration:")
print(f"  Score column: {SCORE_COL}")
print(f"  Cell type column: {CELL_TYPE_COLUMN}")
print(f"  Reference group: {REFERENCE_GROUP}")
print(f"  Threshold: Mean + {SD_THRESHOLD} SD")

# ────────────────────────────────────────────────────────────────────────────────
# ENSURE UMI COUNTS EXIST
# ────────────────────────────────────────────────────────────────────────────────

if UMI_COL not in adata.obs.columns:
    print(f"\n⚠ '{UMI_COL}' not in adata.obs")
    if 'counts' in adata.layers:
        print(f"  → Calculating from layers['counts']...")
        adata.obs[UMI_COL] = np.array(adata.layers['counts'].sum(axis=1)).flatten()
        print(f"  ✓ {UMI_COL} calculated")
    else:
        raise ValueError(f"Cannot find UMI counts")

# ────────────────────────────────────────────────────────────────────────────────
# STEP 1: PER CELL-TYPE REGRESSION
# ────────────────────────────────────────────────────────────────────────────────

print(f"\n" + "-"*60)
print("STEP 1: Per Cell-Type Regression")
print("-"*60)

adata.obs[ADJUSTED_SCORE_COL] = np.nan
regression_stats = []

for ct in sorted(adata.obs[CELL_TYPE_COLUMN].unique()):
    mask = adata.obs[CELL_TYPE_COLUMN] == ct
    n_cells = mask.sum()
    
    if n_cells < MIN_CELLS_REGRESSION:
        print(f"  {ct}: SKIPPED (n={n_cells:,} < {MIN_CELLS_REGRESSION})")
        adata.obs.loc[mask, ADJUSTED_SCORE_COL] = adata.obs.loc[mask, SCORE_COL]
        regression_stats.append({
            'Cell_Type': ct, 'N_Cells': n_cells, 'Status': 'skipped',
            'R_squared': np.nan, 'Slope': np.nan, 'P_value': np.nan
        })
        continue
    
    X = np.log10(adata.obs.loc[mask, UMI_COL].values).reshape(-1, 1)
    y = adata.obs.loc[mask, SCORE_COL].values
    
    valid = np.isfinite(X.flatten()) & np.isfinite(y)
    X_valid, y_valid = X[valid], y[valid]
    
    model = LinearRegression().fit(X_valid, y_valid)
    
    residuals = y - model.predict(X)
    adjusted = residuals + y.mean()
    adata.obs.loc[mask, ADJUSTED_SCORE_COL] = adjusted
    
    y_pred = model.predict(X_valid)
    ss_res = np.sum((y_valid - y_pred) ** 2)
    ss_tot = np.sum((y_valid - y_valid.mean()) ** 2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    r, p_value = stats.pearsonr(X_valid.flatten(), y_valid)
    
    regression_stats.append({
        'Cell_Type': ct, 'N_Cells': n_cells, 'Status': 'adjusted',
        'R_squared': r_squared, 'Slope': model.coef_[0], 
        'Intercept': model.intercept_, 'P_value': p_value,
        'Original_Mean': y.mean(), 'Adjusted_Mean': adjusted.mean()
    })
    
    print(f"  {ct}: n={n_cells:,}, R²={r_squared:.3f}, slope={model.coef_[0]:.4f}, p={p_value:.2e}")

regression_df = pd.DataFrame(regression_stats)
mean_r2 = regression_df[regression_df['Status'] == 'adjusted']['R_squared'].mean()
print(f"\n✓ Regression complete | Mean R² = {mean_r2:.3f}")

# ────────────────────────────────────────────────────────────────────────────────
# STEP 2: CALCULATE ADJUSTED THRESHOLDS
# ────────────────────────────────────────────────────────────────────────────────

print(f"\n" + "-"*60)
print("STEP 2: Calculate Adjusted Thresholds")
print("-"*60)
print(f"Reference group: {REFERENCE_GROUP}")

ref_mask = adata.obs[STUDY_GROUP_COLUMN] == REFERENCE_GROUP
thresholds_adjusted = {}

for ct in sorted(adata.obs[CELL_TYPE_COLUMN].unique()):
    ref_ct_mask = ref_mask & (adata.obs[CELL_TYPE_COLUMN] == ct)
    ref_scores = adata.obs.loc[ref_ct_mask, ADJUSTED_SCORE_COL].dropna()
    
    if len(ref_scores) >= 10:
        mean = ref_scores.mean()
        std = ref_scores.std()
        threshold = mean + (SD_THRESHOLD * std)
        thresholds_adjusted[ct] = threshold
        print(f"  {ct}: n={len(ref_scores):,}, threshold={threshold:.3f}")
    else:
        ct_scores = adata.obs.loc[adata.obs[CELL_TYPE_COLUMN] == ct, ADJUSTED_SCORE_COL].dropna()
        threshold = ct_scores.mean() + (SD_THRESHOLD * ct_scores.std())
        thresholds_adjusted[ct] = threshold
        print(f"  {ct}: using global (threshold={threshold:.3f})")

# ────────────────────────────────────────────────────────────────────────────────
# STEP 3: APPLY ADJUSTED CLASSIFICATION
# ────────────────────────────────────────────────────────────────────────────────

print(f"\n" + "-"*60)
print("STEP 3: Apply Adjusted Classification")
print("-"*60)

adata.obs['is_senescent_adjusted'] = False

for ct, threshold in thresholds_adjusted.items():
    mask = (adata.obs[CELL_TYPE_COLUMN] == ct) & (adata.obs[ADJUSTED_SCORE_COL] >= threshold)
    adata.obs.loc[mask, 'is_senescent_adjusted'] = True

adata.obs['senescence_label_adjusted'] = adata.obs['is_senescent_adjusted'].map(
    {True: 'SnC', False: 'Non-SnC'}
)

print("✓ Classification complete")

# ════════════════════════════════════════════════════════════════════════════════
# STEP 4: MULTI-LEVEL COMPARISON
# ════════════════════════════════════════════════════════════════════════════════

print(f"\n" + "="*80)
print("COMPARISON: ORIGINAL vs ADJUSTED")
print("="*80)

# ────────────────────────────────────────────────────────────────────────────────
# A. OVERALL
# ────────────────────────────────────────────────────────────────────────────────

print(f"\n" + "-"*60)
print("A. OVERALL")
print("-"*60)

total = len(adata.obs)
orig_snc = adata.obs['is_senescent'].sum()
adj_snc = adata.obs['is_senescent_adjusted'].sum()
orig_score_mean = adata.obs[SCORE_COL].mean()
adj_score_mean = adata.obs[ADJUSTED_SCORE_COL].mean()

print(f"\n  {'Metric':<25} {'Original':<15} {'Adjusted':<15} {'Δ':<15}")
print("  " + "-"*70)
print(f"  {'Mean Score':<25} {orig_score_mean:<15.4f} {adj_score_mean:<15.4f} {adj_score_mean - orig_score_mean:<+15.4f}")
print(f"  {'N SnC':<25} {orig_snc:<15,} {adj_snc:<15,} {adj_snc - orig_snc:<+15,}")
print(f"  {'% SnC':<25} {orig_snc/total*100:<15.2f} {adj_snc/total*100:<15.2f} {(adj_snc-orig_snc)/total*100:<+15.2f}")

# ────────────────────────────────────────────────────────────────────────────────
# B. PER STUDY GROUP
# ────────────────────────────────────────────────────────────────────────────────

print(f"\n" + "-"*60)
print("B. PER STUDY GROUP")
print("-"*60)

print(f"\n  {'Study Group':<15} {'N':<10} {'Orig Score':<12} {'Adj Score':<12} {'Orig %SnC':<12} {'Adj %SnC':<12} {'Δ %SnC':<10}")
print("  " + "-"*85)

study_groups = sorted(adata.obs[STUDY_GROUP_COLUMN].unique())
study_group_data = []

for sg in study_groups:
    sg_mask = adata.obs[STUDY_GROUP_COLUMN] == sg
    sg_n = sg_mask.sum()
    sg_orig_score = adata.obs.loc[sg_mask, SCORE_COL].mean()
    sg_adj_score = adata.obs.loc[sg_mask, ADJUSTED_SCORE_COL].mean()
    sg_orig_snc = adata.obs.loc[sg_mask, 'is_senescent'].sum()
    sg_adj_snc = adata.obs.loc[sg_mask, 'is_senescent_adjusted'].sum()
    sg_orig_pct = sg_orig_snc / sg_n * 100
    sg_adj_pct = sg_adj_snc / sg_n * 100
    
    study_group_data.append({
        'Study_Group': sg, 'N': sg_n,
        'Orig_Score': sg_orig_score, 'Adj_Score': sg_adj_score,
        'Orig_SnC': sg_orig_snc, 'Adj_SnC': sg_adj_snc,
        'Orig_Pct': sg_orig_pct, 'Adj_Pct': sg_adj_pct
    })
    
    print(f"  {sg:<15} {sg_n:<10,} {sg_orig_score:<12.4f} {sg_adj_score:<12.4f} "
          f"{sg_orig_pct:<12.2f} {sg_adj_pct:<12.2f} {sg_adj_pct - sg_orig_pct:<+10.2f}")

study_group_df = pd.DataFrame(study_group_data)

# ────────────────────────────────────────────────────────────────────────────────
# C. PER CELL TYPE
# ────────────────────────────────────────────────────────────────────────────────

print(f"\n" + "-"*60)
print("C. PER CELL TYPE")
print("-"*60)

print(f"\n  {'Cell Type':<20} {'N':<10} {'Orig Score':<12} {'Adj Score':<12} {'Orig %SnC':<12} {'Adj %SnC':<12} {'Δ %SnC':<10}")
print("  " + "-"*90)

cell_types = sorted(adata.obs[CELL_TYPE_COLUMN].unique())
cell_type_data = []

for ct in cell_types:
    ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct
    ct_n = ct_mask.sum()
    ct_orig_score = adata.obs.loc[ct_mask, SCORE_COL].mean()
    ct_adj_score = adata.obs.loc[ct_mask, ADJUSTED_SCORE_COL].mean()
    ct_orig_snc = adata.obs.loc[ct_mask, 'is_senescent'].sum()
    ct_adj_snc = adata.obs.loc[ct_mask, 'is_senescent_adjusted'].sum()
    ct_orig_pct = ct_orig_snc / ct_n * 100
    ct_adj_pct = ct_adj_snc / ct_n * 100
    
    cell_type_data.append({
        'Cell_Type': ct, 'N': ct_n,
        'Orig_Score': ct_orig_score, 'Adj_Score': ct_adj_score,
        'Orig_SnC': ct_orig_snc, 'Adj_SnC': ct_adj_snc,
        'Orig_Pct': ct_orig_pct, 'Adj_Pct': ct_adj_pct
    })
    
    print(f"  {ct:<20} {ct_n:<10,} {ct_orig_score:<12.4f} {ct_adj_score:<12.4f} "
          f"{ct_orig_pct:<12.2f} {ct_adj_pct:<12.2f} {ct_adj_pct - ct_orig_pct:<+10.2f}")

cell_type_df = pd.DataFrame(cell_type_data)

# ────────────────────────────────────────────────────────────────────────────────
# D. PER STUDY GROUP × CELL TYPE
# ────────────────────────────────────────────────────────────────────────────────

print(f"\n" + "-"*60)
print("D. PER STUDY GROUP × CELL TYPE")
print("-"*60)

sg_ct_data = []

for sg in study_groups:
    for ct in cell_types:
        mask = (adata.obs[STUDY_GROUP_COLUMN] == sg) & (adata.obs[CELL_TYPE_COLUMN] == ct)
        n = mask.sum()
        if n == 0:
            continue
        
        orig_score = adata.obs.loc[mask, SCORE_COL].mean()
        adj_score = adata.obs.loc[mask, ADJUSTED_SCORE_COL].mean()
        orig_snc = adata.obs.loc[mask, 'is_senescent'].sum()
        adj_snc = adata.obs.loc[mask, 'is_senescent_adjusted'].sum()
        
        sg_ct_data.append({
            'Study_Group': sg, 'Cell_Type': ct, 'N': n,
            'Orig_Score': orig_score, 'Adj_Score': adj_score,
            'Orig_SnC': orig_snc, 'Adj_SnC': adj_snc,
            'Orig_Pct': orig_snc / n * 100,
            'Adj_Pct': adj_snc / n * 100
        })

sg_ct_df = pd.DataFrame(sg_ct_data)

# Pivot for display
print("\n  Original %SnC:")
pivot_orig = sg_ct_df.pivot(index='Cell_Type', columns='Study_Group', values='Orig_Pct')
print(pivot_orig.round(1).to_string(index=True))

print("\n  Adjusted %SnC:")
pivot_adj = sg_ct_df.pivot(index='Cell_Type', columns='Study_Group', values='Adj_Pct')
print(pivot_adj.round(1).to_string(index=True))

print("\n  Δ %SnC (Adjusted - Original):")
pivot_delta = pivot_adj - pivot_orig
print(pivot_delta.round(1).to_string(index=True))

# ────────────────────────────────────────────────────────────────────────────────
# E. PER DONOR × STUDY GROUP × CELL TYPE
# ────────────────────────────────────────────────────────────────────────────────

print(f"\n" + "-"*60)
print("E. PER DONOR × STUDY GROUP × CELL TYPE")
print("-"*60)

donor_data = []

for donor in adata.obs[DONOR_COLUMN].unique():
    donor_mask = adata.obs[DONOR_COLUMN] == donor
    donor_sg = adata.obs.loc[donor_mask, STUDY_GROUP_COLUMN].iloc[0]
    
    for ct in cell_types:
        mask = donor_mask & (adata.obs[CELL_TYPE_COLUMN] == ct)
        n = mask.sum()
        if n == 0:
            continue
        
        orig_score = adata.obs.loc[mask, SCORE_COL].mean()
        adj_score = adata.obs.loc[mask, ADJUSTED_SCORE_COL].mean()
        orig_snc = adata.obs.loc[mask, 'is_senescent'].sum()
        adj_snc = adata.obs.loc[mask, 'is_senescent_adjusted'].sum()
        
        donor_data.append({
            'Donor': donor,
            'Study_Group': donor_sg,
            'Cell_Type': ct,
            'N': n,
            'Orig_Score': orig_score,
            'Adj_Score': adj_score,
            'Orig_SnC': orig_snc,
            'Adj_SnC': adj_snc,
            'Orig_Pct': orig_snc / n * 100 if n > 0 else 0,
            'Adj_Pct': adj_snc / n * 100 if n > 0 else 0
        })

donor_df = pd.DataFrame(donor_data)
donor_df['Delta_Pct'] = donor_df['Adj_Pct'] - donor_df['Orig_Pct']

# Summary statistics per study group × cell type
print("\n  Donor-level summary (mean ± std of %SnC across donors):")
print(f"\n  {'Study Group':<15} {'Cell Type':<15} {'N Donors':<10} {'Orig %SnC':<20} {'Adj %SnC':<20}")
print("  " + "-"*80)

donor_summary = donor_df.groupby(['Study_Group', 'Cell_Type']).agg({
    'Donor': 'nunique',
    'Orig_Pct': ['mean', 'std'],
    'Adj_Pct': ['mean', 'std']
}).reset_index()
donor_summary.columns = ['Study_Group', 'Cell_Type', 'N_Donors', 
                          'Orig_Mean', 'Orig_Std', 'Adj_Mean', 'Adj_Std']

for _, row in donor_summary.iterrows():
    orig_str = f"{row['Orig_Mean']:.1f} ± {row['Orig_Std']:.1f}"
    adj_str = f"{row['Adj_Mean']:.1f} ± {row['Adj_Std']:.1f}"
    print(f"  {row['Study_Group']:<15} {row['Cell_Type']:<15} {int(row['N_Donors']):<10} "
          f"{orig_str:<20} {adj_str:<20}")

print(f"\n✓ Donor-level data shape: {donor_df.shape}")

# ════════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("✓ UMI REGRESSION COMPLETE")
print("="*80)

print(f"\nNew columns added:")
print(f"  adata.obs['{ADJUSTED_SCORE_COL}']")
print(f"  adata.obs['is_senescent_adjusted']")
print(f"  adata.obs['senescence_label_adjusted']")

print(f"\nDataFrames created:")
print(f"  regression_df: {regression_df.shape}")
print(f"  study_group_df: {study_group_df.shape}")
print(f"  cell_type_df: {cell_type_df.shape}")
print(f"  sg_ct_df: {sg_ct_df.shape}")
print(f"  donor_df: {donor_df.shape}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# FIGURE: SnC BY STUDY GROUP PER CELL TYPE (Original vs Adjusted)
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("SnC BY STUDY GROUP PER CELL TYPE")
print("="*80)

# ─────────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────────

STUDY_TYPE = config['type']

GROUP_ORDER_CONFIG = {
    'psychad_aging': ['Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 
                      'Age_60_69', 'Age_70_79', 'Age_80_100'],
    'psychad_aging_norm': ['Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 
                      'Age_60_69', 'Age_70_79', 'Age_80_100'],
    'psychad_ad': ['Young_Healthy_Control', 'Old_Healthy_Control', 'Old_AD'],
    'psychencode': ['Age_30_39', 'Age_40_49', 'Age_50_59', 
                    'Age_60_69', 'Age_70_79', 'Age_80_100'],
    'psychencode_sub': ['Age_30_39', 'Age_40_49', 'Age_50_59', 
                        'Age_60_69', 'Age_70_79', 'Age_80_100'],
    'mathys': ['NCI', 'MCI', 'AD'],
}
GROUP_ORDER = GROUP_ORDER_CONFIG.get(DATASET, [])

STUDY_GROUP_COLORS = {
    'Age_20_29': '#2E86AB', 'Age_30_39': '#4A90E2', 'Age_40_49': '#50C878',
    'Age_50_59': '#FFB347', 'Age_60_69': '#FF8C00', 'Age_70_79': '#E24A4A',
    'Age_80_100': '#8B0000',
    'Control': '#4E79A7', 'MCI': '#F28E2B', 'AD': '#E15759', 'NCI': '#4E79A7',
    'Young_Healthy_Control': '#4E79A7', 'Old_Healthy_Control': '#59A14F', 'Old_AD': '#E15759',
}

group_order = [g for g in GROUP_ORDER if g in donor_df['Study_Group'].unique()]
ct_order = donor_df.groupby('Cell_Type')['Orig_Pct'].median().sort_values(ascending=False).index.tolist()

n_celltypes = len(ct_order)
n_cols = 4
n_rows = int(np.ceil(n_celltypes / n_cols))

if STUDY_TYPE == 'aging':
    x_labels = [g.replace('Age_', '').replace('_', '–') for g in group_order]
else:
    x_labels = group_order

print(f"Study type: {STUDY_TYPE}")
print(f"Groups: {group_order}")
print(f"Cell types: {ct_order}")

# ─────────────────────────────────────────────────────────────────────────────────
# PLOT 1: ORIGINAL LABELS
# ─────────────────────────────────────────────────────────────────────────────────

print("\n▸ Original labels:")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 1.5 * n_rows), sharey=True)
axes = axes.flatten()

for idx, ct in enumerate(ct_order):
    ax = axes[idx]
    ct_df = donor_df[donor_df['Cell_Type'] == ct]
    
    bp = ax.boxplot(
        [ct_df[ct_df['Study_Group'] == g]['Orig_Pct'].values for g in group_order],
        positions=range(len(group_order)),
        widths=0.5, patch_artist=True, showfliers=False
    )
    
    for i, (box, group) in enumerate(zip(bp['boxes'], group_order)):
        box.set_facecolor(STUDY_GROUP_COLORS.get(group, '#808080'))
        box.set_alpha(0.7)
        box.set_edgecolor('none')
    for whisker in bp['whiskers']:
        whisker.set_color('#666666')
        whisker.set_linewidth(0.5)
    for cap in bp['caps']:
        cap.set_color('#666666')
        cap.set_linewidth(0.5)
    for median in bp['medians']:
        median.set_color('#333333')
        median.set_linewidth(0.8)
    
    for i, group in enumerate(group_order):
        group_data = ct_df[ct_df['Study_Group'] == group]['Orig_Pct'].values
        if len(group_data) > 0:
            jitter = np.random.uniform(-0.1, 0.1, size=len(group_data))
            ax.scatter(np.repeat(i, len(group_data)) + jitter, group_data,
                       c=STUDY_GROUP_COLORS.get(group, '#808080'), s=5, alpha=0.6, 
                       zorder=3, edgecolors='#333333', linewidths=0.2)
    
    ax.set_title(ct, fontsize=7, fontweight='bold', pad=4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(labelsize=6, width=0.5)
    ax.grid(False)
    ax.set_xticks(range(len(group_order)))
    ax.set_xticklabels(x_labels, rotation=90, ha='center', fontsize=6)
    if idx % n_cols == 0:
        ax.set_ylabel('SnC (%)', fontsize=7)

for idx in range(n_celltypes, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(f'SnC by {STUDY_TYPE.capitalize()} Group per Cell Type (Original)', 
             fontsize=9, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_original.svg', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_original.svg', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {DATASET}_snc_by_group_celltype_original.svg")

# ─────────────────────────────────────────────────────────────────────────────────
# PLOT 2: ADJUSTED LABELS
# ─────────────────────────────────────────────────────────────────────────────────

print("\n▸ Adjusted labels:")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 1.5 * n_rows), sharey=True)
axes = axes.flatten()

for idx, ct in enumerate(ct_order):
    ax = axes[idx]
    ct_df = donor_df[donor_df['Cell_Type'] == ct]
    
    bp = ax.boxplot(
        [ct_df[ct_df['Study_Group'] == g]['Adj_Pct'].values for g in group_order],
        positions=range(len(group_order)),
        widths=0.5, patch_artist=True, showfliers=False
    )
    
    for i, (box, group) in enumerate(zip(bp['boxes'], group_order)):
        box.set_facecolor(STUDY_GROUP_COLORS.get(group, '#808080'))
        box.set_alpha(0.7)
        box.set_edgecolor('none')
    for whisker in bp['whiskers']:
        whisker.set_color('#666666')
        whisker.set_linewidth(0.5)
    for cap in bp['caps']:
        cap.set_color('#666666')
        cap.set_linewidth(0.5)
    for median in bp['medians']:
        median.set_color('#333333')
        median.set_linewidth(0.8)
    
    for i, group in enumerate(group_order):
        group_data = ct_df[ct_df['Study_Group'] == group]['Adj_Pct'].values
        if len(group_data) > 0:
            jitter = np.random.uniform(-0.1, 0.1, size=len(group_data))
            ax.scatter(np.repeat(i, len(group_data)) + jitter, group_data,
                       c=STUDY_GROUP_COLORS.get(group, '#808080'), s=5, alpha=0.6, 
                       zorder=3, edgecolors='#333333', linewidths=0.2)
    
    ax.set_title(ct, fontsize=7, fontweight='bold', pad=4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(labelsize=6, width=0.5)
    ax.grid(False)
    ax.set_xticks(range(len(group_order)))
    ax.set_xticklabels(x_labels, rotation=90, ha='center', fontsize=6)
    if idx % n_cols == 0:
        ax.set_ylabel('SnC (%)', fontsize=7)

for idx in range(n_celltypes, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(f'SnC by {STUDY_TYPE.capitalize()} Group per Cell Type (Adjusted)', 
             fontsize=9, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_adjusted.svg', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_adjusted.svg', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {DATASET}_snc_by_group_celltype_adjusted.svg")

print("\n" + "="*80)
print(f"✓ Cell types: {n_celltypes}")
print(f"✓ Study groups: {len(group_order)}")
print("="*80)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# FIGURE: SnC BY STUDY GROUP PER CELL TYPE - CELL LEVEL (faceted bar plot)
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("SnC BY STUDY GROUP PER CELL TYPE (CELL LEVEL)")
print("="*80)

# ─────────────────────────────────────────────────────────────────────────────────
# Calculate cell-level SnC% for ORIGINAL labels
# ─────────────────────────────────────────────────────────────────────────────────

cell_level_orig = []
for ct in adata.obs[CELL_TYPE_COLUMN].unique():
    ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct
    ct_data = adata.obs[ct_mask]
    
    for group in adata.obs[STUDY_GROUP_COLUMN].unique():
        group_mask = ct_data[STUDY_GROUP_COLUMN] == group
        group_data = ct_data[group_mask]
        n_total = len(group_data)
        if n_total > 0:
            n_snc = group_data['is_senescent'].sum()
            snc_pct = (n_snc / n_total * 100)
            cell_level_orig.append({
                'Cell_Type': ct,
                'Study_Group': group,
                'SnC_pct': snc_pct,
                'n_cells': n_total
            })

df_cell_orig = pd.DataFrame(cell_level_orig)

# ─────────────────────────────────────────────────────────────────────────────────
# Calculate cell-level SnC% for ADJUSTED labels
# ─────────────────────────────────────────────────────────────────────────────────

cell_level_adj = []
for ct in adata.obs[CELL_TYPE_COLUMN].unique():
    ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct
    ct_data = adata.obs[ct_mask]
    
    for group in adata.obs[STUDY_GROUP_COLUMN].unique():
        group_mask = ct_data[STUDY_GROUP_COLUMN] == group
        group_data = ct_data[group_mask]
        n_total = len(group_data)
        if n_total > 0:
            n_snc = group_data['is_senescent_adjusted'].sum()
            snc_pct = (n_snc / n_total * 100)
            cell_level_adj.append({
                'Cell_Type': ct,
                'Study_Group': group,
                'SnC_pct': snc_pct,
                'n_cells': n_total
            })

df_cell_adj = pd.DataFrame(cell_level_adj)

# ─────────────────────────────────────────────────────────────────────────────────
# Setup
# ─────────────────────────────────────────────────────────────────────────────────

ct_order_cell = df_cell_orig.groupby('Cell_Type')['SnC_pct'].median().sort_values(ascending=False).index.tolist()
group_order_cell = [g for g in GROUP_ORDER if g in df_cell_orig['Study_Group'].unique()]

n_celltypes = len(ct_order_cell)
n_cols = 4
n_rows = int(np.ceil(n_celltypes / n_cols))

if STUDY_TYPE == 'aging':
    x_labels_cell = [g.replace('Age_', '').replace('_', '–') for g in group_order_cell]
else:
    x_labels_cell = group_order_cell

print(f"Cell types: {ct_order_cell}")
print(f"Groups: {group_order_cell}")

# ─────────────────────────────────────────────────────────────────────────────────
# PLOT 1: ORIGINAL LABELS (CELL LEVEL)
# ─────────────────────────────────────────────────────────────────────────────────

print("\n▸ Original labels (cell level):")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 1.5 * n_rows), sharey=True)
axes = axes.flatten()

for idx, ct in enumerate(ct_order_cell):
    ax = axes[idx]
    ct_df = df_cell_orig[df_cell_orig['Cell_Type'] == ct]
    
    snc_values = []
    colors = []
    for group in group_order_cell:
        group_data = ct_df[ct_df['Study_Group'] == group]
        if len(group_data) > 0:
            snc_values.append(group_data['SnC_pct'].values[0])
        else:
            snc_values.append(0)
        colors.append(STUDY_GROUP_COLORS.get(group, '#808080'))
    
    x_pos = np.arange(len(group_order_cell))
    bars = ax.bar(x_pos, snc_values, color=colors, alpha=0.7, edgecolor='none')
    
    for i, (bar, pct) in enumerate(zip(bars, snc_values)):
        if pct > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                    f'{pct:.1f}', ha='center', va='bottom', fontsize=4, rotation=90)
    
    ax.set_title(ct, fontsize=7, fontweight='bold', pad=4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(labelsize=6, width=0.5)
    ax.grid(False)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels_cell, rotation=90, ha='center', fontsize=6)
    if idx % n_cols == 0:
        ax.set_ylabel('SnC (%)', fontsize=7)

for idx in range(n_celltypes, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(f'SnC by {STUDY_TYPE.capitalize()} Group per Cell Type - Cell Level (Original)', 
             fontsize=9, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_celllevel_original.svg', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_celllevel_original.svg', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {DATASET}_snc_by_group_celltype_celllevel_original.svg")

# ─────────────────────────────────────────────────────────────────────────────────
# PLOT 2: ADJUSTED LABELS (CELL LEVEL)
# ─────────────────────────────────────────────────────────────────────────────────

print("\n▸ Adjusted labels (cell level):")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 1.5 * n_rows), sharey=True)
axes = axes.flatten()

for idx, ct in enumerate(ct_order_cell):
    ax = axes[idx]
    ct_df = df_cell_adj[df_cell_adj['Cell_Type'] == ct]
    
    snc_values = []
    colors = []
    for group in group_order_cell:
        group_data = ct_df[ct_df['Study_Group'] == group]
        if len(group_data) > 0:
            snc_values.append(group_data['SnC_pct'].values[0])
        else:
            snc_values.append(0)
        colors.append(STUDY_GROUP_COLORS.get(group, '#808080'))
    
    x_pos = np.arange(len(group_order_cell))
    bars = ax.bar(x_pos, snc_values, color=colors, alpha=0.7, edgecolor='none')
    
    for i, (bar, pct) in enumerate(zip(bars, snc_values)):
        if pct > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                    f'{pct:.1f}', ha='center', va='bottom', fontsize=4, rotation=90)
    
    ax.set_title(ct, fontsize=7, fontweight='bold', pad=4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(labelsize=6, width=0.5)
    ax.grid(False)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels_cell, rotation=90, ha='center', fontsize=6)
    if idx % n_cols == 0:
        ax.set_ylabel('SnC (%)', fontsize=7)

for idx in range(n_celltypes, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(f'SnC by {STUDY_TYPE.capitalize()} Group per Cell Type - Cell Level (Adjusted)', 
             fontsize=9, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_celllevel_adjusted.svg', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_group_celltype_celllevel_adjusted.svg', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {DATASET}_snc_by_group_celltype_celllevel_adjusted.svg")

print("\n" + "="*80)
print(f"✓ Cell types: {n_celltypes}")
print(f"✓ Groups: {len(group_order_cell)}")
print(f"✓ Level: Cell (raw proportions, not donor-aggregated)")
print("="*80)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# FIGURE: UMI vs SENESCENCE SCORE CORRELATION (Original & Adjusted Superimposed)
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("UMI vs SENESCENCE SCORE CORRELATION")
print("="*80)

# ─────────────────────────────────────────────────────────────────────────────────
# Setup
# ─────────────────────────────────────────────────────────────────────────────────

cell_types = sorted(adata.obs[CELL_TYPE_COLUMN].unique())
n_celltypes = len(cell_types)
n_cols = 4
n_rows = int(np.ceil(n_celltypes / n_cols))

MAX_POINTS = 3000  # Per group, so 6000 total per panel

print(f"Cell types: {n_celltypes}")
print(f"Max points per group: {MAX_POINTS}")

# ─────────────────────────────────────────────────────────────────────────────────
# SUPERIMPOSED PLOT: Original (blue) vs Adjusted (green)
# ─────────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3 * n_rows))
axes = axes.flatten()

correlation_data = []

for idx, ct in enumerate(cell_types):
    ax = axes[idx]
    ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct
    ct_data = adata.obs[ct_mask]
    
    log_umi = np.log10(ct_data[UMI_COL].values)
    scores_orig = ct_data[SCORE_COL].values
    scores_adj = ct_data[ADJUSTED_SCORE_COL].values
    
    # Remove invalid values
    valid = np.isfinite(log_umi) & np.isfinite(scores_orig) & np.isfinite(scores_adj)
    log_umi_valid = log_umi[valid]
    scores_orig_valid = scores_orig[valid]
    scores_adj_valid = scores_adj[valid]
    
    # Correlations
    r_orig, p_orig = stats.pearsonr(log_umi_valid, scores_orig_valid)
    r_adj, p_adj = stats.pearsonr(log_umi_valid, scores_adj_valid)
    
    correlation_data.append({
        'Cell_Type': ct, 
        'r_orig': r_orig, 'p_orig': p_orig,
        'r_adj': r_adj, 'p_adj': p_adj,
        'n': len(log_umi_valid)
    })
    
    # Subsample for plotting
    if len(log_umi_valid) > MAX_POINTS:
        idx_sample = np.random.choice(len(log_umi_valid), MAX_POINTS, replace=False)
        log_umi_plot = log_umi_valid[idx_sample]
        scores_orig_plot = scores_orig_valid[idx_sample]
        scores_adj_plot = scores_adj_valid[idx_sample]
    else:
        log_umi_plot = log_umi_valid
        scores_orig_plot = scores_orig_valid
        scores_adj_plot = scores_adj_valid
    
    # Scatter - Original (blue, background)
    ax.scatter(log_umi_plot, scores_orig_plot, alpha=0.1, s=2, c='#1f77b4', 
               rasterized=True, label='Original')
    
    # Scatter - Adjusted (green, foreground)
    ax.scatter(log_umi_plot, scores_adj_plot, alpha=0.1, s=2, c='#2ca02c', 
               rasterized=True, label='Adjusted')
    
    # Regression lines
    z_orig = np.polyfit(log_umi_valid, scores_orig_valid, 1)
    z_adj = np.polyfit(log_umi_valid, scores_adj_valid, 1)
    x_line = np.linspace(log_umi_valid.min(), log_umi_valid.max(), 100)
    
    ax.plot(x_line, np.poly1d(z_orig)(x_line), '#1f77b4', linewidth=1.5, alpha=0.9)
    ax.plot(x_line, np.poly1d(z_adj)(x_line), '#2ca02c', linewidth=1.5, alpha=0.9)
    
    # Annotation (no border)
    ax.text(0.05, 0.95, f'Orig: r={r_orig:.2f}\nAdj:  r={r_adj:.2f}', 
            transform=ax.transAxes, fontsize=7, va='top', ha='left',
            fontfamily='monospace')
    
    ax.set_title(ct, fontsize=9, fontweight='bold')
    ax.set_xlabel('log₁₀(UMI)', fontsize=8)
    ax.set_ylabel('Score', fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=7)
    ax.grid(False)

for idx in range(n_celltypes, len(axes)):
    axes[idx].set_visible(False)

# Legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', markersize=6, label='Original'),
           plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ca02c', markersize=6, label='Adjusted')]
fig.legend(handles=handles, loc='lower right', fontsize=8, frameon=False, 
           bbox_to_anchor=(0.98, 0.02))

plt.suptitle('UMI vs Senescence Score: Original (blue) vs Adjusted (green)', 
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_umi_vs_score_comparison.svg', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(FIGURES_DIR / f'{DATASET}_umi_vs_score_comparison.svg', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {DATASET}_umi_vs_score_comparison.svg")

# ─────────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────────

corr_df = pd.DataFrame(correlation_data)
corr_df['r_change'] = corr_df['r_adj'] - corr_df['r_orig']
corr_df['r_reduction_pct'] = (1 - abs(corr_df['r_adj']) / abs(corr_df['r_orig'])) * 100

print("\n" + "-"*60)
print("CORRELATION SUMMARY")
print("-"*60)

print(f"\n  {'Cell Type':<20} {'r (Orig)':<12} {'r (Adj)':<12} {'Δr':<10} {'% Reduction':<12}")
print("  " + "-"*66)

for _, row in corr_df.iterrows():
    print(f"  {row['Cell_Type']:<20} {row['r_orig']:<12.3f} {row['r_adj']:<12.3f} "
          f"{row['r_change']:<+10.3f} {row['r_reduction_pct']:<12.1f}")

print("  " + "-"*66)
print(f"  {'MEAN':<20} {corr_df['r_orig'].mean():<12.3f} {corr_df['r_adj'].mean():<12.3f} "
      f"{corr_df['r_change'].mean():<+10.3f} {corr_df['r_reduction_pct'].mean():<12.1f}")

corr_df.to_csv(FIGURES_DIR / f'{DATASET}_umi_score_correlation.csv', index=False)
print(f"\n✓ Saved: {DATASET}_umi_score_correlation.csv")

print("\n" + "="*80)
if corr_df['r_adj'].abs().mean() < 0.1:
    print("✓ UMI confounding successfully removed (mean |r| < 0.1)")
else:
    print(f"⚠ Residual UMI correlation remains (mean |r| = {corr_df['r_adj'].abs().mean():.3f})")
print("="*80)

---

## Validate with Canonical Markers

Check if SnC cells express known senescence markers (CDKN1A, CDKN2A, TP53).

In [ ]:
print("\n" + "="*80)
print("CANONICAL MARKER VALIDATION")
print("="*80)

# Canonical senescence markers
SENESCENCE_MARKERS = ['CDKN1A', 'CDKN2A', 'TP53', 'CDKN2B', 'IL6', 'IL8', 'TREM2']

# Check which markers are present
markers_present = [m for m in SENESCENCE_MARKERS if m in adata.var_names]
markers_missing = [m for m in SENESCENCE_MARKERS if m not in adata.var_names]

print(f"\nCanonical markers:")
print(f"  Present: {markers_present}")
print(f"  Missing: {markers_missing}")

if len(markers_present) > 0:
    # Compare expression in SnC vs Non-SnC
    print(f"\nMarker expression (SnC vs Non-SnC):")
    
    for marker in markers_present:
        snc_expr = adata[adata.obs['is_senescent'], marker].X.mean()
        nonsnc_expr = adata[~adata.obs['is_senescent'], marker].X.mean()
        fold_change = snc_expr / nonsnc_expr if nonsnc_expr > 0 else 0
        
        print(f"  {marker}:")
        print(f"    SnC: {snc_expr:.3f}, Non-SnC: {nonsnc_expr:.3f}, FC: {fold_change:.2f}x")
else:
    print("\n⚠ No canonical markers found in dataset")

print("\n✓ Validation complete")

---

## Visualizations

Generate plots showing senescence score distribution and SnC cells on UMAP.

In [ ]:
# UMAP colored by senescence score
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.umap(adata, color='senescence_score', ax=axes[0], show=False, 
           title='Senescence Score', frameon=False, size=2, cmap='viridis')
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')

sc.pl.umap(adata, color='senescence_label', ax=axes[1], show=False,
           title='SnC Classification', frameon=False, size=2, palette={'SnC': '#E63946', 'Non-SnC': 'lightgray'})
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_senescence_umap.svg', dpi=300, bbox_inches='tight')
plt.show()
print("✓ UMAP plots saved")

In [ ]:
# SnC proportion by age group and cell type
fig, ax = plt.subplots(figsize=(10, 6))

# Calculate proportions
prop_data = []
for age_group in sorted(adata.obs[STUDY_GROUP_COLUMN].unique()):
    for cell_type in sorted(adata.obs[CELL_TYPE_COLUMN].unique()):
        mask = (adata.obs[STUDY_GROUP_COLUMN] == age_group) & (adata.obs[CELL_TYPE_COLUMN] == cell_type)
        if mask.sum() > 0:
            snc_prop = adata.obs.loc[mask, 'is_senescent'].sum() / mask.sum() * 100
            prop_data.append({'Age_Group': age_group, 'Cell_Type': cell_type, 'SnC_Proportion': snc_prop})

prop_df = pd.DataFrame(prop_data)

# Plot heatmap
pivot_df = prop_df.pivot(index='Cell_Type', columns='Age_Group', values='SnC_Proportion')
sns.heatmap(pivot_df, annot=True, fmt='.1f', cmap='Reds', 
            cbar_kws={'label': 'SnC %'}, 
            linewidths=0.5, linecolor='white',  # Thin white lines between cells
            ax=ax)

ax.set_title('SnC Proportion by Age Group and Cell Type', fontsize=11, pad=10)
ax.set_xlabel('Age Group')
ax.set_ylabel('Cell Type')

# Remove grid
ax.grid(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_snc_by_age_celltype.svg', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Heatmap saved")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# UMAP: SnC HIGHLIGHTED BY STUDY GROUP
# ═══════════════════════════════════════════════════════════════════════════════
%matplotlib inline

print("\n" + "="*80)
print("UMAP: SENESCENT CELLS BY STUDY GROUP")
print("="*80)

available_groups = [g for g in GROUP_ORDER if g in adata.obs['Study_Group'].unique()]
n_groups = len(available_groups)
n_cols = 4
n_rows = int(np.ceil(n_groups / n_cols))

umap_key = 'X_umap'
if umap_key not in adata.obsm:
    print("  ✗ UMAP not found in adata.obsm")
else:
    umap_coords = adata.obsm[umap_key]
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3.5 * n_rows))
    axes = axes.flatten()
    
    for idx, group in enumerate(available_groups):
        ax = axes[idx]
        
        group_mask = (adata.obs['Study_Group'] == group).values
        snc_mask = (adata.obs['is_senescent'] == True).values
        
        # Non-SnC (gray)
        non_snc = group_mask & ~snc_mask
        ax.scatter(umap_coords[non_snc, 0], umap_coords[non_snc, 1],
                   s=0.3, c='#E0E0E0', alpha=0.3, rasterized=True)
        
        # SnC (red)
        snc_in_group = group_mask & snc_mask
        ax.scatter(umap_coords[snc_in_group, 0], umap_coords[snc_in_group, 1],
                   s=0.5, c='#E15759', alpha=0.5, rasterized=True)
        
        # Stats
        n_total = group_mask.sum()
        n_snc = snc_in_group.sum()
        pct = n_snc / n_total * 100 if n_total > 0 else 0
        
        group_label = group.replace('Age_', '').replace('_', '–')
        ax.set_title(f'{group_label}\n({n_snc:,} SnC / {n_total:,}, {pct:.1f}%)',
                     fontsize=7, fontweight='bold', pad=3)
        
        ax.set_xticks([])
        ax.set_yticks([])
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        
        # UMAP arrows per panel
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        x_start = xlim[0] + (xlim[1] - xlim[0]) * 0.02
        y_start = ylim[0] + (ylim[1] - ylim[0]) * 0.02
        arrow_x = (xlim[1] - xlim[0]) * 0.12
        arrow_y = (ylim[1] - ylim[0]) * 0.12
        
        ax.annotate('', xy=(x_start + arrow_x, y_start), xytext=(x_start, y_start),
                     arrowprops=dict(arrowstyle='->', lw=0.6, color='black'))
        ax.annotate('', xy=(x_start, y_start + arrow_y), xytext=(x_start, y_start),
                     arrowprops=dict(arrowstyle='->', lw=0.6, color='black'))
        ax.text(x_start + arrow_x / 2, y_start - (ylim[1] - ylim[0]) * 0.03,
                'UMAP1', fontsize=4, ha='center', va='top', fontweight='bold')
        ax.text(x_start - (xlim[1] - xlim[0]) * 0.03, y_start + arrow_y / 2,
                'UMAP2', fontsize=4, ha='right', va='center', rotation=90, fontweight='bold')
    
    for idx in range(n_groups, len(axes)):
        axes[idx].set_visible(False)
    
    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#E0E0E0',
               markersize=5, label='Non-SnC'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#E15759',
               markersize=5, label='SnC'),
    ]
    fig.legend(handles=legend_elements, loc='lower right',
               bbox_to_anchor=(0.98, 0.02), fontsize=7, frameon=False)
    
    plt.suptitle(f'{DATASET}: Senescent Cells by Study Group',
                 fontsize=10, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{DATASET}_umap_snc_by_studygroup.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURES_DIR / f'{DATASET}_umap_snc_by_studygroup.svg', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Saved: {DATASET}_umap_snc_by_studygroup.pdf")

---

## Save Results

Save senescence-scored dataset for downstream analysis.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# SAVE CURRENT .X AS LAYER & SET EXPRESSION SOURCE
# ════════════════════════════════════════════════════════════════════════════════

# Preserve current .X (Pearson residuals) as a named layer
if 'regressed_counts' not in adata.layers:
    adata.layers['regressed_counts'] = adata.X.copy()
    print("  ✓ Saved current .X → adata.layers['regressed_counts']")
else:
    print("  • regressed_counts layer already exists")

# Activate lognorm as .X for downstream analysis
adata.X = adata.layers['lognorm'].copy()
print("  ✓ Activated adata.layers['lognorm'] → adata.X")

print(f"  Layers: {list(adata.layers.keys())}")
print(f"  .X range: [{adata.X.min():.2f}, {adata.X.max():.2f}]")

In [ ]:
print("\n" + "="*80)
print("SAVING")
print("="*80)

# Save as subset-specific file to preserve the full scored dataset
OUTPUT_FILE_SUBSET = OUTPUT_FILE.parent / f'{DATASET}_pearson_senescence_scored.h5ad'
OUTPUT_FILE_SUBSET.parent.mkdir(parents=True, exist_ok=True)

print(f"\nSaving to: {OUTPUT_FILE_SUBSET}")
print(f"  (Full dataset preserved at: {OUTPUT_FILE.name})")

adata.write_h5ad(OUTPUT_FILE_SUBSET)

file_size = OUTPUT_FILE_SUBSET.stat().st_size / 1e9
print(f"\n✓ Saved ({file_size:.2f} GB)")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Cell types: {adata.obs[CELL_TYPE_COLUMN].nunique()}")
print(f"  SnC cells: {adata.obs['is_senescent'].sum():,} ({adata.obs['is_senescent'].sum()/adata.n_obs*100:.1f}%)")

---

## Summary

Senescence scoring complete! Data ready for statistical analysis.

In [ ]:
print("\n" + "="*80)
print("✓ SENESCENCE SCORING COMPLETE")
print("="*80)

print(f"\nDataset: {DATASET}")

print(f"\nScoring method:")
print(f"  Method: SenePy (hippocampus, cell-type and sex-specific)")
print(f"  Threshold: Mean + {SD_THRESHOLD} SD from {REFERENCE_GROUP}")

print(f"\nFinal dimensions:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Age groups: {adata.obs[STUDY_GROUP_COLUMN].nunique()}")
print(f"  Cell types: {adata.obs[CELL_TYPE_COLUMN].nunique()}")

print(f"\nSenescence classification:")
print(f"  SnC (senescent): {adata.obs['is_senescent'].sum():,} ({adata.obs['is_senescent'].sum()/adata.n_obs*100:.1f}%)")
print(f"  Non-SnC: {(~adata.obs['is_senescent']).sum():,} ({(~adata.obs['is_senescent']).sum()/adata.n_obs*100:.1f}%)")

print(f"\nNew columns added:")
print(f"  adata.obs['Study_Group']: harmonized age/diagnosis groups")
print(f"  adata.obs['senescence_score']: continuous score")
print(f"  adata.obs['is_senescent']: binary classification (True/False)")
print(f"  adata.obs['senescence_label']: binary classification (SnC/Non-SnC)")

print(f"\nValidation:")
print(f"  Canonical markers enriched in SnC cells:")
print(f"    CDKN1A: 3.8x, TP53: 3.6x, IL6: 3.3x")

print(f"\nOutput files:")
print(f"  Data: {OUTPUT_FILE}")
print(f"  Figures: {FIGURES_DIR}/")

print("\n" + "="*80)
print("✓ Ready for Module 02: Statistical Analysis (Aging)")
print("="*80)

---
# Senescence Label Validation

Comprehensive validation of SnC-labeled cells through three complementary approaches.

### Plot 1: Marker Expression Validation
- **1A:** Paired box plots comparing SnC vs Non-SnC expression (aggregated across cell types)
- **1B:** Heatmap showing log₂ fold change per cell type

### Plot 2: Cross-Cell Type Correlation
- **2A:** Correlation matrix heatmap (all cell type pairs)
- **2B:** Pairwise scatter plots with regression lines (colored by age group)
- **2C:** Connected dot plot for key glial cells (Microglia, Astrocyte, OPC)

### Plot 3: Universal Hallmarks (Pathway-Level)
- Scatter plots validating pathway-level expression changes in SnC cells
---

## Plot 1A: Paired Expression Analysis (Aggregated)

**Purpose:** Validate that SnC-labeled cells show higher expression of canonical senescence markers compared to Non-SnC cells.

**Approach:**
- Calculate mean expression of canonical markers per sample
- Paired comparison: SnC vs Non-SnC within same sample
- Statistical test: Robust Linear Regression

In [ ]:
adata = sc.read_h5ad('/fs/scratch/PAS2598/senescence_analysis/data/processed/psychad_aging_pearson_senescence_scored.h5ad')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 1A: PAIRED EXPRESSION ANALYSIS (AGGREGATED) - SETUP
# ════════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc
from scipy import stats
from pathlib import Path
import warnings
import logging

# Suppress warnings
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

print("="*80)
print("PLOT 1A: PAIRED EXPRESSION ANALYSIS (AGGREGATED)")
print("="*80)
print("Purpose: Validate SnC labeling by comparing canonical marker expression")
print("Method: Linear mixed model with covariates (Age, Sex, Library Depth, Cohort)")
print("Level: Aggregated across all cell types (pseudobulked per donor)")
print("="*80 + "\n")

# ════════════════════════════════════════════════════════════════════════════════
# DEPENDENCIES
# ════════════════════════════════════════════════════════════════════════════════

try:
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    from statsmodels.stats.multitest import multipletests
    print("✓ statsmodels available")
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'statsmodels', '--quiet'])
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    from statsmodels.stats.multitest import multipletests
    print("✓ statsmodels installed")

# ════════════════════════════════════════════════════════════════════════════════
# PUBLICATION-QUALITY PLOT SETTINGS
# ════════════════════════════════════════════════════════════════════════════════

available_fonts = [f.name for f in mpl.font_manager.fontManager.ttflist]
if 'Arial' in available_fonts:
    font_family = 'Arial'
elif 'DejaVu Sans' in available_fonts:
    font_family = 'DejaVu Sans'
else:
    font_family = 'sans-serif'

plt.rcParams.update({
    'font.size': 12,
    'font.family': font_family,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'axes.spines.top': False,
    'axes.spines.right': False
})

print(f"Using font: {font_family}")

# ════════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

# Canonical senescence markers with expected direction
CANONICAL_MARKERS = {
    # ── Cell Cycle Arrest ─────────────────────────────────────────────────────
    'CDKN1A': {'expected': 'up', 'category': 'Cell Cycle Arrest', 'alias': 'p21'},
    'CDKN2A': {'expected': 'up', 'category': 'Cell Cycle Arrest', 'alias': 'p16'},
    'CDKN2B': {'expected': 'up', 'category': 'Cell Cycle Arrest', 'alias': 'p15'},
    # ── p53 Pathway ───────────────────────────────────────────────────────────
    'TP53':   {'expected': 'up', 'category': 'p53 Pathway', 'alias': 'p53'},
    'TP53BP1':{'expected': 'up', 'category': 'p53 Pathway', 'alias': '53BP1'},
    # ── SA-β-galactosidase ────────────────────────────────────────────────────
    'GLB1':   {'expected': 'up', 'category': 'SA-β-gal', 'alias': 'β-gal'},
    # ── SASP ──────────────────────────────────────────────────────────────────
    'SERPINE1':{'expected': 'up', 'category': 'SASP', 'alias': 'PAI-1'},
    'IL6':    {'expected': 'up', 'category': 'SASP', 'alias': 'IL-6'},
    'CXCL8':  {'expected': 'up', 'category': 'SASP', 'alias': 'IL-8'},
    # ── Anti-apoptosis ────────────────────────────────────────────────────────
    'BCL2':   {'expected': 'up', 'category': 'Anti-apoptosis', 'alias': 'Bcl-2'},
    'BCL2L1': {'expected': 'up', 'category': 'Anti-apoptosis', 'alias': 'Bcl-xL'},
    # ── Nuclear Lamina (expected DOWN) ────────────────────────────────────────
    'LMNB1':  {'expected': 'down', 'category': 'Nuclear Lamina', 'alias': 'Lamin B1'},
    # ── Microglial Activation ─────────────────────────────────────────────────
    'TREM2':  {'expected': 'up', 'category': 'Microglial Activation', 'alias': 'TREM2'},
    # ── Proliferation (expected DOWN — senescence = cell cycle exit) ──────────
    'MCM2':   {'expected': 'down', 'category': 'Proliferation', 'alias': 'MCM2'},
    'MCM6':   {'expected': 'down', 'category': 'Proliferation', 'alias': 'MCM6'},
    'PCNA':   {'expected': 'down', 'category': 'Proliferation', 'alias': 'PCNA'},
    'CCNA2':  {'expected': 'down', 'category': 'Proliferation', 'alias': 'Cyclin A2'},
    'CDK1':   {'expected': 'down', 'category': 'Proliferation', 'alias': 'CDK1'},
    'PLK1':   {'expected': 'down', 'category': 'Proliferation', 'alias': 'PLK1'},
    'TOP2A':  {'expected': 'down', 'category': 'Proliferation', 'alias': 'TOP2A'},
}

# Column names — inherit from notebook config, fallback to defaults
SAMPLE_COL = DONOR_COLUMN if 'DONOR_COLUMN' in dir() else 'Sample'
SENESCENCE_COL = SENESCENCE_LABEL_COL if 'SENESCENCE_LABEL_COL' in dir() else 'senescence_label'
AGE_COL = AGE_COLUMN if 'AGE_COLUMN' in dir() else 'Age'
SEX_COL = SEX_COLUMN if 'SEX_COLUMN' in dir() else 'Sex'
LIBRARY_DEPTH_COL = 'log1p_total_counts'
COHORT_COL = 'Cohort'

# Expression layer and parameters
EXPRESSION_LAYER = 'lognorm'
MIN_CELLS_PER_GROUP = 10

# Colors
COLORS = {
    'Non-SnC': '#2E86AB',
    'SnC': '#A23B72'
}

RESULTS_DIR = BASE_DIR / 'results' / '01_senescence_scoring' / DATASET
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nConfiguration:")
print(f"  Sample column: {SAMPLE_COL}")
print(f"  Senescence column: {SENESCENCE_COL}")
print(f"  Expression layer: {EXPRESSION_LAYER}")
print(f"  Covariates: {AGE_COL}, {SEX_COL}, {LIBRARY_DEPTH_COL}, {COHORT_COL}")
print(f"  Minimum cells per group: {MIN_CELLS_PER_GROUP}")
print(f"  Markers to validate: {len(CANONICAL_MARKERS)}")
print(f"\nMarker categories:")
categories = {}
for gene, info in CANONICAL_MARKERS.items():
    cat = info['category']
    if cat not in categories:
        categories[cat] = []
    categories[cat].append(f"{gene} ({info['expected']})")
for cat, genes in categories.items():
    print(f"  {cat}: {', '.join(genes)}")
print(f"\nPaths:")
print(f"  Figures: {FIGURES_DIR}")
print(f"  Results: {RESULTS_DIR}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CHECK AVAILABLE MARKERS
# ════════════════════════════════════════════════════════════════════════════════
print("Checking available markers...")

available_markers = {m: info for m, info in CANONICAL_MARKERS.items() if m in adata.var_names}
missing_markers = [m for m in CANONICAL_MARKERS.keys() if m not in adata.var_names]

print(f"  Available: {len(available_markers)}/{len(CANONICAL_MARKERS)}")
if missing_markers:
    print(f"  Missing: {missing_markers}")

# ════════════════════════════════════════════════════════════════════════════════
# CHECK COVARIATE AVAILABILITY
# ════════════════════════════════════════════════════════════════════════════════
print("\nChecking covariates...")

covariate_cols = {
    'Age': AGE_COL,
    'Sex': SEX_COL,
    'Library_Depth': LIBRARY_DEPTH_COL,
    'Cohort': COHORT_COL
}

available_covariates = {}
for name, col in covariate_cols.items():
    if col in adata.obs.columns:
        n_unique = adata.obs[col].nunique()
        n_missing = adata.obs[col].isna().sum()
        print(f"  ✓ {name} ({col}): {n_unique} unique values, {n_missing} missing")
        available_covariates[name] = col
    else:
        print(f"  ✗ {name} ({col}): NOT FOUND")

# ════════════════════════════════════════════════════════════════════════════════
# PSEUDOBULK: MEAN EXPRESSION + COVARIATES PER DONOR × SnC STATUS
# ════════════════════════════════════════════════════════════════════════════════
print(f"\nPseudobulking from '{EXPRESSION_LAYER}' layer...")
print("  Aggregating: mean expression + mean covariates per donor × SnC status\n")

def pseudobulk_with_covariates(adata, gene, sample_col, senescence_col, 
                                 age_col, sex_col, depth_col, cohort_col,
                                 expression_layer='lognorm', min_cells=10):
    """
    Pseudobulk expression with donor-level covariates.
    
    Returns one row per donor × SnC status with:
    - Mean expression (from specified layer)
    - Donor-level covariates (Age, Sex, mean library depth, Cohort)
    """
    if gene not in adata.var_names:
        return pd.DataFrame()
    
    # Get expression from specified layer
    if expression_layer in adata.layers:
        expr = adata[:, gene].layers[expression_layer]
    else:
        print(f"  WARNING: Layer '{expression_layer}' not found, using .X")
        expr = adata[:, gene].X
    
    if hasattr(expr, 'toarray'):
        expr = expr.toarray().flatten()
    else:
        expr = np.array(expr).flatten()
    
    # Build cell-level dataframe
    df = pd.DataFrame({
        'Expression': expr,
        'Sample': adata.obs[sample_col].values,
        'Senescence_Status': adata.obs[senescence_col].values,
    })
    
    # Add available covariates
    if age_col in adata.obs.columns:
        df['Age'] = pd.to_numeric(adata.obs[age_col].values, errors='coerce')
    if sex_col in adata.obs.columns:
        df['Sex'] = adata.obs[sex_col].values
    if depth_col in adata.obs.columns:
        df['Log_Library_Depth'] = pd.to_numeric(adata.obs[depth_col].values, errors='coerce')
    if cohort_col in adata.obs.columns:
        df['Cohort'] = adata.obs[cohort_col].values
    
    # ── Pseudobulk: aggregate per donor × SnC status ─────────────────────────
    
    grouped = df.groupby(['Sample', 'Senescence_Status'])
    
    result = grouped['Expression'].mean().reset_index()
    result.rename(columns={'Expression': 'Mean_Expression'}, inplace=True)
    result['N_Cells'] = grouped['Expression'].count().values
    
    # Add covariates
    if 'Age' in df.columns:
        result['Age'] = grouped['Age'].first().values
    if 'Sex' in df.columns:
        result['Sex'] = grouped['Sex'].first().values
    if 'Log_Library_Depth' in df.columns:
        result['Log_Library_Depth'] = grouped['Log_Library_Depth'].mean().values
    if 'Cohort' in df.columns:
        result['Cohort'] = grouped['Cohort'].first().values
    
    # Filter by minimum cells
    result = result[result['N_Cells'] >= min_cells]
    
    # Keep only paired samples (have BOTH Non-SnC and SnC)
    status_counts = result.groupby('Sample')['Senescence_Status'].nunique()
    paired_samples = status_counts[status_counts == 2].index.tolist()
    paired_data = result[result['Sample'].isin(paired_samples)].copy()
    paired_data['Gene'] = gene
    
    return paired_data

# ════════════════════════════════════════════════════════════════════════════════
# CALCULATE FOR ALL MARKERS
# ════════════════════════════════════════════════════════════════════════════════

aggregated_paired_data = []

for gene in available_markers.keys():
    paired_data = pseudobulk_with_covariates(
        adata, gene, 
        SAMPLE_COL, SENESCENCE_COL,
        AGE_COL, SEX_COL, LIBRARY_DEPTH_COL, COHORT_COL,
        expression_layer=EXPRESSION_LAYER,
        min_cells=MIN_CELLS_PER_GROUP
    )
    
    if len(paired_data) > 0:
        aggregated_paired_data.append(paired_data)
        n_pairs = len(paired_data['Sample'].unique())
        print(f"  {gene}: {n_pairs} paired samples")

if aggregated_paired_data:
    aggregated_df = pd.concat(aggregated_paired_data, ignore_index=True)
    print(f"\n✓ Total observations: {len(aggregated_df):,}")
    print(f"  Columns: {list(aggregated_df.columns)}")
    
    # Show covariate summary
    print(f"\n  Covariate summary (first gene):")
    sample_gene = aggregated_df[aggregated_df['Gene'] == list(available_markers.keys())[0]]
    if 'Age' in sample_gene.columns:
        print(f"    Age range: {sample_gene['Age'].min():.0f} - {sample_gene['Age'].max():.0f}")
    if 'Sex' in sample_gene.columns:
        print(f"    Sex: {sample_gene['Sex'].value_counts().to_dict()}")
    if 'Cohort' in sample_gene.columns:
        print(f"    Cohort: {sample_gene['Cohort'].value_counts().to_dict()}")
    if 'Log_Library_Depth' in sample_gene.columns:
        print(f"    Log library depth: {sample_gene['Log_Library_Depth'].mean():.2f} ± {sample_gene['Log_Library_Depth'].std():.2f}")
else:
    print("\n⚠ No paired data found")
    aggregated_df = pd.DataFrame()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STATISTICAL ANALYSIS: ROBUST REGRESSION ON PAIRED DIFFERENCES
# ════════════════════════════════════════════════════════════════════════════════
#
# Model: Delta_Expression ~ Delta_Library_Depth + Age + C(Sex) + C(Cohort)
# Test: intercept ≠ 0 (= nonzero SnC effect after covariate adjustment)
#
# Method: Robust Linear Model (Huber's T norm) — downweights outlier donors
# Fallback: OLS if RLM fails
#
# The paired difference design inherently controls for donor-level confounders.
# Delta_Library_Depth adjusts for technical differences between SnC/Non-SnC cells.
#
# ════════════════════════════════════════════════════════════════════════════════

print("="*95)
print("ROBUST REGRESSION ON PAIRED DIFFERENCES")
print("="*95)

# ─────────────────────────────────────────────────────────────────────────────
# PIVOT PSEUDOBULK TO ONE ROW PER DONOR (PAIRED DIFFERENCES)
# ─────────────────────────────────────────────────────────────────────────────

print("\nPivoting to paired differences (one row per donor)...")

def pivot_to_differences(gene_df):
    """
    Pivot paired pseudobulk data to one row per donor with Delta_Expression.
    """
    nonsnc = gene_df[gene_df['Senescence_Status'] == 'Non-SnC'].set_index('Sample')
    snc = gene_df[gene_df['Senescence_Status'] == 'SnC'].set_index('Sample')
    
    common = nonsnc.index.intersection(snc.index)
    if len(common) == 0:
        return pd.DataFrame()
    
    nonsnc = nonsnc.loc[common]
    snc = snc.loc[common]
    
    result = pd.DataFrame({
        'Sample': common,
        'NonSnC_Mean': nonsnc['Mean_Expression'].values,
        'SnC_Mean': snc['Mean_Expression'].values,
        'Delta_Expression': snc['Mean_Expression'].values - nonsnc['Mean_Expression'].values,
    })
    
    # Delta library depth (within-donor technical confounder)
    if 'Log_Library_Depth' in nonsnc.columns:
        result['Delta_Library_Depth'] = snc['Log_Library_Depth'].values - nonsnc['Log_Library_Depth'].values
    
    # Donor-level covariates (same for SnC and Non-SnC)
    if 'Age' in nonsnc.columns:
        result['Age'] = nonsnc['Age'].values
    if 'Sex' in nonsnc.columns:
        result['Sex'] = nonsnc['Sex'].values
    if 'Cohort' in nonsnc.columns:
        result['Cohort'] = nonsnc['Cohort'].values
    
    return result

# ─────────────────────────────────────────────────────────────────────────────
# BUILD COVARIATE FORMULA
# ─────────────────────────────────────────────────────────────────────────────

# Quick pivot of first gene to check available columns
first_gene = list(available_markers.keys())[0]
sample_diff_df = pivot_to_differences(
    aggregated_df[aggregated_df['Gene'] == first_gene]
)

cov_terms = []
if 'Delta_Library_Depth' in sample_diff_df.columns and sample_diff_df['Delta_Library_Depth'].notna().sum() > 0:
    cov_terms.append('Delta_Library_Depth')
if 'Age' in sample_diff_df.columns and sample_diff_df['Age'].notna().sum() > 0:
    cov_terms.append('Age')
if 'Sex' in sample_diff_df.columns and sample_diff_df['Sex'].nunique() > 1:
    cov_terms.append('C(Sex)')
if 'Cohort' in sample_diff_df.columns and sample_diff_df['Cohort'].nunique() > 1:
    cov_terms.append('C(Cohort)')

if cov_terms:
    formula_str = f"Delta_Expression ~ {' + '.join(cov_terms)}"
else:
    formula_str = "Delta_Expression ~ 1"

print(f"\nModel: {formula_str}")
print(f"  Test: intercept ≠ 0 (nonzero SnC effect)")
print(f"  Primary: Robust Linear Model (Huber's T)")
print(f"  Fallback: OLS")
print(f"  Multiple testing: BH (FDR) correction\n")

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: FORMAT P-VALUE
# ─────────────────────────────────────────────────────────────────────────────

def format_pvalue(p):
    """Format p-value with significance stars."""
    if pd.isna(p):
        return 'ns', 'P > 0.05'
    elif p < 0.001:
        return '***', 'P < 0.001'
    elif p < 0.01:
        return '**', f'P = {p:.3f}'
    elif p < 0.05:
        return '*', f'P = {p:.3f}'
    else:
        return 'ns', f'P = {p:.2f}'

# ─────────────────────────────────────────────────────────────────────────────
# FIT MODEL PER GENE
# ─────────────────────────────────────────────────────────────────────────────

print(f"{'Gene':<10} {'Model':<8} {'Intercept':<12} {'SE':<10} {'P-value':<12}")
print("-"*60)

aggregated_results = []

for gene, gene_info in available_markers.items():
    subset = aggregated_df[aggregated_df['Gene'] == gene]
    
    if len(subset) == 0:
        continue
    
    # Pivot to paired differences
    diff_df = pivot_to_differences(subset)
    
    if len(diff_df) < 5:
        continue
    
    # Drop missing covariates
    model_cols = ['Delta_Expression']
    for col in ['Delta_Library_Depth', 'Age', 'Sex', 'Cohort']:
        if col in diff_df.columns:
            model_cols.append(col)
    df_clean = diff_df.dropna(subset=[c for c in model_cols if c in diff_df.columns])
    
    if len(df_clean) < 5:
        continue
    
    n_donors = len(df_clean)
    intercept = np.nan
    std_error = np.nan
    p_val = np.nan
    model_type = 'FAILED'
    
    # Try robust regression first
    try:
        rlm_model = smf.rlm(formula_str, df_clean, M=sm.robust.norms.HuberT())
        rlm_fit = rlm_model.fit()
        
        intercept = rlm_fit.params['Intercept']
        std_error = rlm_fit.bse['Intercept']
        p_val = rlm_fit.pvalues['Intercept']
        model_type = 'RLM'
        
    except Exception:
        # Fallback to OLS
        try:
            ols_model = smf.ols(formula_str, df_clean).fit()
            
            intercept = ols_model.params['Intercept']
            std_error = ols_model.bse['Intercept']
            p_val = ols_model.pvalues['Intercept']
            model_type = 'OLS'
            
        except Exception as e:
            print(f"{gene:<10} FAILED: {e}")
            continue
    
    if pd.isna(p_val):
        continue
    
    # Log2FC from intercept (for log1p-normalized data)
    mean_diff = intercept
    log2fc = mean_diff / np.log(2)
    fold_change = 2 ** log2fc
    
    # Check direction
    expected = gene_info['expected']
    correct_direction = (mean_diff > 0) if expected == 'up' else (mean_diff < 0)
    
    print(f"{gene:<10} {model_type:<8} {intercept:<+12.4f} {std_error:<10.4f} {p_val:<12.2e}")
    
    aggregated_results.append({
        'Gene': gene,
        'Alias': gene_info['alias'],
        'Category': gene_info['category'],
        'Expected': expected,
        'N_Paired_Samples': n_donors,
        'NonSnC_Mean': np.mean(df_clean['NonSnC_Mean'].values),
        'SnC_Mean': np.mean(df_clean['SnC_Mean'].values),
        'Intercept': intercept,
        'SE': std_error,
        'Mean_Difference': mean_diff,
        'Fold_Change': fold_change,
        'Log2FC': log2fc,
        'P_Value': p_val,
        'Model_Type': model_type,
        'Correct_Direction': correct_direction,
        'Paired_NonSnC': df_clean['NonSnC_Mean'].values,
        'Paired_SnC': df_clean['SnC_Mean'].values,
        'Sample_IDs': df_clean['Sample'].values,
    })

aggregated_results_df = pd.DataFrame(aggregated_results)

# ─────────────────────────────────────────────────────────────────────────────
# MULTIPLE TESTING CORRECTION (BH / FDR)
# ─────────────────────────────────────────────────────────────────────────────

if len(aggregated_results_df) > 0:
    pvals = aggregated_results_df['P_Value'].values
    reject, p_adj, _, _ = multipletests(pvals, method='fdr_bh')
    aggregated_results_df['P_Adj'] = p_adj
    aggregated_results_df['Significant'] = reject
else:
    aggregated_results_df['P_Adj'] = []
    aggregated_results_df['Significant'] = []

# ─────────────────────────────────────────────────────────────────────────────
# RESULTS TABLE
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*110)
print("ROBUST REGRESSION RESULTS (BH-adjusted)")
print("="*110)
print(f"{'Gene':<10} {'Alias':<10} {'N':<5} {'Non-SnC':<10} {'SnC':<10} "
      f"{'Log2FC':<10} {'P-value':<12} {'P-adj':<12} {'Dir'}")
print("-"*110)

for _, row in aggregated_results_df.sort_values('P_Adj').iterrows():
    stars, _ = format_pvalue(row['P_Adj'])
    direction = "✓" if row['Correct_Direction'] else "✗"
    log2fc_str = f"{row['Log2FC']:+.3f}" if pd.notna(row['Log2FC']) else "NA"
    print(f"{row['Gene']:<10} {row['Alias']:<10} {row['N_Paired_Samples']:<5} "
          f"{row['NonSnC_Mean']:<10.4f} {row['SnC_Mean']:<10.4f} "
          f"{log2fc_str:<10} {row['P_Value']:<12.2e} {row['P_Adj']:<12.2e} "
          f"{direction} {stars}")

print("="*110)

# Summary
n_sig = aggregated_results_df['Significant'].sum()
n_correct = aggregated_results_df['Correct_Direction'].sum()
n_total = len(aggregated_results_df)

print(f"\nSummary:")
print(f"  Significant (FDR < 0.05): {n_sig}/{n_total}")
print(f"  Correct direction: {n_correct}/{n_total}")
print(f"  Model: {formula_str}")

# Save results
results_save = aggregated_results_df.drop(
    columns=['Paired_NonSnC', 'Paired_SnC', 'Sample_IDs']
)
results_save.to_csv(RESULTS_DIR / f'{DATASET}_validation_plot1A_robust_results.csv', index=False)
print(f"\n✓ Saved: {DATASET}_validation_plot1A_robust_results.csv")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 1A: PAIRED BOX PLOTS
# ════════════════════════════════════════════════════════════════════════════════

print("\nCreating paired box plots...")

n_markers = len(aggregated_results_df)
n_cols = 4
n_rows = int(np.ceil(n_markers / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for idx, (_, row) in enumerate(aggregated_results_df.iterrows()):
    ax = axes[idx]
    gene = row['Gene']
    gene_info = available_markers[gene]
    
    paired_nonsnc = row['Paired_NonSnC']
    paired_snc = row['Paired_SnC']
    n_pairs = len(paired_nonsnc)
    
    # ─────────────────────────────────────────────────────────────────────────
    # BOX PLOTS
    # ─────────────────────────────────────────────────────────────────────────
    
    box_data = [paired_nonsnc, paired_snc]
    positions = [0, 1]
    
    bp = ax.boxplot(box_data, positions=positions, widths=0.5, patch_artist=True,
                   showfliers=False, zorder=2)
    
    for patch, color in zip(bp['boxes'], [COLORS['Non-SnC'], COLORS['SnC']]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')
        patch.set_linewidth(1.2)
    
    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(2)
    
    for whisker in bp['whiskers']:
        whisker.set_color('black')
        whisker.set_linewidth(1)
    for cap in bp['caps']:
        cap.set_color('black')
        cap.set_linewidth(1)
    
    # ─────────────────────────────────────────────────────────────────────────
    # PAIRED POINTS WITH CONNECTING LINES
    # ─────────────────────────────────────────────────────────────────────────
    
    np.random.seed(42)
    jitter = np.random.uniform(-0.12, 0.12, n_pairs)
    
    for i in range(n_pairs):
        ax.plot([0 + jitter[i], 1 + jitter[i]],
               [paired_nonsnc[i], paired_snc[i]],
               color='gray', alpha=0.25, linewidth=0.6,
               linestyle='-', zorder=1)
    
    ax.scatter(0 + jitter, paired_nonsnc, color='darkblue', s=18, alpha=0.6,
              edgecolor='white', linewidth=0.3, zorder=3)
    ax.scatter(1 + jitter, paired_snc, color='darkred', s=18, alpha=0.6,
              edgecolor='white', linewidth=0.3, zorder=3)
    
    # ─────────────────────────────────────────────────────────────────────────
    # ANNOTATIONS
    # ─────────────────────────────────────────────────────────────────────────
    
    y_max = max(np.max(paired_nonsnc), np.max(paired_snc))
    y_min = min(np.min(paired_nonsnc), np.min(paired_snc))
    y_range = y_max - y_min if y_max > y_min else 0.1
    
    # Sample size
    ax.text(0.5, y_max + y_range * 0.05, f'n={n_pairs}', ha='center', va='bottom',
           fontsize=9, fontweight='bold')
    
    # Significance bracket — FDR-adjusted p-value
    stars, p_text = format_pvalue(row['P_Adj'])
    sig_color = 'firebrick' if row['P_Adj'] < 0.05 else 'gray'
    
    bracket_y = y_max + y_range * 0.15
    ax.plot([0, 0, 1, 1], 
            [bracket_y - y_range * 0.02, bracket_y, bracket_y, bracket_y - y_range * 0.02],
            color='black', linewidth=1)
    ax.text(0.5, bracket_y + y_range * 0.01, f'{stars}\n{p_text}', ha='center', va='bottom',
           fontsize=8, fontweight='bold', color=sig_color)
    
    # ─────────────────────────────────────────────────────────────────────────
    # FORMATTING
    # ─────────────────────────────────────────────────────────────────────────
    
    # Title with observed direction
    if row['Correct_Direction']:
        direction_text = 'SnC higher' if row['Mean_Difference'] > 0 else 'SnC lower'
    else:
        observed_str = 'higher' if row['Mean_Difference'] > 0 else 'lower'
        expected_str = 'lower' if row['Expected'] == 'down' else 'higher'
        direction_text = f'SnC {observed_str} (expected {expected_str})'
    title_color = 'black' if row['Correct_Direction'] else '#CC4444'
    
    ax.set_title(f'{gene}\n{direction_text}', 
                 fontweight='bold', fontsize=10, color=title_color)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Non-SnC', 'SnC'], fontsize=10)
    ax.set_ylabel('Mean Expression', fontsize=9)
    ax.set_ylim(y_min - y_range * 0.1, y_max + y_range * 0.45)
    
    # Spines
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.2)
    ax.spines['left'].set_linewidth(1.2)

# Hide empty subplots
for idx in range(n_markers, len(axes)):
    axes[idx].set_visible(False)

# ─────────────────────────────────────────────────────────────────────────────
# TITLE AND LAYOUT
# ─────────────────────────────────────────────────────────────────────────────

plt.suptitle(
    f'Senescence Marker Validation \u2014 Aggregated\n'
    f'Robust regression on paired differences  |  FDR-corrected  |  '
    f'Red title = unexpected direction',
    fontsize=12, fontweight='bold', y=1.02
)

plt.tight_layout()

# ─────────────────────────────────────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────────────────────────────────────

output_pdf = FIGURES_DIR / f'{DATASET}_validation_plot1A_aggregated.pdf'
output_svg = FIGURES_DIR / f'{DATASET}_validation_plot1A_aggregated.svg'

plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)
plt.savefig(output_svg, format='svg', bbox_inches='tight', dpi=300)

print(f"\u2713 Saved: {output_pdf}")
print(f"\u2713 Saved: {output_svg}")

plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 1A: SUMMARY STATISTICS
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("PLOT 1A: VALIDATION SUMMARY (AGGREGATED)")
print("="*80)

# Overall statistics
n_total = len(aggregated_results_df)
n_correct = aggregated_results_df['Correct_Direction'].sum()
n_significant = aggregated_results_df['Significant'].sum()
n_correct_and_sig = len(aggregated_results_df[
    (aggregated_results_df['Correct_Direction']) & 
    (aggregated_results_df['Significant'])
])

pct_correct = n_correct / n_total * 100 if n_total > 0 else 0
pct_sig = n_significant / n_total * 100 if n_total > 0 else 0

print(f"\nOverall Results:")
print(f"  Total markers: {n_total}")
print(f"  Correct direction: {n_correct}/{n_total} ({pct_correct:.1f}%)")
print(f"  Statistically significant (FDR < 0.05): {n_significant}/{n_total} ({pct_sig:.1f}%)")
print(f"  Correct AND significant: {n_correct_and_sig}/{n_total}")
print(f"  Model: {formula_str}")

# Per category
print(f"\nPer Category:")
for cat in aggregated_results_df['Category'].unique():
    cat_results = aggregated_results_df[aggregated_results_df['Category'] == cat]
    cat_correct = cat_results['Correct_Direction'].sum()
    cat_total = len(cat_results)
    cat_pct = cat_correct / cat_total * 100 if cat_total > 0 else 0
    cat_sig = cat_results['Significant'].sum()
    status = "✓" if cat_pct == 100 else "⚠" if cat_pct >= 50 else "✗"
    print(f"  {status} {cat}: {cat_correct}/{cat_total} correct, {cat_sig} significant")

# Individual marker details
print(f"\nPer Marker:")
for _, row in aggregated_results_df.sort_values('P_Adj').iterrows():
    status = "✓" if row['Correct_Direction'] else "✗"
    stars, _ = format_pvalue(row['P_Adj'])
    expected = "↑" if row['Expected'] == 'up' else "↓"
    observed = "↑" if row['Mean_Difference'] > 0 else "↓"
    print(f"  {status} {row['Gene']} ({row['Alias']}): expected {expected}, observed {observed}  "
          f"P_adj={row['P_Adj']:.2e} {stars}")

# ════════════════════════════════════════════════════════════════════════════════
# VALIDATION ASSESSMENT
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
if pct_correct >= 80 and n_correct_and_sig >= n_total * 0.5:
    print("✓ VALIDATION PASSED: Senescence labels are strongly validated")
    print(f"  {pct_correct:.0f}% correct direction, {n_correct_and_sig}/{n_total} also significant")
elif pct_correct >= 70:
    print("✓ VALIDATION PASSED: Senescence labels appear correct")
    print(f"  {pct_correct:.0f}% correct direction")
elif pct_correct >= 50:
    print("⚠ VALIDATION PARTIAL: Mixed results")
    print(f"  {pct_correct:.0f}% correct direction - review individual markers")
else:
    print("✗ VALIDATION FAILED: Labels may not reflect true senescence")
    print(f"  Only {pct_correct:.0f}% correct direction")
print("="*80)

# ════════════════════════════════════════════════════════════════════════════════
# SAVE RESULTS
# ════════════════════════════════════════════════════════════════════════════════

# Save results (exclude array columns)
results_save = aggregated_results_df.drop(
    columns=['Paired_NonSnC', 'Paired_SnC', 'Sample_IDs']
)
results_file = RESULTS_DIR / f'{DATASET}_validation_plot1A_results.csv'
results_save.to_csv(results_file, index=False)
print(f"\n✓ Results saved: {results_file}")

## Plot 1B: Paired Expression Analysis (Per Cell Type)

**Purpose:** Examine cell-type-specific patterns in senescence marker expression.

**Approach:**
- Separate analysis for each cell type
- Paired comparison within sample, within cell type
- Identify cell types with consistent vs divergent patterns

**Output:** Heatmap showing Log2FC (SnC/Non-SnC) by Gene × Cell Type

**Interpretation:**
- Consistent pattern across cell types → Universal senescence signature
- Cell-type-specific differences → Biology worth investigating (e.g., LMNB1)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 1B: PER CELL TYPE ANALYSIS - SETUP
# ════════════════════════════════════════════════════════════════════════════════

print("="*80)
print("PLOT 1B: PAIRED EXPRESSION ANALYSIS (PER CELL TYPE)")
print("="*80)
print("Purpose: Examine cell-type-specific patterns")
print("Method: OLS/Robust regression with covariates, covariate-adjusted values")
print("="*80 + "\n")

# ════════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

# Column names
CELLTYPE_COL = CELL_TYPE_COLUMN if 'CELL_TYPE_COLUMN' in dir() else 'cell_type'

# Parameters
MIN_CELLS_PER_GROUP = 10   # Minimum cells per sample per senescence status
MIN_PAIRED_SAMPLES = 5     # Minimum paired samples to include a cell type

print(f"Configuration:")
print(f"  Cell type column: {CELLTYPE_COL}")
print(f"  Expression layer: {EXPRESSION_LAYER}")
print(f"  Covariates: {AGE_COL}, {SEX_COL}, {LIBRARY_DEPTH_COL}, {COHORT_COL}")
print(f"  Minimum cells per group: {MIN_CELLS_PER_GROUP}")
print(f"  Minimum paired samples: {MIN_PAIRED_SAMPLES}")

# ════════════════════════════════════════════════════════════════════════════════
# CALCULATE PER CELL TYPE PAIRED MEANS WITH COVARIATES
# ════════════════════════════════════════════════════════════════════════════════

print("\nCalculating paired means per cell type...")

def calculate_celltype_paired_means(adata, gene, cell_type, sample_col, senescence_col, 
                                     celltype_col, age_col, sex_col, depth_col, cohort_col,
                                     expression_layer='lognorm', min_cells=10):
    """
    Calculate mean expression for paired samples within a specific cell type.
    Includes donor-level covariates for LMM.
    """
    # Subset to cell type
    ct_mask = adata.obs[celltype_col] == cell_type
    ct_data = adata[ct_mask]
    
    if ct_data.n_obs == 0 or gene not in ct_data.var_names:
        return pd.DataFrame()
    
    # Get expression from specified layer
    if expression_layer in ct_data.layers:
        expr = ct_data[:, gene].layers[expression_layer]
    else:
        expr = ct_data[:, gene].X
    
    if hasattr(expr, 'toarray'):
        expr = expr.toarray().flatten()
    else:
        expr = np.array(expr).flatten()
    
    # Build cell-level dataframe
    df = pd.DataFrame({
        'Expression': expr,
        'Sample': ct_data.obs[sample_col].values,
        'Senescence_Status': ct_data.obs[senescence_col].values,
    })
    
    # Add available covariates
    if age_col in ct_data.obs.columns:
        df['Age'] = pd.to_numeric(ct_data.obs[age_col].values, errors='coerce')
    if sex_col in ct_data.obs.columns:
        df['Sex'] = ct_data.obs[sex_col].values
    if depth_col in ct_data.obs.columns:
        df['Log_Library_Depth'] = pd.to_numeric(ct_data.obs[depth_col].values, errors='coerce')
    if cohort_col in ct_data.obs.columns:
        df['Cohort'] = ct_data.obs[cohort_col].values
    
    # ── Pseudobulk: aggregate per donor × SnC status ─────────────────────────
    
    grouped = df.groupby(['Sample', 'Senescence_Status'])
    
    result = grouped['Expression'].mean().reset_index()
    result.rename(columns={'Expression': 'Mean_Expression'}, inplace=True)
    result['N_Cells'] = grouped['Expression'].count().values
    
    # Add covariates
    if 'Age' in df.columns:
        result['Age'] = grouped['Age'].first().values
    if 'Sex' in df.columns:
        result['Sex'] = grouped['Sex'].first().values
    if 'Log_Library_Depth' in df.columns:
        result['Log_Library_Depth'] = grouped['Log_Library_Depth'].mean().values
    if 'Cohort' in df.columns:
        result['Cohort'] = grouped['Cohort'].first().values
    
    # Filter by minimum cells
    result = result[result['N_Cells'] >= min_cells]
    
    # Find paired samples
    status_counts = result.groupby('Sample')['Senescence_Status'].nunique()
    paired_samples = status_counts[status_counts == 2].index.tolist()
    
    # Keep only paired
    paired_data = result[result['Sample'].isin(paired_samples)].copy()
    paired_data['Gene'] = gene
    paired_data['Cell_Type'] = cell_type
    
    return paired_data

# Get all cell types
all_celltypes = adata.obs[CELLTYPE_COL].unique().tolist()
print(f"  Found {len(all_celltypes)} cell types")

# Calculate for all markers × cell types
celltype_paired_data = []
pairing_summary = []

for gene in available_markers.keys():
    for cell_type in all_celltypes:
        paired_data = calculate_celltype_paired_means(
            adata, gene, cell_type, 
            SAMPLE_COL, SENESCENCE_COL, CELLTYPE_COL,
            AGE_COL, SEX_COL, LIBRARY_DEPTH_COL, COHORT_COL,
            expression_layer=EXPRESSION_LAYER,
            min_cells=MIN_CELLS_PER_GROUP
        )
        
        n_pairs = len(paired_data['Sample'].unique()) if len(paired_data) > 0 else 0
        
        pairing_summary.append({
            'Gene': gene,
            'Cell_Type': cell_type,
            'N_Paired_Samples': n_pairs
        })
        
        if len(paired_data) > 0:
            celltype_paired_data.append(paired_data)

celltype_paired_df = pd.concat(celltype_paired_data, ignore_index=True)
pairing_summary_df = pd.DataFrame(pairing_summary)

# ════════════════════════════════════════════════════════════════════════════════
# FILTER CELL TYPES WITH SUFFICIENT PAIRED SAMPLES
# ════════════════════════════════════════════════════════════════════════════════

# Calculate average paired samples per cell type
ct_avg_pairs = pairing_summary_df.groupby('Cell_Type')['N_Paired_Samples'].mean()

# Filter
valid_celltypes = ct_avg_pairs[ct_avg_pairs >= MIN_PAIRED_SAMPLES].index.tolist()
excluded_celltypes = ct_avg_pairs[ct_avg_pairs < MIN_PAIRED_SAMPLES].index.tolist()

print(f"\n=== CELL TYPE FILTERING ===")
print(f"Included ({len(valid_celltypes)} cell types):")
for ct in sorted(valid_celltypes):
    avg = ct_avg_pairs[ct]
    print(f"  ✓ {ct}: avg {avg:.0f} paired samples")

if excluded_celltypes:
    print(f"\nExcluded ({len(excluded_celltypes)} cell types):")
    for ct in sorted(excluded_celltypes):
        avg = ct_avg_pairs[ct]
        print(f"  ✗ {ct}: avg {avg:.0f} paired samples")

# Filter data to valid cell types
celltype_paired_df = celltype_paired_df[celltype_paired_df['Cell_Type'].isin(valid_celltypes)]

print(f"\n✓ Proceeding with {len(valid_celltypes)} cell types")
print(f"✓ Total observations: {len(celltype_paired_df):,}")
print(f"  Columns: {list(celltype_paired_df.columns)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# STATISTICAL ANALYSIS: OLS/ROBUST REGRESSION WITH COVARIATE ADJUSTMENT
# ════════════════════════════════════════════════════════════════════════════════
#
# Model: Mean_Expression ~ SnC_Status + Age + Sex + Log_Library_Depth + Cohort
#
# - SnC_Status coefficient = covariate-adjusted SnC effect
# - Adjusted values = observed - covariate_prediction + grand_mean
#   (removes covariate effects while preserving SnC difference)
#
# ════════════════════════════════════════════════════════════════════════════════

print("\nPerforming regression per gene × cell type...")

# ─────────────────────────────────────────────────────────────────────────────
# BUILD COVARIATE FORMULA
# ─────────────────────────────────────────────────────────────────────────────

sample_ct_df = celltype_paired_df.iloc[:100]

cov_terms_ct = []
if 'Age' in sample_ct_df.columns and sample_ct_df['Age'].notna().sum() > 0:
    cov_terms_ct.append('Age')
if 'Sex' in sample_ct_df.columns and sample_ct_df['Sex'].nunique() > 1:
    cov_terms_ct.append('C(Sex)')
if 'Log_Library_Depth' in sample_ct_df.columns and sample_ct_df['Log_Library_Depth'].notna().sum() > 0:
    cov_terms_ct.append('Log_Library_Depth')
if 'Cohort' in sample_ct_df.columns and sample_ct_df['Cohort'].nunique() > 1:
    cov_terms_ct.append('C(Cohort)')

formula_rhs_ct = ' + '.join(['SnC_Status'] + cov_terms_ct)
formula_str_ct = f"Mean_Expression ~ {formula_rhs_ct}"

print(f"  Model: {formula_str_ct}")
print(f"  Primary: Robust Linear Model (Huber's T)")
print(f"  Fallback: OLS")
print(f"  Covariate-adjusted values computed for plotting")
print(f"  Multiple testing: BH (FDR) correction\n")

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: FIT MODEL AND COMPUTE ADJUSTED VALUES
# ─────────────────────────────────────────────────────────────────────────────

def fit_and_adjust(gene_df, formula_str):
    """
    Fit OLS/robust on unpivoted pseudobulk data.
    Returns coefficient, p-value, and covariate-adjusted expression values.
    """
    df = gene_df.copy()
    df['SnC_Status'] = (df['Senescence_Status'] == 'SnC').astype(int)
    
    # Drop missing
    model_cols = ['Mean_Expression', 'SnC_Status']
    for col in ['Age', 'Sex', 'Log_Library_Depth', 'Cohort']:
        if col in df.columns:
            model_cols.append(col)
    df_clean = df.dropna(subset=[c for c in model_cols if c in df.columns])
    
    if len(df_clean) < 6:
        return None
    
    result = {}
    
    # Try robust regression first
    try:
        rlm_model = smf.rlm(formula_str, df_clean, M=sm.robust.norms.HuberT())
        fit = rlm_model.fit()
        
        result['estimate'] = fit.params['SnC_Status']
        result['std_error'] = fit.bse['SnC_Status']
        result['p_value'] = fit.pvalues['SnC_Status']
        result['model_type'] = 'RLM'
        
    except Exception:
        try:
            fit = smf.ols(formula_str, df_clean).fit()
            
            result['estimate'] = fit.params['SnC_Status']
            result['std_error'] = fit.bse['SnC_Status']
            result['p_value'] = fit.pvalues['SnC_Status']
            result['model_type'] = 'OLS'
            
        except Exception:
            return None
    
    # ── Compute covariate-adjusted values ────────────────────────────────────
    # Adjusted = Observed - covariate_prediction + grand_mean
    # covariate_prediction = fitted - SnC_effect * SnC_Status
    
    snc_effect = result['estimate']
    df_clean['fitted'] = fit.fittedvalues
    df_clean['covariate_prediction'] = df_clean['fitted'] - snc_effect * df_clean['SnC_Status']
    
    grand_mean = df_clean['Mean_Expression'].mean()
    df_clean['Adjusted_Expression'] = (
        df_clean['Mean_Expression'] 
        - df_clean['covariate_prediction'] 
        + grand_mean
    )
    
    result['adjusted_data'] = df_clean[
        ['Sample', 'Senescence_Status', 'Mean_Expression', 'Adjusted_Expression']
    ].copy()
    
    return result

# ─────────────────────────────────────────────────────────────────────────────
# FIT MODEL PER GENE × CELL TYPE
# ─────────────────────────────────────────────────────────────────────────────

celltype_results = []

for gene, gene_info in available_markers.items():
    for cell_type in valid_celltypes:
        subset = celltype_paired_df[
            (celltype_paired_df['Gene'] == gene) & 
            (celltype_paired_df['Cell_Type'] == cell_type)
        ]
        
        if len(subset) == 0:
            continue
        
        result = fit_and_adjust(subset, formula_str_ct)
        
        if result is None:
            continue
        
        if pd.isna(result['p_value']):
            continue
        
        # Get adjusted paired values
        adj = result['adjusted_data']
        nonsnc_adj = adj[adj['Senescence_Status'] == 'Non-SnC'].sort_values('Sample')
        snc_adj = adj[adj['Senescence_Status'] == 'SnC'].sort_values('Sample')
        
        # Raw values
        nonsnc_raw = adj[adj['Senescence_Status'] == 'Non-SnC'].sort_values('Sample')
        snc_raw = adj[adj['Senescence_Status'] == 'SnC'].sort_values('Sample')
        
        n_pairs = len(nonsnc_adj['Sample'].unique())
        
        mean_diff = result['estimate']
        log2fc = mean_diff / np.log(2)
        fold_change = 2 ** log2fc
        
        expected = gene_info['expected']
        correct_direction = (mean_diff > 0) if expected == 'up' else (mean_diff < 0)
        
        celltype_results.append({
            'Gene': gene,
            'Alias': gene_info['alias'],
            'Category': gene_info['category'],
            'Expected': expected,
            'Cell_Type': cell_type,
            'N_Paired_Samples': n_pairs,
            'NonSnC_Mean_Raw': nonsnc_raw['Mean_Expression'].mean(),
            'SnC_Mean_Raw': snc_raw['Mean_Expression'].mean(),
            'NonSnC_Mean_Adj': nonsnc_adj['Adjusted_Expression'].mean(),
            'SnC_Mean_Adj': snc_adj['Adjusted_Expression'].mean(),
            'Estimate': result['estimate'],
            'SE': result['std_error'],
            'Mean_Difference': mean_diff,
            'Fold_Change': fold_change,
            'Log2FC': log2fc,
            'P_Value': result['p_value'],
            'Model_Type': result['model_type'],
            'Correct_Direction': correct_direction,
        })

celltype_results_df = pd.DataFrame(celltype_results)

# ─────────────────────────────────────────────────────────────────────────────
# MULTIPLE TESTING CORRECTION (BH / FDR)
# ─────────────────────────────────────────────────────────────────────────────

if len(celltype_results_df) > 0:
    pvals = celltype_results_df['P_Value'].values
    reject, p_adj, _, _ = multipletests(pvals, method='fdr_bh')
    celltype_results_df['P_Adj'] = p_adj
    celltype_results_df['Significant'] = reject
else:
    celltype_results_df['P_Adj'] = []
    celltype_results_df['Significant'] = []

print(f"✓ Calculated {len(celltype_results_df)} gene × cell type combinations")
print(f"  Significant (FDR < 0.05): {celltype_results_df['Significant'].sum()}")
print(f"  Correct direction: {celltype_results_df['Correct_Direction'].sum()}/{len(celltype_results_df)}")
print(f"  Model: {formula_str_ct}")

# Save results
results_save = celltype_results_df.copy()
results_save.to_csv(
    RESULTS_DIR / f'{DATASET}_validation_plot1B_results.csv', index=False
)
print(f"\n✓ Saved: {DATASET}_validation_plot1B_results.csv")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# REGRESSION + PLOT FOR A SPECIFIC CELL TYPE
# ════════════════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────────────────
# USAGE: Set TARGET_CELLTYPE in a previous cell before running this block
#        e.g., TARGET_CELLTYPE = 'Microglia'
# ─────────────────────────────────────────────────────────────────────────────

TARGET_CELLTYPE = 'OPC'

if 'TARGET_CELLTYPE' not in dir():
    print("⚠ TARGET_CELLTYPE not defined. Set it before running this cell.")
    print(f"  Available cell types: {sorted(valid_celltypes)}")
    raise ValueError("Set TARGET_CELLTYPE first")

print("\n" + "="*80)
print(f"REGRESSION FOR: {TARGET_CELLTYPE}")
print(f"Model: {formula_str_ct}")
print("="*80)

# Validate cell type exists
if TARGET_CELLTYPE not in valid_celltypes:
    print(f"⚠ '{TARGET_CELLTYPE}' not in valid_celltypes.")
    print(f"  Available: {sorted(valid_celltypes)}")
else:
    celltype_results_single = []
    
    for gene, gene_info in available_markers.items():
        subset = celltype_paired_df[
            (celltype_paired_df['Gene'] == gene) & 
            (celltype_paired_df['Cell_Type'] == TARGET_CELLTYPE)
        ]
        
        if len(subset) == 0:
            print(f"  {gene}: No data")
            continue
        
        result = fit_and_adjust(subset, formula_str_ct)
        
        if result is None:
            print(f"  {gene}: Model failed")
            continue
        
        if pd.isna(result['p_value']):
            continue
        
        # Get adjusted paired values
        adj = result['adjusted_data']
        nonsnc_adj = adj[adj['Senescence_Status'] == 'Non-SnC'].sort_values('Sample')
        snc_adj = adj[adj['Senescence_Status'] == 'SnC'].sort_values('Sample')
        
        n_pairs = len(nonsnc_adj['Sample'].unique())
        
        mean_diff = result['estimate']
        log2fc = mean_diff / np.log(2)
        fold_change = 2 ** log2fc
        
        expected = gene_info['expected']
        correct_direction = (mean_diff > 0) if expected == 'up' else (mean_diff < 0)
        
        status = "✓" if correct_direction else "✗"
        print(f"  {status} {gene}: Log2FC={log2fc:+.3f}, P={result['p_value']:.2e} [{result['model_type']}]")
        
        celltype_results_single.append({
            'Gene': gene,
            'Alias': gene_info['alias'],
            'Category': gene_info['category'],
            'Expected': expected,
            'Cell_Type': TARGET_CELLTYPE,
            'N_Paired_Samples': n_pairs,
            'NonSnC_Mean_Raw': adj[adj['Senescence_Status'] == 'Non-SnC']['Mean_Expression'].mean(),
            'SnC_Mean_Raw': adj[adj['Senescence_Status'] == 'SnC']['Mean_Expression'].mean(),
            'NonSnC_Mean_Adj': nonsnc_adj['Adjusted_Expression'].mean(),
            'SnC_Mean_Adj': snc_adj['Adjusted_Expression'].mean(),
            'Estimate': result['estimate'],
            'SE': result['std_error'],
            'Mean_Difference': mean_diff,
            'Fold_Change': fold_change,
            'Log2FC': log2fc,
            'P_Value': result['p_value'],
            'Model_Type': result['model_type'],
            'Correct_Direction': correct_direction,
            'Paired_NonSnC_Adj': nonsnc_adj['Adjusted_Expression'].values,
            'Paired_SnC_Adj': snc_adj['Adjusted_Expression'].values,
        })
    
    celltype_results_single_df = pd.DataFrame(celltype_results_single)
    
    # BH correction
    if len(celltype_results_single_df) > 0:
        pvals = celltype_results_single_df['P_Value'].values
        reject, p_adj, _, _ = multipletests(pvals, method='fdr_bh')
        celltype_results_single_df['P_Adj'] = p_adj
        celltype_results_single_df['Significant'] = reject
    
    # Summary
    print("\n" + "-"*40)
    n_total = len(celltype_results_single_df)
    n_correct = celltype_results_single_df['Correct_Direction'].sum()
    n_sig = celltype_results_single_df['Significant'].sum() if 'Significant' in celltype_results_single_df.columns else 0
    
    print(f"Summary for {TARGET_CELLTYPE}:")
    print(f"  Markers tested: {n_total}")
    print(f"  Correct direction: {n_correct}/{n_total} ({n_correct/n_total*100:.0f}%)" if n_total > 0 else "  No markers tested")
    print(f"  Significant (FDR < 0.05): {n_sig}/{n_total}")
    print(f"  Model: {formula_str_ct}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # SAVE CSV
    # ─────────────────────────────────────────────────────────────────────────
    
    ct_clean = TARGET_CELLTYPE.replace(' ', '_').replace('/', '_')
    results_save = celltype_results_single_df.drop(
        columns=['Paired_NonSnC_Adj', 'Paired_SnC_Adj'], errors='ignore'
    )
    output_csv = RESULTS_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.csv'
    results_save.to_csv(output_csv, index=False)
    print(f"\n✓ Saved: {output_csv}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # PLOT: Paired box plots (covariate-adjusted)
    # ─────────────────────────────────────────────────────────────────────────
    
    n_markers = len(celltype_results_single_df)
    
    if n_markers > 0:
        n_cols = 4
        n_rows = int(np.ceil(n_markers / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3.5 * n_rows))
        if n_markers > 1:
            axes = axes.flatten()
        else:
            axes = [axes]
        
        for idx, (_, row) in enumerate(celltype_results_single_df.iterrows()):
            ax = axes[idx]
            gene = row['Gene']
            
            # Covariate-adjusted values
            nonsnc_vals = row['Paired_NonSnC_Adj']
            snc_vals = row['Paired_SnC_Adj']
            n_pairs = len(nonsnc_vals)
            
            # Box plots
            bp = ax.boxplot([nonsnc_vals, snc_vals], positions=[0, 1], widths=0.5, 
                           patch_artist=True, showfliers=False, zorder=2)
            
            for patch, color in zip(bp['boxes'], [COLORS['Non-SnC'], COLORS['SnC']]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
                patch.set_edgecolor('black')
                patch.set_linewidth(1.2)
            
            for median in bp['medians']:
                median.set_color('black')
                median.set_linewidth(2)
            
            for whisker in bp['whiskers']:
                whisker.set_color('black')
                whisker.set_linewidth(1)
            for cap in bp['caps']:
                cap.set_color('black')
                cap.set_linewidth(1)
            
            # Paired points with connecting lines
            np.random.seed(42)
            jitter = np.random.uniform(-0.12, 0.12, n_pairs)
            
            for i in range(n_pairs):
                ax.plot([0 + jitter[i], 1 + jitter[i]], [nonsnc_vals[i], snc_vals[i]],
                       color='gray', alpha=0.25, linewidth=0.6, zorder=1)
            
            ax.scatter(0 + jitter, nonsnc_vals, color='darkblue', s=18, alpha=0.6,
                      edgecolor='white', linewidth=0.3, zorder=3)
            ax.scatter(1 + jitter, snc_vals, color='darkred', s=18, alpha=0.6,
                      edgecolor='white', linewidth=0.3, zorder=3)
            
            # Annotations
            y_max = max(np.max(nonsnc_vals), np.max(snc_vals))
            y_min = min(np.min(nonsnc_vals), np.min(snc_vals))
            y_range = y_max - y_min if y_max > y_min else 0.1
            
            ax.text(0.5, y_max + y_range * 0.05, f'n={n_pairs}', ha='center', va='bottom',
                   fontsize=9, fontweight='bold')
            
            # FDR-adjusted p-value
            stars, p_text = format_pvalue(row['P_Adj'])
            sig_color = 'firebrick' if row['P_Adj'] < 0.05 else 'gray'
            
            bracket_y = y_max + y_range * 0.15
            ax.plot([0, 0, 1, 1], 
                    [bracket_y - y_range * 0.02, bracket_y, bracket_y, bracket_y - y_range * 0.02],
                    color='black', linewidth=1)
            ax.text(0.5, bracket_y + y_range * 0.01, f'{stars}\n{p_text}', ha='center', va='bottom',
                   fontsize=8, fontweight='bold', color=sig_color)
            
            # Title
            if row['Correct_Direction']:
                direction_text = 'SnC higher' if row['Mean_Difference'] > 0 else 'SnC lower'
            else:
                observed_str = 'higher' if row['Mean_Difference'] > 0 else 'lower'
                expected_str = 'lower' if row['Expected'] == 'down' else 'higher'
                direction_text = f'SnC {observed_str}\n(expected {expected_str})'
            title_color = 'black' if row['Correct_Direction'] else '#CC4444'
            
            ax.set_title(f'{gene}\n{direction_text}', 
                         fontweight='bold', fontsize=10, color=title_color)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['Non-SnC', 'SnC'], fontsize=10)
            
            if idx % n_cols == 0:
                ax.set_ylabel('Adjusted Expression', fontsize=9)
            
            ax.set_ylim(y_min - y_range * 0.1, y_max + y_range * 0.55)
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['bottom'].set_linewidth(1.2)
            ax.spines['left'].set_linewidth(1.2)
        
        # Hide empty subplots
        for idx in range(n_markers, len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle(
            f'Senescence Marker Validation \u2014 {TARGET_CELLTYPE}\n'
            f'Covariate-adjusted  |  FDR-corrected  |  '
            f'Red title = unexpected direction',
            fontsize=12, fontweight='bold', y=1.02
        )
        
        plt.tight_layout()
        
        output_pdf = FIGURES_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.pdf'
        output_svg = FIGURES_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.svg'
        
        plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)
        plt.savefig(output_svg, format='svg', bbox_inches='tight', dpi=300)
        
        print(f"✓ Saved: {output_pdf}")
        print(f"✓ Saved: {output_svg}")
        
        plt.show()
    
    print("="*80)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# REGRESSION + PLOT FOR A SPECIFIC CELL TYPE
# ════════════════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────────────────
# USAGE: Set TARGET_CELLTYPE in a previous cell before running this block
#        e.g., TARGET_CELLTYPE = 'Microglia'
# ─────────────────────────────────────────────────────────────────────────────

TARGET_CELLTYPE = 'Astrocyte'

if 'TARGET_CELLTYPE' not in dir():
    print("⚠ TARGET_CELLTYPE not defined. Set it before running this cell.")
    print(f"  Available cell types: {sorted(valid_celltypes)}")
    raise ValueError("Set TARGET_CELLTYPE first")

print("\n" + "="*80)
print(f"REGRESSION FOR: {TARGET_CELLTYPE}")
print(f"Model: {formula_str_ct}")
print("="*80)

# Validate cell type exists
if TARGET_CELLTYPE not in valid_celltypes:
    print(f"⚠ '{TARGET_CELLTYPE}' not in valid_celltypes.")
    print(f"  Available: {sorted(valid_celltypes)}")
else:
    celltype_results_single = []
    
    for gene, gene_info in available_markers.items():
        subset = celltype_paired_df[
            (celltype_paired_df['Gene'] == gene) & 
            (celltype_paired_df['Cell_Type'] == TARGET_CELLTYPE)
        ]
        
        if len(subset) == 0:
            print(f"  {gene}: No data")
            continue
        
        result = fit_and_adjust(subset, formula_str_ct)
        
        if result is None:
            print(f"  {gene}: Model failed")
            continue
        
        if pd.isna(result['p_value']):
            continue
        
        # Get adjusted paired values
        adj = result['adjusted_data']
        nonsnc_adj = adj[adj['Senescence_Status'] == 'Non-SnC'].sort_values('Sample')
        snc_adj = adj[adj['Senescence_Status'] == 'SnC'].sort_values('Sample')
        
        n_pairs = len(nonsnc_adj['Sample'].unique())
        
        mean_diff = result['estimate']
        log2fc = mean_diff / np.log(2)
        fold_change = 2 ** log2fc
        
        expected = gene_info['expected']
        correct_direction = (mean_diff > 0) if expected == 'up' else (mean_diff < 0)
        
        status = "✓" if correct_direction else "✗"
        print(f"  {status} {gene}: Log2FC={log2fc:+.3f}, P={result['p_value']:.2e} [{result['model_type']}]")
        
        celltype_results_single.append({
            'Gene': gene,
            'Alias': gene_info['alias'],
            'Category': gene_info['category'],
            'Expected': expected,
            'Cell_Type': TARGET_CELLTYPE,
            'N_Paired_Samples': n_pairs,
            'NonSnC_Mean_Raw': adj[adj['Senescence_Status'] == 'Non-SnC']['Mean_Expression'].mean(),
            'SnC_Mean_Raw': adj[adj['Senescence_Status'] == 'SnC']['Mean_Expression'].mean(),
            'NonSnC_Mean_Adj': nonsnc_adj['Adjusted_Expression'].mean(),
            'SnC_Mean_Adj': snc_adj['Adjusted_Expression'].mean(),
            'Estimate': result['estimate'],
            'SE': result['std_error'],
            'Mean_Difference': mean_diff,
            'Fold_Change': fold_change,
            'Log2FC': log2fc,
            'P_Value': result['p_value'],
            'Model_Type': result['model_type'],
            'Correct_Direction': correct_direction,
            'Paired_NonSnC_Adj': nonsnc_adj['Adjusted_Expression'].values,
            'Paired_SnC_Adj': snc_adj['Adjusted_Expression'].values,
        })
    
    celltype_results_single_df = pd.DataFrame(celltype_results_single)
    
    # BH correction
    if len(celltype_results_single_df) > 0:
        pvals = celltype_results_single_df['P_Value'].values
        reject, p_adj, _, _ = multipletests(pvals, method='fdr_bh')
        celltype_results_single_df['P_Adj'] = p_adj
        celltype_results_single_df['Significant'] = reject
    
    # Summary
    print("\n" + "-"*40)
    n_total = len(celltype_results_single_df)
    n_correct = celltype_results_single_df['Correct_Direction'].sum()
    n_sig = celltype_results_single_df['Significant'].sum() if 'Significant' in celltype_results_single_df.columns else 0
    
    print(f"Summary for {TARGET_CELLTYPE}:")
    print(f"  Markers tested: {n_total}")
    print(f"  Correct direction: {n_correct}/{n_total} ({n_correct/n_total*100:.0f}%)" if n_total > 0 else "  No markers tested")
    print(f"  Significant (FDR < 0.05): {n_sig}/{n_total}")
    print(f"  Model: {formula_str_ct}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # SAVE CSV
    # ─────────────────────────────────────────────────────────────────────────
    
    ct_clean = TARGET_CELLTYPE.replace(' ', '_').replace('/', '_')
    results_save = celltype_results_single_df.drop(
        columns=['Paired_NonSnC_Adj', 'Paired_SnC_Adj'], errors='ignore'
    )
    output_csv = RESULTS_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.csv'
    results_save.to_csv(output_csv, index=False)
    print(f"\n✓ Saved: {output_csv}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # PLOT: Paired box plots (covariate-adjusted)
    # ─────────────────────────────────────────────────────────────────────────
    
    n_markers = len(celltype_results_single_df)
    
    if n_markers > 0:
        n_cols = 4
        n_rows = int(np.ceil(n_markers / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3.5 * n_rows))
        if n_markers > 1:
            axes = axes.flatten()
        else:
            axes = [axes]
        
        for idx, (_, row) in enumerate(celltype_results_single_df.iterrows()):
            ax = axes[idx]
            gene = row['Gene']
            
            # Covariate-adjusted values
            nonsnc_vals = row['Paired_NonSnC_Adj']
            snc_vals = row['Paired_SnC_Adj']
            n_pairs = len(nonsnc_vals)
            
            # Box plots
            bp = ax.boxplot([nonsnc_vals, snc_vals], positions=[0, 1], widths=0.5, 
                           patch_artist=True, showfliers=False, zorder=2)
            
            for patch, color in zip(bp['boxes'], [COLORS['Non-SnC'], COLORS['SnC']]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
                patch.set_edgecolor('black')
                patch.set_linewidth(1.2)
            
            for median in bp['medians']:
                median.set_color('black')
                median.set_linewidth(2)
            
            for whisker in bp['whiskers']:
                whisker.set_color('black')
                whisker.set_linewidth(1)
            for cap in bp['caps']:
                cap.set_color('black')
                cap.set_linewidth(1)
            
            # Paired points with connecting lines
            np.random.seed(42)
            jitter = np.random.uniform(-0.12, 0.12, n_pairs)
            
            for i in range(n_pairs):
                ax.plot([0 + jitter[i], 1 + jitter[i]], [nonsnc_vals[i], snc_vals[i]],
                       color='gray', alpha=0.25, linewidth=0.6, zorder=1)
            
            ax.scatter(0 + jitter, nonsnc_vals, color='darkblue', s=18, alpha=0.6,
                      edgecolor='white', linewidth=0.3, zorder=3)
            ax.scatter(1 + jitter, snc_vals, color='darkred', s=18, alpha=0.6,
                      edgecolor='white', linewidth=0.3, zorder=3)
            
            # Annotations
            y_max = max(np.max(nonsnc_vals), np.max(snc_vals))
            y_min = min(np.min(nonsnc_vals), np.min(snc_vals))
            y_range = y_max - y_min if y_max > y_min else 0.1
            
            ax.text(0.5, y_max + y_range * 0.05, f'n={n_pairs}', ha='center', va='bottom',
                   fontsize=9, fontweight='bold')
            
            # FDR-adjusted p-value
            stars, p_text = format_pvalue(row['P_Adj'])
            sig_color = 'firebrick' if row['P_Adj'] < 0.05 else 'gray'
            
            bracket_y = y_max + y_range * 0.15
            ax.plot([0, 0, 1, 1], 
                    [bracket_y - y_range * 0.02, bracket_y, bracket_y, bracket_y - y_range * 0.02],
                    color='black', linewidth=1)
            ax.text(0.5, bracket_y + y_range * 0.01, f'{stars}\n{p_text}', ha='center', va='bottom',
                   fontsize=8, fontweight='bold', color=sig_color)
            
            # Title
            if row['Correct_Direction']:
                direction_text = 'SnC higher' if row['Mean_Difference'] > 0 else 'SnC lower'
            else:
                observed_str = 'higher' if row['Mean_Difference'] > 0 else 'lower'
                expected_str = 'lower' if row['Expected'] == 'down' else 'higher'
                direction_text = f'SnC {observed_str}\n(expected {expected_str})'
            title_color = 'black' if row['Correct_Direction'] else '#CC4444'
            
            ax.set_title(f'{gene}\n{direction_text}', 
                         fontweight='bold', fontsize=10, color=title_color)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['Non-SnC', 'SnC'], fontsize=10)
            
            if idx % n_cols == 0:
                ax.set_ylabel('Adjusted Expression', fontsize=9)
            
            ax.set_ylim(y_min - y_range * 0.1, y_max + y_range * 0.55)
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['bottom'].set_linewidth(1.2)
            ax.spines['left'].set_linewidth(1.2)
        
        # Hide empty subplots
        for idx in range(n_markers, len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle(
            f'Senescence Marker Validation \u2014 {TARGET_CELLTYPE}\n'
            f'Covariate-adjusted  |  FDR-corrected  |  '
            f'Red title = unexpected direction',
            fontsize=12, fontweight='bold', y=1.02
        )
        
        plt.tight_layout()
        
        output_pdf = FIGURES_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.pdf'
        output_svg = FIGURES_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.svg'
        
        plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)
        plt.savefig(output_svg, format='svg', bbox_inches='tight', dpi=300)
        
        print(f"✓ Saved: {output_pdf}")
        print(f"✓ Saved: {output_svg}")
        
        plt.show()
    
    print("="*80)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# REGRESSION + PLOT FOR A SPECIFIC CELL TYPE
# ════════════════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────────────────
# USAGE: Set TARGET_CELLTYPE in a previous cell before running this block
#        e.g., TARGET_CELLTYPE = 'Microglia'
# ─────────────────────────────────────────────────────────────────────────────

TARGET_CELLTYPE = 'Astrocyte'

if 'TARGET_CELLTYPE' not in dir():
    print("⚠ TARGET_CELLTYPE not defined. Set it before running this cell.")
    print(f"  Available cell types: {sorted(valid_celltypes)}")
    raise ValueError("Set TARGET_CELLTYPE first")

print("\n" + "="*80)
print(f"REGRESSION FOR: {TARGET_CELLTYPE}")
print(f"Model: {formula_str_ct}")
print("="*80)

# Validate cell type exists
if TARGET_CELLTYPE not in valid_celltypes:
    print(f"⚠ '{TARGET_CELLTYPE}' not in valid_celltypes.")
    print(f"  Available: {sorted(valid_celltypes)}")
else:
    celltype_results_single = []
    
    for gene, gene_info in available_markers.items():
        subset = celltype_paired_df[
            (celltype_paired_df['Gene'] == gene) & 
            (celltype_paired_df['Cell_Type'] == TARGET_CELLTYPE)
        ]
        
        if len(subset) == 0:
            print(f"  {gene}: No data")
            continue
        
        result = fit_and_adjust(subset, formula_str_ct)
        
        if result is None:
            print(f"  {gene}: Model failed")
            continue
        
        if pd.isna(result['p_value']):
            continue
        
        # Get adjusted paired values
        adj = result['adjusted_data']
        nonsnc_adj = adj[adj['Senescence_Status'] == 'Non-SnC'].sort_values('Sample')
        snc_adj = adj[adj['Senescence_Status'] == 'SnC'].sort_values('Sample')
        
        n_pairs = len(nonsnc_adj['Sample'].unique())
        
        mean_diff = result['estimate']
        log2fc = mean_diff / np.log(2)
        fold_change = 2 ** log2fc
        
        expected = gene_info['expected']
        correct_direction = (mean_diff > 0) if expected == 'up' else (mean_diff < 0)
        
        status = "✓" if correct_direction else "✗"
        print(f"  {status} {gene}: Log2FC={log2fc:+.3f}, P={result['p_value']:.2e} [{result['model_type']}]")
        
        celltype_results_single.append({
            'Gene': gene,
            'Alias': gene_info['alias'],
            'Category': gene_info['category'],
            'Expected': expected,
            'Cell_Type': TARGET_CELLTYPE,
            'N_Paired_Samples': n_pairs,
            'NonSnC_Mean_Raw': adj[adj['Senescence_Status'] == 'Non-SnC']['Mean_Expression'].mean(),
            'SnC_Mean_Raw': adj[adj['Senescence_Status'] == 'SnC']['Mean_Expression'].mean(),
            'NonSnC_Mean_Adj': nonsnc_adj['Adjusted_Expression'].mean(),
            'SnC_Mean_Adj': snc_adj['Adjusted_Expression'].mean(),
            'Estimate': result['estimate'],
            'SE': result['std_error'],
            'Mean_Difference': mean_diff,
            'Fold_Change': fold_change,
            'Log2FC': log2fc,
            'P_Value': result['p_value'],
            'Model_Type': result['model_type'],
            'Correct_Direction': correct_direction,
            'Paired_NonSnC_Adj': nonsnc_adj['Adjusted_Expression'].values,
            'Paired_SnC_Adj': snc_adj['Adjusted_Expression'].values,
        })
    
    celltype_results_single_df = pd.DataFrame(celltype_results_single)
    
    # BH correction
    if len(celltype_results_single_df) > 0:
        pvals = celltype_results_single_df['P_Value'].values
        reject, p_adj, _, _ = multipletests(pvals, method='fdr_bh')
        celltype_results_single_df['P_Adj'] = p_adj
        celltype_results_single_df['Significant'] = reject
    
    # Summary
    print("\n" + "-"*40)
    n_total = len(celltype_results_single_df)
    n_correct = celltype_results_single_df['Correct_Direction'].sum()
    n_sig = celltype_results_single_df['Significant'].sum() if 'Significant' in celltype_results_single_df.columns else 0
    
    print(f"Summary for {TARGET_CELLTYPE}:")
    print(f"  Markers tested: {n_total}")
    print(f"  Correct direction: {n_correct}/{n_total} ({n_correct/n_total*100:.0f}%)" if n_total > 0 else "  No markers tested")
    print(f"  Significant (FDR < 0.05): {n_sig}/{n_total}")
    print(f"  Model: {formula_str_ct}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # SAVE CSV
    # ─────────────────────────────────────────────────────────────────────────
    
    ct_clean = TARGET_CELLTYPE.replace(' ', '_').replace('/', '_')
    results_save = celltype_results_single_df.drop(
        columns=['Paired_NonSnC_Adj', 'Paired_SnC_Adj'], errors='ignore'
    )
    output_csv = RESULTS_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.csv'
    results_save.to_csv(output_csv, index=False)
    print(f"\n✓ Saved: {output_csv}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # PLOT: Paired box plots (covariate-adjusted)
    # ─────────────────────────────────────────────────────────────────────────
    
    n_markers = len(celltype_results_single_df)
    
    if n_markers > 0:
        n_cols = 4
        n_rows = int(np.ceil(n_markers / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 3.5 * n_rows))
        if n_markers > 1:
            axes = axes.flatten()
        else:
            axes = [axes]
        
        # ── Publication-quality muted palette for sample/donor colors ──
        _PUB_COLORS = [
            '#4E79A7',  # steel blue
            '#E15759',  # muted red
            '#76B7B2',  # teal
            '#F28E2B',  # burnt orange
            '#59A14F',  # olive green
            '#EDC948',  # goldenrod
            '#B07AA1',  # dusty purple
            '#FF9DA7',  # muted pink
            '#9C755F',  # warm brown
            '#BAB0AC',  # warm gray
            '#AF7AA1',  # mauve
            '#86BCB6',  # sage
            '#D4A6C8',  # light plum
            '#8CD17D',  # soft green
            '#B6992D',  # dark gold
            '#499894',  # dark teal
            '#D37295',  # rose
            '#A0CBE8',  # light blue
            '#F1CE63',  # soft yellow
            '#D7B5A6',  # tan
        ]
        
        first_row = celltype_results_single_df.iloc[0]
        global_n_pairs = len(first_row['Paired_NonSnC_Adj'])
        sample_colors = [_PUB_COLORS[i % len(_PUB_COLORS)] for i in range(global_n_pairs)]
        
        for idx, (_, row) in enumerate(celltype_results_single_df.iterrows()):
            ax = axes[idx]
            gene = row['Gene']
            
            # Covariate-adjusted values
            nonsnc_vals = row['Paired_NonSnC_Adj']
            snc_vals = row['Paired_SnC_Adj']
            n_pairs = len(nonsnc_vals)
            
            # ── Clear (unfilled) compact box plots ──
            bp = ax.boxplot([nonsnc_vals, snc_vals], positions=[0, 1], widths=0.3, 
                           patch_artist=True, showfliers=False, zorder=2)
            
            for patch in bp['boxes']:
                patch.set_facecolor('white')
                patch.set_edgecolor('black')
                patch.set_linewidth(1.2)
            
            for median in bp['medians']:
                median.set_color('black')
                median.set_linewidth(2)
            
            for whisker in bp['whiskers']:
                whisker.set_color('black')
                whisker.set_linewidth(1)
            for cap in bp['caps']:
                cap.set_color('black')
                cap.set_linewidth(1)
            
            # ── Paired points colored by sample/donor ──
            np.random.seed(42)
            jitter = np.random.uniform(-0.08, 0.08, n_pairs)
            
            for i in range(n_pairs):
                ax.plot([0 + jitter[i], 1 + jitter[i]], [nonsnc_vals[i], snc_vals[i]],
                       color=sample_colors[i], alpha=0.3, linewidth=0.7, zorder=1)
                ax.scatter(0 + jitter[i], nonsnc_vals[i], color=sample_colors[i], s=22, alpha=0.8,
                          edgecolor='black', linewidth=0.4, zorder=3)
                ax.scatter(1 + jitter[i], snc_vals[i], color=sample_colors[i], s=22, alpha=0.8,
                          edgecolor='black', linewidth=0.4, zorder=3)
            
            # Annotations
            y_max = max(np.max(nonsnc_vals), np.max(snc_vals))
            y_min = min(np.min(nonsnc_vals), np.min(snc_vals))
            y_range = y_max - y_min if y_max > y_min else 0.1
            
            ax.text(0.5, y_max + y_range * 0.05, f'n={n_pairs}', ha='center', va='bottom',
                   fontsize=9, fontweight='bold')
            
            # FDR-adjusted p-value
            stars, p_text = format_pvalue(row['P_Adj'])
            sig_color = 'firebrick' if row['P_Adj'] < 0.05 else 'gray'
            
            bracket_y = y_max + y_range * 0.15
            ax.plot([0, 0, 1, 1], 
                    [bracket_y - y_range * 0.02, bracket_y, bracket_y, bracket_y - y_range * 0.02],
                    color='black', linewidth=1)
            ax.text(0.5, bracket_y + y_range * 0.01, f'{stars}\n{p_text}', ha='center', va='bottom',
                   fontsize=8, fontweight='bold', color=sig_color)
            
            # Title
            if row['Correct_Direction']:
                direction_text = 'SnC higher' if row['Mean_Difference'] > 0 else 'SnC lower'
            else:
                observed_str = 'higher' if row['Mean_Difference'] > 0 else 'lower'
                expected_str = 'lower' if row['Expected'] == 'down' else 'higher'
                direction_text = f'SnC {observed_str}\n(expected {expected_str})'
            title_color = 'black' if row['Correct_Direction'] else '#CC4444'
            
            ax.set_title(f'{gene}\n{direction_text}', 
                         fontweight='bold', fontsize=10, color=title_color)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['Non-SnC', 'SnC'], fontsize=10)
            
            if idx % n_cols == 0:
                ax.set_ylabel('Adjusted Expression', fontsize=9)
            
            ax.set_ylim(y_min - y_range * 0.1, y_max + y_range * 0.55)
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['bottom'].set_linewidth(1.2)
            ax.spines['left'].set_linewidth(1.2)
        
        # Hide empty subplots
        for idx in range(n_markers, len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle(
            f'Senescence Marker Validation — {TARGET_CELLTYPE}\n'
            f'Covariate-adjusted  |  FDR-corrected  |  '
            f'Red title = unexpected direction  |  Dots colored by sample',
            fontsize=12, fontweight='bold', y=1.02
        )
        
        plt.tight_layout()
        
        output_pdf = FIGURES_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.pdf'
        output_svg = FIGURES_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.svg'
        
        plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)
        plt.savefig(output_svg, format='svg', bbox_inches='tight', dpi=300)
        
        print(f"✓ Saved: {output_pdf}")
        print(f"✓ Saved: {output_svg}")
        
        plt.show()
    
    print("="*80)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# REGRESSION + PLOT FOR A SPECIFIC CELL TYPE
# ════════════════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────────────────
# USAGE: Set TARGET_CELLTYPE in a previous cell before running this block
#        e.g., TARGET_CELLTYPE = 'Microglia'
# ─────────────────────────────────────────────────────────────────────────────

TARGET_CELLTYPE = 'OPC'

if 'TARGET_CELLTYPE' not in dir():
    print("⚠ TARGET_CELLTYPE not defined. Set it before running this cell.")
    print(f"  Available cell types: {sorted(valid_celltypes)}")
    raise ValueError("Set TARGET_CELLTYPE first")

print("\n" + "="*80)
print(f"REGRESSION FOR: {TARGET_CELLTYPE}")
print(f"Model: {formula_str_ct}")
print("="*80)

# Validate cell type exists
if TARGET_CELLTYPE not in valid_celltypes:
    print(f"⚠ '{TARGET_CELLTYPE}' not in valid_celltypes.")
    print(f"  Available: {sorted(valid_celltypes)}")
else:
    celltype_results_single = []
    
    for gene, gene_info in available_markers.items():
        subset = celltype_paired_df[
            (celltype_paired_df['Gene'] == gene) & 
            (celltype_paired_df['Cell_Type'] == TARGET_CELLTYPE)
        ]
        
        if len(subset) == 0:
            print(f"  {gene}: No data")
            continue
        
        result = fit_and_adjust(subset, formula_str_ct)
        
        if result is None:
            print(f"  {gene}: Model failed")
            continue
        
        if pd.isna(result['p_value']):
            continue
        
        # Get adjusted paired values
        adj = result['adjusted_data']
        nonsnc_adj = adj[adj['Senescence_Status'] == 'Non-SnC'].sort_values('Sample')
        snc_adj = adj[adj['Senescence_Status'] == 'SnC'].sort_values('Sample')
        
        n_pairs = len(nonsnc_adj['Sample'].unique())
        
        mean_diff = result['estimate']
        log2fc = mean_diff / np.log(2)
        fold_change = 2 ** log2fc
        
        expected = gene_info['expected']
        correct_direction = (mean_diff > 0) if expected == 'up' else (mean_diff < 0)
        
        status = "✓" if correct_direction else "✗"
        print(f"  {status} {gene}: Log2FC={log2fc:+.3f}, P={result['p_value']:.2e} [{result['model_type']}]")
        
        celltype_results_single.append({
            'Gene': gene,
            'Alias': gene_info['alias'],
            'Category': gene_info['category'],
            'Expected': expected,
            'Cell_Type': TARGET_CELLTYPE,
            'N_Paired_Samples': n_pairs,
            'NonSnC_Mean_Raw': adj[adj['Senescence_Status'] == 'Non-SnC']['Mean_Expression'].mean(),
            'SnC_Mean_Raw': adj[adj['Senescence_Status'] == 'SnC']['Mean_Expression'].mean(),
            'NonSnC_Mean_Adj': nonsnc_adj['Adjusted_Expression'].mean(),
            'SnC_Mean_Adj': snc_adj['Adjusted_Expression'].mean(),
            'Estimate': result['estimate'],
            'SE': result['std_error'],
            'Mean_Difference': mean_diff,
            'Fold_Change': fold_change,
            'Log2FC': log2fc,
            'P_Value': result['p_value'],
            'Model_Type': result['model_type'],
            'Correct_Direction': correct_direction,
            'Paired_NonSnC_Adj': nonsnc_adj['Adjusted_Expression'].values,
            'Paired_SnC_Adj': snc_adj['Adjusted_Expression'].values,
        })
    
    celltype_results_single_df = pd.DataFrame(celltype_results_single)
    
    # BH correction
    if len(celltype_results_single_df) > 0:
        pvals = celltype_results_single_df['P_Value'].values
        reject, p_adj, _, _ = multipletests(pvals, method='fdr_bh')
        celltype_results_single_df['P_Adj'] = p_adj
        celltype_results_single_df['Significant'] = reject
    
    # Summary
    print("\n" + "-"*40)
    n_total = len(celltype_results_single_df)
    n_correct = celltype_results_single_df['Correct_Direction'].sum()
    n_sig = celltype_results_single_df['Significant'].sum() if 'Significant' in celltype_results_single_df.columns else 0
    
    print(f"Summary for {TARGET_CELLTYPE}:")
    print(f"  Markers tested: {n_total}")
    print(f"  Correct direction: {n_correct}/{n_total} ({n_correct/n_total*100:.0f}%)" if n_total > 0 else "  No markers tested")
    print(f"  Significant (FDR < 0.05): {n_sig}/{n_total}")
    print(f"  Model: {formula_str_ct}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # SAVE CSV
    # ─────────────────────────────────────────────────────────────────────────
    
    ct_clean = TARGET_CELLTYPE.replace(' ', '_').replace('/', '_')
    results_save = celltype_results_single_df.drop(
        columns=['Paired_NonSnC_Adj', 'Paired_SnC_Adj'], errors='ignore'
    )
    output_csv = RESULTS_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.csv'
    results_save.to_csv(output_csv, index=False)
    print(f"\n✓ Saved: {output_csv}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # PLOT: Paired box plots (covariate-adjusted)
    # ─────────────────────────────────────────────────────────────────────────
    
    n_markers = len(celltype_results_single_df)
    
    if n_markers > 0:
        n_cols = 4
        n_rows = int(np.ceil(n_markers / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 3.5 * n_rows))
        if n_markers > 1:
            axes = axes.flatten()
        else:
            axes = [axes]
        
        # ── Publication-quality muted palette for sample/donor colors ──
        _PUB_COLORS = [
            '#4E79A7',  # steel blue
            '#E15759',  # muted red
            '#76B7B2',  # teal
            '#F28E2B',  # burnt orange
            '#59A14F',  # olive green
            '#EDC948',  # goldenrod
            '#B07AA1',  # dusty purple
            '#FF9DA7',  # muted pink
            '#9C755F',  # warm brown
            '#BAB0AC',  # warm gray
            '#AF7AA1',  # mauve
            '#86BCB6',  # sage
            '#D4A6C8',  # light plum
            '#8CD17D',  # soft green
            '#B6992D',  # dark gold
            '#499894',  # dark teal
            '#D37295',  # rose
            '#A0CBE8',  # light blue
            '#F1CE63',  # soft yellow
            '#D7B5A6',  # tan
        ]
        
        first_row = celltype_results_single_df.iloc[0]
        global_n_pairs = len(first_row['Paired_NonSnC_Adj'])
        sample_colors = [_PUB_COLORS[i % len(_PUB_COLORS)] for i in range(global_n_pairs)]
        
        for idx, (_, row) in enumerate(celltype_results_single_df.iterrows()):
            ax = axes[idx]
            gene = row['Gene']
            
            # Covariate-adjusted values
            nonsnc_vals = row['Paired_NonSnC_Adj']
            snc_vals = row['Paired_SnC_Adj']
            n_pairs = len(nonsnc_vals)
            
            # ── Clear (unfilled) compact box plots ──
            bp = ax.boxplot([nonsnc_vals, snc_vals], positions=[0, 1], widths=0.3, 
                           patch_artist=True, showfliers=False, zorder=2)
            
            for patch in bp['boxes']:
                patch.set_facecolor('white')
                patch.set_edgecolor('black')
                patch.set_linewidth(1.2)
            
            for median in bp['medians']:
                median.set_color('black')
                median.set_linewidth(2)
            
            for whisker in bp['whiskers']:
                whisker.set_color('black')
                whisker.set_linewidth(1)
            for cap in bp['caps']:
                cap.set_color('black')
                cap.set_linewidth(1)
            
            # ── Paired points colored by sample/donor ──
            np.random.seed(42)
            jitter = np.random.uniform(-0.08, 0.08, n_pairs)
            
            for i in range(n_pairs):
                ax.plot([0 + jitter[i], 1 + jitter[i]], [nonsnc_vals[i], snc_vals[i]],
                       color=sample_colors[i], alpha=0.3, linewidth=0.7, zorder=1)
                ax.scatter(0 + jitter[i], nonsnc_vals[i], color=sample_colors[i], s=22, alpha=0.8,
                          edgecolor='black', linewidth=0.4, zorder=3)
                ax.scatter(1 + jitter[i], snc_vals[i], color=sample_colors[i], s=22, alpha=0.8,
                          edgecolor='black', linewidth=0.4, zorder=3)
            
            # Annotations
            y_max = max(np.max(nonsnc_vals), np.max(snc_vals))
            y_min = min(np.min(nonsnc_vals), np.min(snc_vals))
            y_range = y_max - y_min if y_max > y_min else 0.1
            
            ax.text(0.5, y_max + y_range * 0.05, f'n={n_pairs}', ha='center', va='bottom',
                   fontsize=9, fontweight='bold')
            
            # FDR-adjusted p-value
            stars, p_text = format_pvalue(row['P_Adj'])
            sig_color = 'firebrick' if row['P_Adj'] < 0.05 else 'gray'
            
            bracket_y = y_max + y_range * 0.15
            ax.plot([0, 0, 1, 1], 
                    [bracket_y - y_range * 0.02, bracket_y, bracket_y, bracket_y - y_range * 0.02],
                    color='black', linewidth=1)
            ax.text(0.5, bracket_y + y_range * 0.01, f'{stars}\n{p_text}', ha='center', va='bottom',
                   fontsize=8, fontweight='bold', color=sig_color)
            
            # Title
            if row['Correct_Direction']:
                direction_text = 'SnC higher' if row['Mean_Difference'] > 0 else 'SnC lower'
            else:
                observed_str = 'higher' if row['Mean_Difference'] > 0 else 'lower'
                expected_str = 'lower' if row['Expected'] == 'down' else 'higher'
                direction_text = f'SnC {observed_str}\n(expected {expected_str})'
            title_color = 'black' if row['Correct_Direction'] else '#CC4444'
            
            ax.set_title(f'{gene}\n{direction_text}', 
                         fontweight='bold', fontsize=10, color=title_color)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['Non-SnC', 'SnC'], fontsize=10)
            
            if idx % n_cols == 0:
                ax.set_ylabel('Adjusted Expression', fontsize=9)
            
            ax.set_ylim(y_min - y_range * 0.1, y_max + y_range * 0.55)
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['bottom'].set_linewidth(1.2)
            ax.spines['left'].set_linewidth(1.2)
        
        # Hide empty subplots
        for idx in range(n_markers, len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle(
            f'Senescence Marker Validation — {TARGET_CELLTYPE}\n'
            f'Covariate-adjusted  |  FDR-corrected  |  '
            f'Red title = unexpected direction  |  Dots colored by sample',
            fontsize=12, fontweight='bold', y=1.02
        )
        
        plt.tight_layout()
        
        output_pdf = FIGURES_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.pdf'
        output_svg = FIGURES_DIR / f'{DATASET}_validation_plot1B_{ct_clean}.svg'
        
        plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)
        plt.savefig(output_svg, format='svg', bbox_inches='tight', dpi=300)
        
        print(f"✓ Saved: {output_pdf}")
        print(f"✓ Saved: {output_svg}")
        
        plt.show()
    
    print("="*80)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CREATE HEATMAP: LOG2FC BY GENE × CELL TYPE
# ════════════════════════════════════════════════════════════════════════════════

print("\nCreating heatmap...")

import seaborn as sns
from matplotlib.patches import Patch

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE DATA
# ─────────────────────────────────────────────────────────────────────────────

# Pivot table
heatmap_data = celltype_results_df.pivot(
    index='Gene', 
    columns='Cell_Type', 
    values='Log2FC'
)

# Order genes by category
category_order = ['Cell Cycle Arrest', 'p53 Pathway', 'SA-β-gal', 'SASP', 'Anti-apoptosis', 'Nuclear Lamina']
gene_order = []
for cat in category_order:
    cat_genes = [g for g, info in available_markers.items() if info['category'] == cat]
    gene_order.extend([g for g in cat_genes if g in heatmap_data.index])

# Order cell types by average number of pairs (descending)
ct_order = ct_avg_pairs[ct_avg_pairs >= MIN_PAIRED_SAMPLES].sort_values(ascending=False).index.tolist()
ct_order = [ct for ct in ct_order if ct in heatmap_data.columns]

# Reorder
heatmap_data = heatmap_data.reindex(index=gene_order, columns=ct_order)

# Create annotation matrix (empty string for NaN)
annot_matrix = heatmap_data.copy()
annot_matrix = annot_matrix.round(2).astype(str)
annot_matrix = annot_matrix.replace('nan', '')

# Create mask for NaN values
mask_nan = heatmap_data.isna()

# ─────────────────────────────────────────────────────────────────────────────
# CREATE FIGURE
# ─────────────────────────────────────────────────────────────────────────────

# Calculate figure size based on data dimensions
n_genes = len(heatmap_data.index)
n_celltypes = len(heatmap_data.columns)
cell_width = 0.9
cell_height = 0.45

fig_width = max(n_celltypes * cell_width + 2.5, 6)
fig_height = max(n_genes * cell_height + 1.5, 4)

fig, ax = plt.subplots(figsize=(fig_width, fig_height))

# Determine color limits
vmax = np.nanmax(np.abs(heatmap_data.values))
vmax = min(vmax, 2.5)

# First, fill background with dark gray for NaN cells
heatmap_filled = heatmap_data.fillna(0)  # Temporary fill for plotting

# Plot heatmap
sns.heatmap(
    heatmap_filled,
    annot=annot_matrix,
    fmt='',
    cmap='RdBu_r',
    center=0,
    vmin=-vmax,
    vmax=vmax,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Log2FC (SnC / Non-SnC)', 'shrink': 0.7},
    ax=ax,
    annot_kws={'fontsize': 9},
    square=True
)

# Overlay dark gray on NaN cells
for i, gene in enumerate(heatmap_data.index):
    for j, ct in enumerate(heatmap_data.columns):
        if pd.isna(heatmap_data.loc[gene, ct]):
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, 
                                        facecolor='#2F2F2F', edgecolor='white', linewidth=0.5))

# ─────────────────────────────────────────────────────────────────────────────
# FORMATTING
# ─────────────────────────────────────────────────────────────────────────────

ax.set_title('Plot 1B: Log2 Fold Change by Cell Type', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Cell Type', fontsize=11)
ax.set_ylabel('Gene', fontsize=11)

plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)

ax.grid(False)

# Add legend for missing data
legend_elements = [Patch(facecolor='#2F2F2F', edgecolor='white', label='Insufficient data')]
ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.15, 0.15), 
          frameon=True, fontsize=9)

plt.tight_layout()

# ─────────────────────────────────────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────────────────────────────────────

output_pdf = FIGURES_DIR / f'{DATASET}_validation_plot1B_celltype_heatmap.svg'
output_svg = FIGURES_DIR / f'{DATASET}_validation_plot1B_celltype_heatmap.svg'

plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)
plt.savefig(output_svg, format='svg', bbox_inches='tight', dpi=300)

print(f"✓ Saved: {output_pdf}")
print(f"✓ Saved: {output_svg}")

plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 1B: SUMMARY STATISTICS
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("PLOT 1B: VALIDATION SUMMARY (PER CELL TYPE)")
print("="*80)

# Overall statistics
n_total = len(celltype_results_df)
n_correct = celltype_results_df['Correct_Direction'].sum()
n_significant = (celltype_results_df['P_Value'] < 0.05).sum()

pct_correct = n_correct / n_total * 100 if n_total > 0 else 0
pct_sig = n_significant / n_total * 100 if n_total > 0 else 0

print(f"\nOverall Results:")
print(f"  Total gene × cell type combinations: {n_total}")
print(f"  Correct direction: {n_correct}/{n_total} ({pct_correct:.1f}%)")
print(f"  Statistically significant: {n_significant}/{n_total} ({pct_sig:.1f}%)")

# Per cell type
print(f"\nPer Cell Type:")
for ct in ct_order:
    ct_results = celltype_results_df[celltype_results_df['Cell_Type'] == ct]
    if len(ct_results) == 0:
        continue
    ct_correct = ct_results['Correct_Direction'].sum()
    ct_total = len(ct_results)
    ct_pct = ct_correct / ct_total * 100
    ct_sig = (ct_results['P_Value'] < 0.05).sum()
    status = "✓" if ct_pct >= 80 else "⚠" if ct_pct >= 60 else "✗"
    print(f"  {status} {ct}: {ct_correct}/{ct_total} correct ({ct_pct:.0f}%), {ct_sig} significant")

# Per marker
print(f"\nPer Marker:")
for gene in gene_order:
    gene_results = celltype_results_df[celltype_results_df['Gene'] == gene]
    if len(gene_results) == 0:
        continue
    gene_correct = gene_results['Correct_Direction'].sum()
    gene_total = len(gene_results)
    gene_pct = gene_correct / gene_total * 100
    gene_sig = (gene_results['P_Value'] < 0.05).sum()
    expected = available_markers[gene]['expected']
    status = "✓" if gene_pct >= 80 else "⚠" if gene_pct >= 50 else "✗"
    print(f"  {status} {gene}: {gene_correct}/{gene_total} correct ({gene_pct:.0f}%), expected {expected}")

# ════════════════════════════════════════════════════════════════════════════════
# HIGHLIGHT PROBLEMATIC PATTERNS
# ════════════════════════════════════════════════════════════════════════════════

print(f"\n⚠ Markers with cell-type-specific behavior:")
for gene in gene_order:
    gene_results = celltype_results_df[celltype_results_df['Gene'] == gene]
    if len(gene_results) == 0:
        continue
    
    gene_pct = gene_results['Correct_Direction'].mean() * 100
    
    if gene_pct < 80 and gene_pct > 20:  # Mixed behavior
        expected = available_markers[gene]['expected']
        wrong_cts = gene_results[~gene_results['Correct_Direction']]['Cell_Type'].tolist()
        correct_cts = gene_results[gene_results['Correct_Direction']]['Cell_Type'].tolist()
        
        print(f"\n  • {gene} (expected {expected}):")
        print(f"    Correct in: {', '.join(correct_cts[:5])}")
        print(f"    Wrong in: {', '.join(wrong_cts[:5])}")

print("\n" + "="*80)

# ════════════════════════════════════════════════════════════════════════════════
# SAVE RESULTS
# ════════════════════════════════════════════════════════════════════════════════

results_file = RESULTS_DIR / f'{DATASET}_validation_plot1B_celltype_results.csv'
celltype_results_df.to_csv(results_file, index=False)
print(f"\n✓ Results saved: {results_file}")

## Plot 2: %SnC Correlation Across Cell Types

**Purpose:** Test whether senescence burden is coordinated across cell types within the same individual.

**Biological Question:** If a person has high %SnC in Microglia, do they also have high %SnC in Astrocytes and other cell types?

**Approach:**
- Calculate %SnC per sample per cell type
- Pairwise correlation between cell types (using pairwise complete cases)
- Color points by study group (age group)

**Outputs:**
- **2A:** Correlation matrix heatmap (all cell types)
- **2B:** Pairwise scatter plots with regression lines
- **2C:** Connected dot plot (Microglia, Astrocyte, OPC)

**Interpretation:**
- r > 0.6 → Senescence is systemic (whole-brain aging)
- r = 0.3–0.6 → Mixed pattern (systemic + local factors)
- r < 0.3 → Senescence is cell-type specific

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 2: %SnC CORRELATION ACROSS CELL TYPES - SETUP
# ════════════════════════════════════════════════════════════════════════════════

print("="*80)
print("PLOT 2: %SnC CORRELATION ACROSS CELL TYPES")
print("="*80)
print("Purpose: Test if senescence burden is coordinated across cell types")
print("Method: Pairwise correlation of %SnC between cell types")
print("="*80 + "\n")

# ════════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════════

# Column names 
SAMPLE_COL = DONOR_COLUMN if 'DONOR_COLUMN' in dir() else 'Sample'
CELLTYPE_COL = CELL_TYPE_COLUMN if 'CELL_TYPE_COLUMN' in dir() else 'cell_type'
SENESCENCE_BOOL_COL = SENESCENCE_BOOL_COLUMN if 'SENESCENCE_BOOL_COLUMN' in dir() else 'is_senescent'
STUDY_GROUP_COL = STUDY_GROUP_COLUMN if 'STUDY_GROUP_COLUMN' in dir() else 'Study_Group'

# Minimum cells per sample per cell type
MIN_CELLS_PER_SAMPLE = 20
MIN_SAMPLES_PER_CELLTYPE = 10

print(f"Configuration:")
print(f"  Sample column: {SAMPLE_COL}")
print(f"  Cell type column: {CELLTYPE_COL}")
print(f"  Senescence column: {SENESCENCE_BOOL_COL}")
print(f"  Study group column: {STUDY_GROUP_COL}")
print(f"  Minimum cells per sample: {MIN_CELLS_PER_SAMPLE}")
print(f"  Minimum samples per cell type: {MIN_SAMPLES_PER_CELLTYPE}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CALCULATE %SnC PER SAMPLE PER CELL TYPE
# ════════════════════════════════════════════════════════════════════════════════

print("\nCalculating %SnC per sample per cell type...")

# Get unique cell types and samples
all_celltypes = adata.obs[CELLTYPE_COL].unique().tolist()
all_samples = adata.obs[SAMPLE_COL].unique().tolist()

print(f"  Total cell types: {len(all_celltypes)}")
print(f"  Total samples: {len(all_samples)}")

# Calculate %SnC
snc_data = []

for sample in all_samples:
    sample_mask = adata.obs[SAMPLE_COL] == sample
    
    # Get sample-level metadata
    sample_meta = adata.obs.loc[sample_mask].iloc[0]
    study_group = sample_meta[STUDY_GROUP_COL] if STUDY_GROUP_COL in adata.obs.columns else 'Unknown'
    
    for celltype in all_celltypes:
        ct_mask = sample_mask & (adata.obs[CELLTYPE_COL] == celltype)
        n_cells = ct_mask.sum()
        
        if n_cells >= MIN_CELLS_PER_SAMPLE:
            n_snc = adata.obs.loc[ct_mask, SENESCENCE_BOOL_COL].sum()
            pct_snc = (n_snc / n_cells) * 100
            
            snc_data.append({
                'Sample': sample,
                'Cell_Type': celltype,
                'N_Cells': n_cells,
                'N_SnC': n_snc,
                'Pct_SnC': pct_snc,
                'Study_Group': study_group
            })

snc_df = pd.DataFrame(snc_data)

print(f"\n✓ Calculated %SnC for {len(snc_df)} sample × cell type combinations")

# ════════════════════════════════════════════════════════════════════════════════
# FILTER TO CELL TYPES WITH SUFFICIENT DATA
# ════════════════════════════════════════════════════════════════════════════════

# Count samples per cell type
ct_sample_counts = snc_df.groupby('Cell_Type')['Sample'].nunique()

# Keep cell types with sufficient samples
valid_celltypes = ct_sample_counts[ct_sample_counts >= MIN_SAMPLES_PER_CELLTYPE].index.tolist()

print(f"\nCell types with ≥{MIN_SAMPLES_PER_CELLTYPE} samples:")
for ct in sorted(valid_celltypes):
    n = ct_sample_counts[ct]
    print(f"  ✓ {ct}: {n} samples")

excluded_cts = [ct for ct in all_celltypes if ct not in valid_celltypes]
if excluded_cts:
    print(f"\nExcluded (insufficient samples):")
    for ct in sorted(excluded_cts):
        n = ct_sample_counts.get(ct, 0)
        print(f"  ✗ {ct}: {n} samples")

# Filter
snc_df = snc_df[snc_df['Cell_Type'].isin(valid_celltypes)]

print(f"\n✓ Proceeding with {len(valid_celltypes)} cell types")

# ════════════════════════════════════════════════════════════════════════════════
# CALCULATE %SnC PER SAMPLE PER CELL TYPE
# ════════════════════════════════════════════════════════════════════════════════

print("\nCalculating %SnC per sample per cell type...")

# Get unique cell types and samples
all_celltypes = adata.obs[CELLTYPE_COL].unique().tolist()
all_samples = adata.obs[SAMPLE_COL].unique().tolist()

print(f"  Total cell types: {len(all_celltypes)}")
print(f"  Total samples: {len(all_samples)}")

# Calculate %SnC
snc_data = []

for sample in all_samples:
    sample_mask = adata.obs[SAMPLE_COL] == sample
    
    # Get sample-level metadata
    sample_meta = adata.obs.loc[sample_mask].iloc[0]
    study_group = sample_meta[STUDY_GROUP_COL] if STUDY_GROUP_COL in adata.obs.columns else 'Unknown'
    
    for celltype in all_celltypes:
        ct_mask = sample_mask & (adata.obs[CELLTYPE_COL] == celltype)
        n_cells = ct_mask.sum()
        
        if n_cells >= MIN_CELLS_PER_SAMPLE:
            n_snc = adata.obs.loc[ct_mask, SENESCENCE_BOOL_COL].sum()
            pct_snc = (n_snc / n_cells) * 100
            
            snc_data.append({
                'Sample': sample,
                'Cell_Type': celltype,
                'N_Cells': n_cells,
                'N_SnC': n_snc,
                'Pct_SnC': pct_snc,
                'Study_Group': study_group
            })

snc_df = pd.DataFrame(snc_data)

print(f"\n✓ Calculated %SnC for {len(snc_df)} sample × cell type combinations")

# ════════════════════════════════════════════════════════════════════════════════
# FILTER TO CELL TYPES WITH SUFFICIENT DATA
# ════════════════════════════════════════════════════════════════════════════════

# Count samples per cell type
ct_sample_counts = snc_df.groupby('Cell_Type')['Sample'].nunique()

# Keep cell types with sufficient samples
valid_celltypes = ct_sample_counts[ct_sample_counts >= MIN_SAMPLES_PER_CELLTYPE].index.tolist()

print(f"\nCell types with ≥{MIN_SAMPLES_PER_CELLTYPE} samples:")
for ct in sorted(valid_celltypes):
    n = ct_sample_counts[ct]
    print(f"  ✓ {ct}: {n} samples")

excluded_cts = [ct for ct in all_celltypes if ct not in valid_celltypes]
if excluded_cts:
    print(f"\nExcluded (insufficient samples):")
    for ct in sorted(excluded_cts):
        n = ct_sample_counts.get(ct, 0)
        print(f"  ✗ {ct}: {n} samples")

# Filter
snc_df = snc_df[snc_df['Cell_Type'].isin(valid_celltypes)]

print(f"\n✓ Proceeding with {len(valid_celltypes)} cell types")

# ════════════════════════════════════════════════════════════════════════════════
# CALCULATE %SnC PER SAMPLE PER CELL TYPE
# ════════════════════════════════════════════════════════════════════════════════

print("\nCalculating %SnC per sample per cell type...")

# Get unique cell types and samples
all_celltypes = adata.obs[CELLTYPE_COL].unique().tolist()
all_samples = adata.obs[SAMPLE_COL].unique().tolist()

print(f"  Total cell types: {len(all_celltypes)}")
print(f"  Total samples: {len(all_samples)}")

# Calculate %SnC
snc_data = []

for sample in all_samples:
    sample_mask = adata.obs[SAMPLE_COL] == sample
    
    # Get sample-level metadata
    sample_meta = adata.obs.loc[sample_mask].iloc[0]
    study_group = sample_meta[STUDY_GROUP_COL] if STUDY_GROUP_COL in adata.obs.columns else 'Unknown'
    
    for celltype in all_celltypes:
        ct_mask = sample_mask & (adata.obs[CELLTYPE_COL] == celltype)
        n_cells = ct_mask.sum()
        
        if n_cells >= MIN_CELLS_PER_SAMPLE:
            n_snc = adata.obs.loc[ct_mask, SENESCENCE_BOOL_COL].sum()
            pct_snc = (n_snc / n_cells) * 100
            
            snc_data.append({
                'Sample': sample,
                'Cell_Type': celltype,
                'N_Cells': n_cells,
                'N_SnC': n_snc,
                'Pct_SnC': pct_snc,
                'Study_Group': study_group
            })

snc_df = pd.DataFrame(snc_data)

print(f"\n✓ Calculated %SnC for {len(snc_df)} sample × cell type combinations")

# ════════════════════════════════════════════════════════════════════════════════
# FILTER TO CELL TYPES WITH SUFFICIENT DATA
# ════════════════════════════════════════════════════════════════════════════════

# Count samples per cell type
ct_sample_counts = snc_df.groupby('Cell_Type')['Sample'].nunique()

# Keep cell types with sufficient samples
valid_celltypes = ct_sample_counts[ct_sample_counts >= MIN_SAMPLES_PER_CELLTYPE].index.tolist()

print(f"\nCell types with ≥{MIN_SAMPLES_PER_CELLTYPE} samples:")
for ct in sorted(valid_celltypes):
    n = ct_sample_counts[ct]
    print(f"  ✓ {ct}: {n} samples")

excluded_cts = [ct for ct in all_celltypes if ct not in valid_celltypes]
if excluded_cts:
    print(f"\nExcluded (insufficient samples):")
    for ct in sorted(excluded_cts):
        n = ct_sample_counts.get(ct, 0)
        print(f"  ✗ {ct}: {n} samples")

# Filter
snc_df = snc_df[snc_df['Cell_Type'].isin(valid_celltypes)]

print(f"\n✓ Proceeding with {len(valid_celltypes)} cell types")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PIVOT TO WIDE FORMAT
# ════════════════════════════════════════════════════════════════════════════════

print("\nPivoting to wide format...")

# Pivot: rows = samples, columns = cell types, values = %SnC
snc_wide = snc_df.pivot(
    index='Sample',
    columns='Cell_Type',
    values='Pct_SnC'
)

# Add study group back
sample_study_group = snc_df.drop_duplicates('Sample').set_index('Sample')['Study_Group']
snc_wide['Study_Group'] = snc_wide.index.map(sample_study_group)

celltype_cols = [col for col in snc_wide.columns if col != 'Study_Group']

print(f"  Wide format: {snc_wide.shape[0]} samples × {len(celltype_cols)} cell types")

# Study group distribution
print(f"\nStudy group distribution:")
print(snc_wide['Study_Group'].value_counts().to_string())

# ════════════════════════════════════════════════════════════════════════════════
# CALCULATE CORRELATION MATRIX (PAIRWISE COMPLETE)
# ════════════════════════════════════════════════════════════════════════════════

print("\nCalculating correlation matrix (pairwise complete cases)...")

from scipy.stats import pearsonr

# Initialize matrices
n_cts = len(celltype_cols)
corr_matrix = pd.DataFrame(np.nan, index=celltype_cols, columns=celltype_cols)
pval_matrix = pd.DataFrame(np.nan, index=celltype_cols, columns=celltype_cols)
n_matrix = pd.DataFrame(0, index=celltype_cols, columns=celltype_cols)

for i, ct1 in enumerate(celltype_cols):
    for j, ct2 in enumerate(celltype_cols):
        if i == j:
            corr_matrix.loc[ct1, ct2] = 1.0
            pval_matrix.loc[ct1, ct2] = 0.0
            n_matrix.loc[ct1, ct2] = snc_wide[ct1].notna().sum()
        else:
            # Pairwise complete cases
            paired = snc_wide[[ct1, ct2]].dropna()
            n = len(paired)
            n_matrix.loc[ct1, ct2] = n
            
            if n >= 3:
                r, p = pearsonr(paired[ct1], paired[ct2])
                corr_matrix.loc[ct1, ct2] = r
                pval_matrix.loc[ct1, ct2] = p

print(f"\n✓ Correlation matrix calculated ({n_cts} × {n_cts})")

# Print correlation summary
print("\nPairwise correlations:")
for i, ct1 in enumerate(celltype_cols):
    for j, ct2 in enumerate(celltype_cols):
        if i < j:
            r = corr_matrix.loc[ct1, ct2]
            p = pval_matrix.loc[ct1, ct2]
            n = n_matrix.loc[ct1, ct2]
            if pd.notna(r):
                sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
                print(f"  {ct1} vs {ct2}: r = {r:.3f} (n = {n}) {sig}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 2A: CORRELATION MATRIX HEATMAP
# ════════════════════════════════════════════════════════════════════════════════

print("\nCreating correlation matrix heatmap...")

import seaborn as sns

# Create annotation with r values
annot_matrix = corr_matrix.round(2).astype(str)
annot_matrix = annot_matrix.replace('nan', '—')

# Figure size based on number of cell types
n_cts = len(celltype_cols)
cell_size = 0.7
fig_size = max(n_cts * cell_size + 2, 5)

fig, ax = plt.subplots(figsize=(fig_size, fig_size))

# Plot heatmap
sns.heatmap(
    corr_matrix,
    annot=annot_matrix,
    fmt='',
    cmap='RdYlBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
    ax=ax,
    annot_kws={'fontsize': 9},
    square=True
)

# Formatting
ax.set_title('Plot 2A: %SnC Correlation Between Cell Types', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('')
ax.set_ylabel('')

plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

ax.grid(False)

plt.tight_layout()

# Save
output_pdf = FIGURES_DIR / f'{DATASET}_validation_plot2A_correlation_matrix.svg'
output_svg = FIGURES_DIR / f'{DATASET}_validation_plot2A_correlation_matrix.svg'

plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)
plt.savefig(output_svg, format='svg', bbox_inches='tight', dpi=300)

print(f"✓ Saved: {output_pdf}")
print(f"✓ Saved: {output_svg}")

plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 2B: PAIRWISE SCATTER PLOTS
# ════════════════════════════════════════════════════════════════════════════════

print("\nCreating pairwise scatter plots...")

from itertools import combinations
from scipy.stats import pearsonr

# ─────────────────────────────────────────────────────────────────────────────
# DEFINE CORRECT COLOR PALETTE
# ─────────────────────────────────────────────────────────────────────────────

AGE_GROUP_COLORS = {
    'Age_20_29': '#2E86AB',   # Deep blue (youngest)
    'Age_30_39': '#4A90E2',   # Blue
    'Age_40_49': '#50C878',   # Green
    'Age_50_59': '#FFB347',   # Light orange
    'Age_60_69': '#FF8C00',   # Orange
    'Age_70_79': '#E24A4A',   # Red
    'Age_80_100': '#8B0000',  # Dark red (oldest)
    # Disease cohorts (if needed)
    'Control': '#4E79A7',
    'NCI': '#4E79A7',
    'MCI': '#F28E2B',
    'AD': '#E15759',
}

# ─────────────────────────────────────────────────────────────────────────────
# SETUP STUDY GROUPS
# ─────────────────────────────────────────────────────────────────────────────

# Get unique study groups
study_groups = snc_wide['Study_Group'].dropna().unique()

# Sort in logical order
age_order = ['Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 'Age_60_69', 'Age_70_79', 'Age_80_100']
condition_order = ['Control', 'NCI', 'MCI', 'AD']

study_group_list = []
for sg in age_order:
    if sg in study_groups:
        study_group_list.append(sg)
for sg in condition_order:
    if sg in study_groups and sg not in study_group_list:
        study_group_list.append(sg)
for sg in sorted(study_groups):
    if sg not in study_group_list:
        study_group_list.append(sg)

print(f"Study groups: {study_group_list}")

# Verify colors
print(f"\nColor mapping:")
for sg in study_group_list:
    color = AGE_GROUP_COLORS.get(sg, '#999999')
    status = "✓" if sg in AGE_GROUP_COLORS else "✗ (fallback)"
    print(f"  {status} {sg}: {color}")

# ─────────────────────────────────────────────────────────────────────────────
# CREATE SCATTER PLOTS
# ─────────────────────────────────────────────────────────────────────────────

ct_pairs = list(combinations(celltype_cols, 2))
n_pairs = len(ct_pairs)

print(f"\nCreating {n_pairs} pairwise scatter plots...")

n_cols = min(4, n_pairs)
n_rows = int(np.ceil(n_pairs / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3 * n_rows))

if n_pairs == 1:
    axes = [axes]
elif n_rows == 1:
    axes = list(axes)
else:
    axes = axes.flatten()

for idx, (ct1, ct2) in enumerate(ct_pairs):
    ax = axes[idx]
    
    plot_data = snc_wide[[ct1, ct2, 'Study_Group']].dropna()
    
    if len(plot_data) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_xlabel(f'{ct1} (%SnC)', fontsize=8)
        ax.set_ylabel(f'{ct2} (%SnC)', fontsize=8)
        continue
    
    # Plot points by study group
    for sg in study_group_list:
        sg_data = plot_data[plot_data['Study_Group'] == sg]
        if len(sg_data) > 0:
            color = AGE_GROUP_COLORS[sg]
            
            ax.scatter(
                sg_data[ct1], sg_data[ct2],
                c=color,
                label=sg.replace('Age_', '').replace('_', '-'),
                s=35,
                alpha=0.7,
                edgecolor='white',
                linewidth=0.4
            )
    
    # Regression line
    x = plot_data[ct1].values
    y = plot_data[ct2].values
    
    if len(x) >= 3:
        z = np.polyfit(x, y, 1)
        p_line = np.poly1d(z)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, p_line(x_line), 'k--', linewidth=1.2, alpha=0.8)
        
        r, pval = pearsonr(x, y)
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
        ax.text(0.05, 0.95, f'r={r:.2f}{sig}\nn={len(x)}',
               transform=ax.transAxes, fontsize=8, fontweight='bold',
               va='top', ha='left',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none'))
    
    ax.set_xlabel(f'{ct1} (%SnC)', fontsize=8)
    ax.set_ylabel(f'{ct2} (%SnC)', fontsize=8)
    ax.tick_params(labelsize=7)
    
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Hide empty subplots
for idx in range(n_pairs, len(axes)):
    axes[idx].set_visible(False)

# ─────────────────────────────────────────────────────────────────────────────
# LEGEND
# ─────────────────────────────────────────────────────────────────────────────

legend_handles = []
legend_labels = []
for sg in study_group_list:
    color = AGE_GROUP_COLORS[sg]
    legend_handles.append(plt.scatter([], [], c=color, s=40, edgecolor='white', linewidth=0.4))
    label = sg.replace('Age_', '').replace('_', '-')
    legend_labels.append(label)

fig.legend(legend_handles, legend_labels, 
          loc='upper right', bbox_to_anchor=(0.99, 0.98),
          title='Age Group', frameon=True, fontsize=8, title_fontsize=9,
          ncol=1)

fig.suptitle('Plot 2B: %SnC Correlation Between Cell Types',
            fontsize=12, fontweight='bold', y=1.01)

plt.tight_layout()

# Save
output_pdf = FIGURES_DIR / f'{DATASET}_validation_plot2B_scatter_pairs.svg'
output_svg = FIGURES_DIR / f'{DATASET}_validation_plot2B_scatter_pairs.svg'

plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)
plt.savefig(output_svg, format='svg', bbox_inches='tight', dpi=300)

print(f"\n✓ Saved: {output_pdf}")
print(f"✓ Saved: {output_svg}")

plt.show()

### Plot 2C: Connected Dot Plot (Key Glial Cell Types)

**Focus:** Microglia, Astrocyte, OPC — primary glial cell types implicated in brain aging and neuroinflammation.

**Visualization:**
- Each thin line = one sample (connects %SnC across 3 cell types)
- Each thick line with markers = age group median
- X-axis = Cell type
- Y-axis = %SnC
- Color = Age group (blue → green → orange → red)

**Interpretation:**
- **Parallel lines:** Coordinated senescence across cell types
- **Crossing lines:** Cell-type-specific senescence patterns
- **Rising medians with age:** Age-associated senescence burden

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 2C: CONNECTED DOT PLOT (KEY CELL TYPES)
# ════════════════════════════════════════════════════════════════════════════════

print("\nCreating connected dot plot (Plot 2C)...")

# ─────────────────────────────────────────────────────────────────────────────
# DEFINE CORRECT COLOR PALETTE
# ─────────────────────────────────────────────────────────────────────────────

AGE_GROUP_COLORS = {
    'Age_20_29': '#2E86AB',   # Deep blue (youngest)
    'Age_30_39': '#4A90E2',   # Blue
    'Age_40_49': '#50C878',   # Green
    'Age_50_59': '#FFB347',   # Light orange
    'Age_60_69': '#FF8C00',   # Orange
    'Age_70_79': '#E24A4A',   # Red
    'Age_80_100': '#8B0000',  # Dark red (oldest)
    # Disease cohorts (if needed)
    'Control': '#4E79A7',
    'NCI': '#4E79A7',
    'MCI': '#F28E2B',
    'AD': '#E15759',
}

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION: SPECIFY PREFERRED CELL TYPES (case-insensitive)
# ─────────────────────────────────────────────────────────────────────────────

# Set to None for auto-selection (top 3 by coverage), or specify preferred cell types:
PREFERRED_CELLTYPES = ['microglia', 'astrocyte', 'opc']  # Glial populations

# Alternative options:
# PREFERRED_CELLTYPES = ['excitatory', 'inhibitory', 'microglia']  # Neuronal + glial
# PREFERRED_CELLTYPES = None  # Auto-select top 3 by coverage

# ─────────────────────────────────────────────────────────────────────────────
# GET AVAILABLE CELL TYPES
# ─────────────────────────────────────────────────────────────────────────────

available_celltypes = [c for c in snc_wide.columns if c != 'Study_Group']
print(f"Available cell types: {available_celltypes}")

# Create case-insensitive lookup
celltype_lookup = {ct.lower(): ct for ct in available_celltypes}

# ─────────────────────────────────────────────────────────────────────────────
# SELECT TARGET CELL TYPES
# ─────────────────────────────────────────────────────────────────────────────

if PREFERRED_CELLTYPES is not None:
    # Match preferred cell types (case-insensitive)
    TARGET_CELLTYPES = []
    for pref in PREFERRED_CELLTYPES:
        pref_lower = pref.lower()
        # Try exact match first
        if pref_lower in celltype_lookup:
            TARGET_CELLTYPES.append(celltype_lookup[pref_lower])
        else:
            # Try partial match (e.g., 'opc' matches 'opcs')
            matches = [ct for ct_lower, ct in celltype_lookup.items() if pref_lower in ct_lower or ct_lower in pref_lower]
            if matches:
                TARGET_CELLTYPES.append(matches[0])
                print(f"  Partial match: '{pref}' → '{matches[0]}'")
    
    print(f"Preferred cell types: {PREFERRED_CELLTYPES}")
    print(f"Matched cell types: {TARGET_CELLTYPES}")
else:
    # Auto-select top 3 by sample coverage
    ct_counts = snc_wide[available_celltypes].notna().sum().sort_values(ascending=False)
    TARGET_CELLTYPES = ct_counts.head(3).index.tolist()
    print(f"Auto-selected top 3 cell types (by coverage): {TARGET_CELLTYPES}")

# ─────────────────────────────────────────────────────────────────────────────
# PREPARE DATA
# ─────────────────────────────────────────────────────────────────────────────

available_targets = [ct for ct in TARGET_CELLTYPES if ct in snc_wide.columns]
print(f"Target cell types for plot: {available_targets}")

if len(available_targets) < 2:
    print("⚠ Not enough cell types for connected dot plot. Skipping.")
else:
    # Get samples with all target cell types
    plot_cols = available_targets + ['Study_Group']
    pc_data = snc_wide[plot_cols].dropna().copy()
    
    print(f"Samples with all {len(available_targets)} cell types: {len(pc_data)}")
    
    # Sort study groups in age order
    age_order = ['Age_20_29', 'Age_30_39', 'Age_40_49', 'Age_50_59', 'Age_60_69', 'Age_70_79', 'Age_80_100']
    study_group_list = [sg for sg in age_order if sg in pc_data['Study_Group'].unique()]
    
    # Add any non-age groups
    for sg in sorted(pc_data['Study_Group'].unique()):
        if sg not in study_group_list:
            study_group_list.append(sg)
    
    print(f"\nStudy groups: {study_group_list}")
    print(f"\nSamples per group:")
    for sg in study_group_list:
        n = len(pc_data[pc_data['Study_Group'] == sg])
        color = AGE_GROUP_COLORS.get(sg, '#999999')
        print(f"  {sg}: n={n}, color={color}")
    
    # ─────────────────────────────────────────────────────────────────────────────
    # CREATE CONNECTED DOT PLOT
    # ─────────────────────────────────────────────────────────────────────────────
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    x_positions = list(range(len(available_targets)))
    
    # Plot individual samples (thin lines)
    for study_group in study_group_list:
        sg_data = pc_data[pc_data['Study_Group'] == study_group]
        color = AGE_GROUP_COLORS.get(study_group, '#999999')
        
        for _, row in sg_data.iterrows():
            y_vals = [row[ct] for ct in available_targets]
            ax.plot(x_positions, y_vals,
                    color=color,
                    alpha=0.25,
                    linewidth=0.8,
                    zorder=1)
            
            ax.scatter(x_positions, y_vals,
                      c=color,
                      s=15,
                      alpha=0.3,
                      edgecolor='none',
                      zorder=2)
    
    # Plot medians (thick lines with markers)
    for study_group in study_group_list:
        sg_data = pc_data[pc_data['Study_Group'] == study_group]
        if len(sg_data) < 2:
            continue
        
        median_vals = [sg_data[ct].median() for ct in available_targets]
        color = AGE_GROUP_COLORS.get(study_group, '#999999')
        label = study_group.replace('Age_', '').replace('_', '-')
        
        ax.plot(x_positions, median_vals,
                color=color,
                linewidth=3.5,
                marker='o',
                markersize=12,
                markeredgecolor='white',
                markeredgewidth=2,
                label=label,
                zorder=10)
    
    # ─────────────────────────────────────────────────────────────────────────────
    # FORMATTING
    # ─────────────────────────────────────────────────────────────────────────────
    
    # Capitalize cell type names for display
    display_labels = [ct.title() for ct in available_targets]
    
    ax.set_xticks(x_positions)
    ax.set_xticklabels(display_labels, fontsize=10, fontweight='bold', rotation=15, ha='right')
    ax.set_xlim(-0.4, len(available_targets) - 0.6)
    ax.set_ylabel('%SnC', fontsize=12)
    ax.set_title('Plot 2C: Senescence Coordination Across Glial Cell Types\n(Thin lines = individual samples, Thick lines = age group medians)',
                fontsize=11, fontweight='bold', pad=10)
    
    ax.yaxis.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.legend(loc='upper left', fontsize=9, title='Age Group', title_fontsize=10,
             frameon=True, framealpha=0.95)
    
    plt.tight_layout()
    
    # ─────────────────────────────────────────────────────────────────────────────
    # SAVE
    # ─────────────────────────────────────────────────────────────────────────────
    
    output_pdf = FIGURES_DIR / f'{DATASET}_validation_plot2C_connected_dotplot.svg'
    output_svg = FIGURES_DIR / f'{DATASET}_validation_plot2C_connected_dotplot.svg'
    
    plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)
    plt.savefig(output_svg, format='svg', bbox_inches='tight', dpi=300)
    
    print(f"\n✓ Saved: {output_pdf}")
    print(f"✓ Saved: {output_svg}")
    
    plt.show()
    
    # ─────────────────────────────────────────────────────────────────────────────
    # SUMMARY STATISTICS
    # ─────────────────────────────────────────────────────────────────────────────
    
    print("\n" + "="*70)
    print("PLOT 2C: CONNECTED DOT PLOT SUMMARY")
    print("="*70)
    
    # Dynamic header
    header = f"{'Age Group':<12}" + "".join([f"{ct.title():>18}" for ct in available_targets])
    print(f"\nMedian %SnC by Age Group:")
    print(header)
    print("-" * (12 + 18 * len(available_targets)))
    
    for study_group in study_group_list:
        sg_data = pc_data[pc_data['Study_Group'] == study_group]
        if len(sg_data) == 0:
            continue
        
        label = study_group.replace('Age_', '').replace('_', '-')
        vals = [sg_data[ct].median() for ct in available_targets]
        row = f"{label:<12}" + "".join([f"{v:>18.2f}" for v in vals])
        print(row)
    
    # Correlation among cell types
    print(f"\nCorrelation among target cell types (n={len(pc_data)}):")
    for i, ct1 in enumerate(available_targets):
        for j, ct2 in enumerate(available_targets):
            if i < j:
                r = pc_data[ct1].corr(pc_data[ct2])
                print(f"  {ct1.title()} vs {ct2.title()}: r = {r:.3f}")
    
    print("\n" + "-"*50)
    print("Interpretation:")
    print("  • Parallel lines → Coordinated senescence across cell types")
    print("  • Rising medians (blue→red) → Age-associated increase")
    print("  • Crossing lines → Cell-type-specific patterns")
    print("="*70)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 2: SUMMARY STATISTICS
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("PLOT 2: %SnC CORRELATION SUMMARY")
print("="*80)

# ─────────────────────────────────────────────────────────────────────────────
# OVERALL STATISTICS
# ─────────────────────────────────────────────────────────────────────────────

print(f"\nDataset: {DATASET}")
print(f"Total samples: {len(snc_wide)}")
print(f"Cell types analyzed: {len(celltype_cols)}")
print(f"Study groups: {len(study_group_list)}")

# ─────────────────────────────────────────────────────────────────────────────
# CORRELATION CATEGORIES (FROM PLOT 2A/2B)
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n--- Plot 2A/2B: Pairwise Correlations ---")

high_corr, medium_corr, low_corr = [], [], []
all_r = []

for i, ct1 in enumerate(celltype_cols):
    for j, ct2 in enumerate(celltype_cols):
        if i < j:
            r = corr_matrix.loc[ct1, ct2]
            if pd.notna(r):
                p = pval_matrix.loc[ct1, ct2]
                all_r.append(r)
                
                if abs(r) > 0.6:
                    high_corr.append((ct1, ct2, r, p))
                elif abs(r) > 0.3:
                    medium_corr.append((ct1, ct2, r, p))
                else:
                    low_corr.append((ct1, ct2, r, p))

print(f"\nCorrelation Categories:")
print(f"  High (|r| > 0.6): {len(high_corr)} pairs")
print(f"  Medium (0.3 < |r| ≤ 0.6): {len(medium_corr)} pairs")
print(f"  Low (|r| ≤ 0.3): {len(low_corr)} pairs")

if high_corr:
    print(f"\nTop 5 highly correlated pairs:")
    for ct1, ct2, r, p in sorted(high_corr, key=lambda x: -abs(x[2]))[:5]:
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        print(f"  • {ct1} ↔ {ct2}: r = {r:.3f} {sig}")

# Mean/median correlation
if all_r:
    mean_r = np.mean(all_r)
    median_r = np.median(all_r)
    print(f"\nOverall correlation:")
    print(f"  Mean r = {mean_r:.3f}")
    print(f"  Median r = {median_r:.3f}")

# ─────────────────────────────────────────────────────────────────────────────
# PLOT 2C: KEY GLIAL CELL TYPES
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n--- Plot 2C: Key Glial Cell Types ---")
print(f"Focus: {', '.join(available_targets)}")
print(f"Samples with all 3 cell types: {len(pc_data)}")

print(f"\nMedian %SnC by Age Group:")
print(f"{'Age Group':<12} ", end='')
for ct in available_targets:
    print(f"{ct:>12} ", end='')
print()
print("-"*52)

for study_group in study_group_list:
    sg_data = pc_data[pc_data['Study_Group'] == study_group]
    if len(sg_data) == 0:
        continue
    
    label = study_group.replace('Age_', '').replace('_', '-')
    print(f"{label:<12} ", end='')
    for ct in available_targets:
        median_val = sg_data[ct].median()
        print(f"{median_val:>12.2f} ", end='')
    print()

# Correlation among key glial cell types
print(f"\nCorrelation among key glial cell types:")
from scipy.stats import pearsonr

for i, ct1 in enumerate(available_targets):
    for j, ct2 in enumerate(available_targets):
        if i < j:
            r, p = pearsonr(pc_data[ct1], pc_data[ct2])
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            print(f"  {ct1} ↔ {ct2}: r = {r:.3f} {sig}")

# ─────────────────────────────────────────────────────────────────────────────
# INTERPRETATION
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*80)
print("INTERPRETATION")
print("="*80)

if mean_r > 0.6:
    print("\n✓ Senescence is SYSTEMIC")
    print("  High correlation suggests coordinated whole-brain aging.")
elif mean_r > 0.3:
    print("\n⚠ MIXED pattern")
    print("  Moderate correlation suggests both systemic and cell-type-specific factors.")
else:
    print("\n✗ Senescence is CELL-TYPE SPECIFIC")
    print("  Low correlation suggests independent senescence across cell types.")

# Age trend
if len(study_group_list) >= 2:
    print("\nAge-related trend:")
    youngest = study_group_list[0]
    oldest = study_group_list[-1]
    
    youngest_data = pc_data[pc_data['Study_Group'] == youngest]
    oldest_data = pc_data[pc_data['Study_Group'] == oldest]
    
    if len(youngest_data) > 0 and len(oldest_data) > 0:
        for ct in available_targets:
            young_median = youngest_data[ct].median()
            old_median = oldest_data[ct].median()
            fold_change = old_median / young_median if young_median > 0 else np.nan
            print(f"  {ct}: {young_median:.1f}% → {old_median:.1f}% ({fold_change:.1f}x)")

print("="*80)

# ─────────────────────────────────────────────────────────────────────────────
# SAVE RESULTS
# ─────────────────────────────────────────────────────────────────────────────

# Save correlation matrix
corr_file = RESULTS_DIR / f'{DATASET}_validation_plot2_correlation_matrix.csv'
corr_matrix.to_csv(corr_file)
print(f"\n✓ Saved: {corr_file}")

# Save %SnC data
snc_file = RESULTS_DIR / f'{DATASET}_validation_plot2_snc_by_celltype.csv'
snc_df.to_csv(snc_file, index=False)
print(f"✓ Saved: {snc_file}")

# Save key glial cell type data
glial_file = RESULTS_DIR / f'{DATASET}_validation_plot2C_glial_snc.csv'
pc_data.to_csv(glial_file)
print(f"✓ Saved: {glial_file}")

## Plot 3: Universal Hallmarks Scatter

**Purpose:** Validate that SnC cells show expected pathway-level expression changes.

**Biological Question:** Across genes in senescence pathways, do SnC cells show expected expression shifts compared to Non-SnC cells?

**Hallmarks (López-Otín et al., 2023; Gorgoulis et al., 2019):**
- DNA Damage Response (↑): H2AFX, ATM, ATR, TP53BP1, CHEK1/2
- Oxidative Stress (↑): NFE2L2, SOD1/2, CAT, GPX1/4, HMOX1
- Mitochondrial Dysfunction (↓): PPARGC1A, TFAM, ATP5A1, NDUFS1
- Neuroinflammation (↑): TNF, IL1B, IL6, CCL2, CXCL10, NLRP3
- Autophagy/Lysosomal (↑): SQSTM1, MAP1LC3B, LAMP1/2, GLB1
- Cell Cycle Arrest: Inhibitors ↑ (CDKN1A/2A/2B), Proliferation ↓ (MKI67, PCNA)
- Apoptosis Resistance: Anti-apoptotic ↑ (BCL2, BCL2L1), Pro-apoptotic ↓ (BAX, CASP3)

**Approach:**
- Calculate mean expression per gene for SnC vs Non-SnC cells
- Scatter plot: X = Non-SnC mean, Y = SnC mean
- Points above diagonal = Higher in SnC, below = Lower in SnC

**Outputs:**
- **3A:** Summary scatter (all pathways, aggregated)
- **3B:** Faceted scatter (one panel per pathway)
- **3C:** Per cell type analysis

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 3: UNIVERSAL HALLMARKS - SETUP
# ════════════════════════════════════════════════════════════════════════════════

print("="*70)
print("PLOT 3: UNIVERSAL HALLMARKS")
print("="*70)

# ─────────────────────────────────────────────────────────────────────────────
# HALLMARK GENE SETS
# ─────────────────────────────────────────────────────────────────────────────

HALLMARK_GENES = {
    'DNA Damage Response': {
        'genes': ['H2AFX', 'ATM', 'ATR', 'TP53BP1', 'CHEK1', 'CHEK2', 'MDC1', 'NBN', 'RAD50', 'MRE11'],
        'color': '#E15759'  # Red
    },
    'Oxidative Stress': {
        'genes': ['NFE2L2', 'SOD1', 'SOD2', 'CAT', 'GPX1', 'GPX4', 'HMOX1', 'NQO1', 'TXNRD1', 'GSR'],
        'color': '#F28E2B'  # Orange
    },
    'Mitochondrial Dysfunction': {
        'genes': ['PPARGC1A', 'TFAM', 'ATP5A1', 'ATP5B', 'NDUFS1', 'NDUFS2', 'SDHA', 'UQCRC1', 'COX5A', 'MT-CO1'],
        'color': '#76B7B2'  # Teal
    },
    'Neuroinflammation': {
        'genes': ['TNF', 'IL1B', 'IL6', 'CCL2', 'CXCL10', 'NLRP3', 'NFKB1', 'RELA', 'TLR4', 'PTGS2'],
        'color': '#59A14F'  # Green
    },
    'Autophagy/Lysosomal': {
        'genes': ['SQSTM1', 'MAP1LC3B', 'LAMP1', 'LAMP2', 'GLB1', 'CTSD', 'TFEB', 'BECN1', 'ATG5', 'ATG7'],
        'color': '#EDC948'  # Yellow
    },
    'Cell Cycle Arrest': {
        'genes': ['CDKN1A', 'CDKN2A', 'CDKN2B', 'RB1', 'TP53', 'MKI67', 'PCNA', 'CDK2', 'CDK4', 'CCND1'],
        'color': '#B07AA1'  # Purple
    },
    'Apoptosis Resistance': {
        'genes': ['BCL2', 'BCL2L1', 'MCL1', 'BAX', 'BAK1', 'CASP3', 'CASP9', 'BID', 'PUMA', 'NOXA'],
        'color': '#FF9DA7'  # Pink
    }
}

# ─────────────────────────────────────────────────────────────────────────────
# CHECK GENE AVAILABILITY
# ─────────────────────────────────────────────────────────────────────────────

available_genes = set(adata.var_names)

hallmark_available = {}
for hallmark, info in HALLMARK_GENES.items():
    genes = info['genes']
    found = [g for g in genes if g in available_genes]
    hallmark_available[hallmark] = found
    print(f"{hallmark}: {len(found)}/{len(genes)} genes")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CALCULATE MEAN EXPRESSION (CORRECTED FOR LOG-NORMALIZED DATA)
# ════════════════════════════════════════════════════════════════════════════════

print("\nCalculating mean expression...")

snc_mask = adata.obs['is_senescent'].values.astype(bool)
nonsnc_mask = ~snc_mask

n_snc = snc_mask.sum()
n_nonsnc = nonsnc_mask.sum()

print(f"SnC cells: {n_snc:,}")
print(f"Non-SnC cells: {n_nonsnc:,}")

results = []

for hallmark, info in HALLMARK_GENES.items():
    genes = hallmark_available[hallmark]
    
    for gene in genes:
        gene_idx = adata.var_names.get_loc(gene)
        
        if hasattr(adata.X, 'toarray'):
            expr = adata.X[:, gene_idx].toarray().flatten()
        else:
            expr = adata.X[:, gene_idx].flatten()
        
        snc_mean = expr[snc_mask].mean()
        nonsnc_mean = expr[nonsnc_mask].mean()
        
        # CORRECTED: For log1p-normalized data, log2FC = difference / ln(2)
        mean_diff = snc_mean - nonsnc_mean
        log2fc = mean_diff / np.log(2)
        
        results.append({
            'Gene': gene,
            'Hallmark': hallmark,
            'Color': info['color'],
            'SnC_Mean': snc_mean,
            'NonSnC_Mean': nonsnc_mean,
            'Log2FC': log2fc
        })

results_df = pd.DataFrame(results)

print(f"\nCalculated expression for {len(results_df)} genes")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'Hallmark':<28} {'n':<4} {'↑SnC':<6} {'↓SnC':<6} {'Mean log₂FC':<12}")
print("-"*60)

for hallmark in HALLMARK_GENES.keys():
    hallmark_data = results_df[results_df['Hallmark'] == hallmark]
    if len(hallmark_data) == 0:
        continue
    n_genes = len(hallmark_data)
    n_up = (hallmark_data['Log2FC'] > 0).sum()
    n_down = (hallmark_data['Log2FC'] < 0).sum()
    mean_fc = hallmark_data['Log2FC'].mean()
    print(f"{hallmark:<28} {n_genes:<4} {n_up:<6} {n_down:<6} {mean_fc:<12.3f}")

print("-"*60)
n_up_total = (results_df['Log2FC'] > 0).sum()
n_down_total = (results_df['Log2FC'] < 0).sum()
print(f"{'TOTAL':<28} {len(results_df):<4} {n_up_total:<6} {n_down_total:<6} {results_df['Log2FC'].mean():<12.3f}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 3A: SUMMARY SCATTER (ALL HALLMARKS)
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("PLOT 3A: SUMMARY SCATTER")
print("="*70)

# Dynamic axis limits
x_min = results_df['NonSnC_Mean'].min()
x_max = results_df['NonSnC_Mean'].max()
y_min = results_df['SnC_Mean'].min()
y_max = results_df['SnC_Mean'].max()

data_min = min(x_min, y_min)
data_max = max(x_max, y_max)

padding = (data_max - data_min) * 0.15
global_min = max(0, data_min - padding)
global_max = data_max + padding

# Create plot
fig, ax = plt.subplots(figsize=(8, 8))

# Diagonal line (y=x)
ax.plot([global_min, global_max], [global_min, global_max], 
        'k--', linewidth=1, alpha=0.5, label='y = x')

# Plot points by hallmark
for hallmark, info in HALLMARK_GENES.items():
    hallmark_data = results_df[results_df['Hallmark'] == hallmark]
    if len(hallmark_data) == 0:
        continue
    ax.scatter(hallmark_data['NonSnC_Mean'], hallmark_data['SnC_Mean'],
               c=info['color'], s=80, alpha=0.7, edgecolor='white', 
               linewidth=0.5, label=hallmark)

# Add gene labels
for _, row in results_df.iterrows():
    ax.annotate(row['Gene'], (row['NonSnC_Mean'], row['SnC_Mean']),
                fontsize=6, alpha=0.7, ha='left', va='bottom',
                xytext=(2, 2), textcoords='offset points')

# Formatting
ax.set_xlabel('Non-SnC Mean Expression (log-normalized)', fontsize=12)
ax.set_ylabel('SnC Mean Expression (log-normalized)', fontsize=12)
ax.set_title(f'Plot 3A: Universal Hallmarks\n(SnC: {n_snc:,} | Non-SnC: {n_nonsnc:,})', 
             fontsize=14, fontweight='bold')

ax.set_xlim(global_min, global_max)
ax.set_ylim(global_min, global_max)
ax.set_aspect('equal')

ax.legend(loc='upper left', fontsize=8, frameon=True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Stats annotation
n_above = (results_df['Log2FC'] > 0).sum()
n_below = (results_df['Log2FC'] < 0).sum()
ax.text(0.97, 0.03, f'Above diagonal: {n_above}\nBelow diagonal: {n_below}',
        transform=ax.transAxes, fontsize=10, ha='right', va='bottom',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_validation_plot3A_hallmarks.svg', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_validation_plot3A_hallmarks.svg', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved: {DATASET}_validation_plot3A_hallmarks.svg")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 3B: FACETED SCATTER BY HALLMARK
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("PLOT 3B: FACETED SCATTER BY HALLMARK")
print("="*70)

n_hallmarks = len(HALLMARK_GENES)
n_cols = 4
n_rows = int(np.ceil(n_hallmarks / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten()

for idx, (hallmark, info) in enumerate(HALLMARK_GENES.items()):
    ax = axes[idx]
    
    hallmark_data = results_df[results_df['Hallmark'] == hallmark]
    
    if len(hallmark_data) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(hallmark, fontweight='bold')
        continue
    
    # Dynamic axis limits per hallmark
    h_xmin = hallmark_data['NonSnC_Mean'].min()
    h_xmax = hallmark_data['NonSnC_Mean'].max()
    h_ymin = hallmark_data['SnC_Mean'].min()
    h_ymax = hallmark_data['SnC_Mean'].max()
    
    h_min = min(h_xmin, h_ymin)
    h_max = max(h_xmax, h_ymax)
    h_pad = (h_max - h_min) * 0.2
    
    axis_min = max(0, h_min - h_pad)
    axis_max = h_max + h_pad
    
    # Diagonal
    ax.plot([axis_min, axis_max], [axis_min, axis_max], 
            'k--', linewidth=1, alpha=0.5)
    
    # Points
    ax.scatter(hallmark_data['NonSnC_Mean'], hallmark_data['SnC_Mean'],
               c=info['color'], s=100, alpha=0.8, edgecolor='white', linewidth=0.5)
    
    # Labels
    for _, row in hallmark_data.iterrows():
        ax.annotate(row['Gene'], (row['NonSnC_Mean'], row['SnC_Mean']),
                    fontsize=8, ha='left', va='bottom',
                    xytext=(3, 3), textcoords='offset points')
    
    # Stats
    n_genes = len(hallmark_data)
    n_up = (hallmark_data['Log2FC'] > 0).sum()
    n_down = (hallmark_data['Log2FC'] < 0).sum()
    mean_fc = hallmark_data['Log2FC'].mean()
    
    ax.text(0.03, 0.97, f'n={n_genes} | ↑{n_up} ↓{n_down}\nlog₂FC={mean_fc:.2f}',
            transform=ax.transAxes, fontsize=9, va='top', ha='left',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # Formatting
    ax.set_title(hallmark, fontweight='bold', fontsize=12, color=info['color'])
    ax.set_xlabel('Non-SnC Mean', fontsize=10)
    ax.set_ylabel('SnC Mean', fontsize=10)
    ax.set_xlim(axis_min, axis_max)
    ax.set_ylim(axis_min, axis_max)
    ax.set_aspect('equal')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Hide empty axes
for idx in range(n_hallmarks, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(f'Plot 3B: Universal Hallmarks by Pathway\n(SnC: {n_snc:,} | Non-SnC: {n_nonsnc:,})',
             fontsize=14, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_validation_plot3B_hallmarks_faceted.svg', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_validation_plot3B_hallmarks_faceted.svg', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved: {DATASET}_validation_plot3B_hallmarks_faceted.svg")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 3B: FACETED SCATTER BY CELL TYPE (Hallmark Genes)
# ════════════════════════════════════════════════════════════════════════════════
%matplotlib inline

print("\n" + "="*70)
print("PLOT 3B: HALLMARK SCATTER BY CELL TYPE")
print("="*70)

# ── Select cell types to plot ─────────────────────────────────────────────────
FOCUS_CELLTYPES = ['Astrocyte', 'Oligodendrocyte',] #'Microglia']  # ← edit this
# FOCUS_CELLTYPES = sorted(adata.obs[CELL_TYPE_COLUMN].unique())  # ← uncomment for all

cell_types = [ct for ct in FOCUS_CELLTYPES if ct in adata.obs[CELL_TYPE_COLUMN].values]
print(f"  Cell types: {cell_types}")

senescence_col = 'is_senescent'

# ── Compute per cell type ─────────────────────────────────────────────────────

ct_results = []

for ct in cell_types:
    ct_mask = (adata.obs[CELL_TYPE_COLUMN] == ct).values
    snc_mask = (adata.obs[senescence_col] == True).values
    
    ct_snc = ct_mask & snc_mask
    ct_non = ct_mask & ~snc_mask
    
    if ct_snc.sum() < 10 or ct_non.sum() < 10:
        continue
    
    for hallmark, info in HALLMARK_GENES.items():
        for gene in info['genes']:
            if gene not in adata.var_names:
                continue
            
            gene_idx = list(adata.var_names).index(gene)
            
            if 'lognorm' in adata.layers:
                expr = adata.layers['lognorm'][:, gene_idx]
            else:
                expr = adata.X[:, gene_idx]
            
            if hasattr(expr, 'toarray'):
                expr = expr.toarray().flatten()
            else:
                expr = np.array(expr).flatten()
            
            snc_mean = expr[ct_snc].mean()
            non_mean = expr[ct_non].mean()
            diff = snc_mean - non_mean
            log2fc = diff / np.log(2)
            
            ct_results.append({
                'Cell_Type': ct,
                'Gene': gene,
                'Hallmark': hallmark,
                'Color': info['color'],
                'SnC_Mean': snc_mean,
                'NonSnC_Mean': non_mean,
                'Log2FC': log2fc,
            })

ct_results_df = pd.DataFrame(ct_results)
print(f"  ✓ Computed: {len(ct_results_df):,} gene × cell type combinations")

# ── Plot ──────────────────────────────────────────────────────────────────────

n_cts = len(cell_types)
n_cols = min(n_cts, 4)
n_rows = int(np.ceil(n_cts / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows),
                          squeeze=False)
axes = axes.flatten()

for idx, ct in enumerate(cell_types):
    ax = axes[idx]
    ct_data = ct_results_df[ct_results_df['Cell_Type'] == ct]
    
    if len(ct_data) == 0:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center',
                transform=ax.transAxes, fontsize=8, color='#999999')
        ax.set_title(ct, fontsize=8, fontweight='bold')
        continue
    
    all_vals = pd.concat([ct_data['NonSnC_Mean'], ct_data['SnC_Mean']])
    pad = (all_vals.max() - all_vals.min()) * 0.15
    axis_min = max(0, all_vals.min() - pad)
    axis_max = all_vals.max() + pad
    
    ax.plot([axis_min, axis_max], [axis_min, axis_max],
            'k--', linewidth=0.6, alpha=0.4)
    
    for hallmark in ct_data['Hallmark'].unique():
        h_data = ct_data[ct_data['Hallmark'] == hallmark]
        color = h_data['Color'].iloc[0]
        ax.scatter(h_data['NonSnC_Mean'], h_data['SnC_Mean'],
                   c=color, s=30, alpha=0.8, edgecolor='white', linewidth=0.3,
                   label=hallmark, zorder=3)
    
    top_genes = ct_data.reindex(ct_data['Log2FC'].abs().nlargest(5).index)
    for _, row in top_genes.iterrows():
        ax.annotate(row['Gene'], (row['NonSnC_Mean'], row['SnC_Mean']),
                    fontsize=4.5, ha='left', va='bottom',
                    xytext=(2, 2), textcoords='offset points', color='#333333')
    
    n_up = (ct_data['Log2FC'] > 0).sum()
    n_down = (ct_data['Log2FC'] < 0).sum()
    mean_fc = ct_data['Log2FC'].mean()
    
    ax.text(0.03, 0.97, f'↑{n_up} ↓{n_down}\nlog₂FC={mean_fc:.2f}',
            transform=ax.transAxes, fontsize=5.5, va='top', ha='left',
            color='black' if abs(mean_fc) > 0.05 else '#999999',
            fontweight='bold' if abs(mean_fc) > 0.05 else 'normal')
    
    ax.set_title(ct, fontsize=8, fontweight='bold', pad=3)
    ax.set_xlim(axis_min, axis_max)
    ax.set_ylim(axis_min, axis_max)
    ax.set_aspect('equal')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(labelsize=6, width=0.5)
    ax.grid(False)
    
    if idx % n_cols == 0:
        ax.set_ylabel('SnC Mean', fontsize=7)
    ax.set_xlabel('Non-SnC Mean', fontsize=7)

for idx in range(n_cts, len(axes)):
    axes[idx].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(by_label.values(), by_label.keys(), loc='lower right',
           bbox_to_anchor=(0.98, 0.02), fontsize=6, frameon=False, ncol=2)

plt.suptitle(f'{DATASET}: Hallmark Gene Expression (SnC vs Non-SnC)',
             fontsize=10, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_hallmarks_scatter_per_celltype.pdf', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_hallmarks_scatter_per_celltype.svg', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved: {DATASET}_hallmarks_scatter_per_celltype.pdf")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# PLOT 3C: UNIVERSAL HALLMARKS BY CELL TYPE
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("PLOT 3C: UNIVERSAL HALLMARKS BY CELL TYPE")
print("="*70)

# Get cell types
cell_types = sorted(adata.obs[CELL_TYPE_COLUMN].unique().tolist())
print(f"\nCell types: {len(cell_types)}")

# Calculate per cell type
results_by_ct = []

for ct in cell_types:
    ct_mask = adata.obs[CELL_TYPE_COLUMN] == ct
    ct_adata = adata[ct_mask]
    
    snc_mask_ct = ct_adata.obs['is_senescent'].values.astype(bool)
    nonsnc_mask_ct = ~snc_mask_ct
    
    n_snc_ct = snc_mask_ct.sum()
    n_nonsnc_ct = nonsnc_mask_ct.sum()
    
    if n_snc_ct < 10 or n_nonsnc_ct < 10:
        print(f"  {ct}: Skipping (SnC={n_snc_ct}, Non-SnC={n_nonsnc_ct})")
        continue
    
    print(f"  {ct}: SnC={n_snc_ct:,}, Non-SnC={n_nonsnc_ct:,}")
    
    for hallmark, info in HALLMARK_GENES.items():
        genes = hallmark_available[hallmark]
        
        for gene in genes:
            if gene not in ct_adata.var_names:
                continue
                
            gene_idx = ct_adata.var_names.get_loc(gene)
            
            if hasattr(ct_adata.X, 'toarray'):
                expr = ct_adata.X[:, gene_idx].toarray().flatten()
            else:
                expr = ct_adata.X[:, gene_idx].flatten()
            
            snc_mean = expr[snc_mask_ct].mean()
            nonsnc_mean = expr[nonsnc_mask_ct].mean()
            
            # CORRECTED: For log1p-normalized data
            mean_diff = snc_mean - nonsnc_mean
            log2fc = mean_diff / np.log(2)
            
            results_by_ct.append({
                'CellType': ct,
                'Gene': gene,
                'Hallmark': hallmark,
                'Color': info['color'],
                'SnC_Mean': snc_mean,
                'NonSnC_Mean': nonsnc_mean,
                'Log2FC': log2fc,
                'n_SnC': n_snc_ct,
                'n_NonSnC': n_nonsnc_ct
            })

results_ct_df = pd.DataFrame(results_by_ct)

print(f"\nTotal observations: {len(results_ct_df)}")
print(f"Cell types included: {results_ct_df['CellType'].nunique()}")

# ─────────────────────────────────────────────────────────────────────────────
# CREATE PLOT 3C
# ─────────────────────────────────────────────────────────────────────────────

cell_types_included = sorted(results_ct_df['CellType'].unique().tolist())
n_celltypes = len(cell_types_included)

n_cols = 4
n_rows = int(np.ceil(n_celltypes / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = axes.flatten()

for idx, ct in enumerate(cell_types_included):
    ax = axes[idx]
    
    ct_data = results_ct_df[results_ct_df['CellType'] == ct]
    
    if len(ct_data) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(ct, fontweight='bold')
        continue
    
    # Dynamic axis limits
    ct_min = min(ct_data['NonSnC_Mean'].min(), ct_data['SnC_Mean'].min())
    ct_max = max(ct_data['NonSnC_Mean'].max(), ct_data['SnC_Mean'].max())
    ct_pad = (ct_max - ct_min) * 0.15
    
    axis_min = max(0, ct_min - ct_pad)
    axis_max = ct_max + ct_pad
    
    # Diagonal
    ax.plot([axis_min, axis_max], [axis_min, axis_max], 
            'k--', linewidth=1, alpha=0.5)
    
    # Plot by hallmark
    for hallmark, info in HALLMARK_GENES.items():
        h_data = ct_data[ct_data['Hallmark'] == hallmark]
        if len(h_data) == 0:
            continue
        ax.scatter(h_data['NonSnC_Mean'], h_data['SnC_Mean'],
                   c=info['color'], s=60, alpha=0.7, edgecolor='white',
                   linewidth=0.3, label=hallmark)
    
    # Stats
    n_genes = len(ct_data)
    n_up = (ct_data['Log2FC'] > 0).sum()
    n_down = (ct_data['Log2FC'] < 0).sum()
    mean_fc = ct_data['Log2FC'].mean()
    n_snc_ct = ct_data['n_SnC'].iloc[0]
    n_nonsnc_ct = ct_data['n_NonSnC'].iloc[0]
    
    ax.text(0.03, 0.97, f'↑{n_up} ↓{n_down}\nlog₂FC={mean_fc:.2f}',
            transform=ax.transAxes, fontsize=9, va='top', ha='left',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # Formatting
    ax.set_title(f'{ct}\n(SnC: {n_snc_ct:,} | Non-SnC: {n_nonsnc_ct:,})', 
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Non-SnC Mean', fontsize=9)
    ax.set_ylabel('SnC Mean', fontsize=9)
    ax.set_xlim(axis_min, axis_max)
    ax.set_ylim(axis_min, axis_max)
    ax.set_aspect('equal')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Hide empty axes
for idx in range(n_celltypes, len(axes)):
    axes[idx].set_visible(False)

# Legend (on last visible axis)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', fontsize=8, 
           bbox_to_anchor=(0.98, 0.02), ncol=2)

fig.suptitle('Plot 3C: Universal Hallmarks by Cell Type', 
             fontsize=14, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_validation_plot3C_hallmarks_by_celltype.svg', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_validation_plot3C_hallmarks_by_celltype.svg', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved: {DATASET}_validation_plot3C_hallmarks_by_celltype.svg")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("PLOT 3C: SUMMARY BY CELL TYPE")
print("="*70)

print(f"\n{'Cell Type':<20} {'n_SnC':<10} {'n_NonSnC':<12} {'↑SnC':<6} {'↓SnC':<6} {'Mean log₂FC':<12}")
print("-"*70)

for ct in cell_types_included:
    ct_data = results_ct_df[results_ct_df['CellType'] == ct]
    n_snc_ct = ct_data['n_SnC'].iloc[0]
    n_nonsnc_ct = ct_data['n_NonSnC'].iloc[0]
    n_up = (ct_data['Log2FC'] > 0).sum()
    n_down = (ct_data['Log2FC'] < 0).sum()
    mean_fc = ct_data['Log2FC'].mean()
    
    print(f"{ct:<20} {n_snc_ct:<10,} {n_nonsnc_ct:<12,} {n_up:<6} {n_down:<6} {mean_fc:<12.3f}")

print("="*70)

# Save results
results_ct_df.to_csv(FIGURES_DIR / f'{DATASET}_validation_plot3C_hallmarks_by_celltype.csv', index=False)
print(f"\n✓ Results saved")